# NB72_embedding_collapse_lm

**What this runs:** Does the rare-row collapse happen in federated embedding models in general? Federated next-word prediction on WikiText-2 (clients = groups of Wikipedia articles), local Adam with coupled L2 vs AdamW / no decay / holder-weighted averaging / SGD / reset optimiser state / centralised, K = 10, 50, 100, seeds 0-2. Pre-registered predictions E1-E4 in experiments/fedembed.py.
**Time:** about 4.0 h (Kaggle's limit is 12 h per session).

## Before you start (once)
1. Right panel → **Settings** → **Accelerator** → **GPU T4 x2**. 2. **Settings** → **Internet** → **On**.

## How to run
**Save Version** → **Save & Run All (Commit)** → **Save**. Close the tab if you like.
When the version shows a green tick: **Output** tab → download
- **`embedding_collapse_lm.zip`** (results, small): send this one back;
- **`embedding_collapse_lm__models.zip`** (trained models): keep it.

In [ ]:
# Cell 2 - environment check
import os, time, torch, urllib.request
n = torch.cuda.device_count(); names = [torch.cuda.get_device_name(i) for i in range(n)]
print("GPUs found:", n, names)
if n < 2 or not all("T4" in x for x in names):
    raise RuntimeError("STOP: set Settings -> Accelerator -> 'GPU T4 x2', then run again.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15); print("Internet: on")
except Exception as e:
    raise RuntimeError("STOP: set Settings -> Internet -> On, then run again. (" + str(e) + ")")
print("OK - environment is correct.")

In [ ]:
# Cell 3 - pack(): zip results and models now (called after every step, so a timeout never loses finished work)
import os, shutil
def pack():
    for src, dst in [("/kaggle/working/results", "/kaggle/working/embedding_collapse_lm"), ("/kaggle/working/models", "/kaggle/working/embedding_collapse_lm__models")]:
        if os.path.isdir(src):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            shutil.make_archive(dst, "zip", "/kaggle/working", os.path.basename(dst))
            shutil.rmtree(dst)
            print("wrote", dst + ".zip")

In [ ]:
%%writefile core.py
# ======================================================================
#  GeFL-F experiment core (inlined into every experiments/E*.py file)
# ======================================================================
#  A compact, self-contained re-implementation of GeFL-F (Kang et al.,
#  IEEE TMC 2025, Algorithm 3) that follows the paper's appendix exactly:
#    FE      : Tables XXI-XXIII (3x16x16, or 10x16x16 for CIFAR-10)
#    headers : the ten CNN-1..CNN-10 of Tables XXI-XXIII
#    CVAE-F  : Table XX; Adam 1e-3, weight decay 1e-3 (Table XV)
#    DCGAN-F : Table XVIII; Adam 2e-4, betas (0.5, 0.999)
#    rounds  : Table XIV; SGD lr 0.1, momentum 0, Tw/Tg/Ts/Tr = 5/5/1/5
#  plus the class-imbalance components studied in this project (see COMPONENTS below):
#    HWA  holder-weighted aggregation of class-conditioning rows (Mech. A)
#    LCD  lazy conditioning decay (plan section 2.1)
#    GF   gap-filling synthetic label sampling (Mech. B', water-filling)
#    PCM  prior-completing mixed batches (new; see method notes below)
#    LA   logit adjustment by the client's own training prior
#    BCR  server-side generative classifier re-calibration
#
#  Speed: all data live on the GPU as tensors; the FE is frozen after
#  stage (i), so real features are computed once. Stage outputs are cached
#  inside one experiment file, so methods that differ only in later stages
#  reuse earlier ones with identical random streams (paired comparisons).
#
#  Source of truth: experiments/core.py + experiments/specs/<name>.py,
#  combined by experiments/build.py. Edit those, then rebuild.
# ======================================================================
import argparse
import csv
import json
import math
import os
import sys
import time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

# ---------------------------------------------------------------- paper settings
# Figure 4 (GeFL-F, 10 / 50 / 100 clients) is the parity target.
DATASETS = {
    "mnist": dict(in_ch=1, norm="none", header_bn=False, fe="mnist", frac=0.1, B=64,
                  T_FE=20, T_KA=100, T_TN=50, latent=16, augment=False,
                  paper={"cvae": (95.47, 95.04, 94.63), "dcgan": (95.13, 93.67, 93.08),
                         "fedprox": (92.57, 92.21, 91.33)}),
    "fmnist": dict(in_ch=1, norm="half", header_bn=True, fe="mnist", frac=0.1, B=64,
                   T_FE=20, T_KA=100, T_TN=50, latent=16, augment=False,
                   paper={"cvae": (83.14, 82.21, 81.65), "dcgan": (82.62, 81.29, 80.71),
                          "fedprox": (80.50, 79.94, 79.67)}),
    "svhn": dict(in_ch=3, norm=((0.4376821, 0.4437697, 0.47280442), (0.19803012, 0.20101562, 0.19703614)),
                 header_bn=True, fe="mnist", frac=0.1, B=64, T_FE=50, T_KA=100, T_TN=100, latent=16,
                 augment=False,
                 paper={"cvae": (76.26, 73.64, 76.00), "dcgan": (68.78, 69.32, 65.66),
                        "fedprox": (65.18, 63.88, 65.59)}),
    "cifar10": dict(in_ch=3, norm=((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                    header_bn=False, fe="cifar10", frac=0.5, B=128, T_FE=50, T_KA=200, T_TN=100,
                    latent=50, augment=True,
                    paper={"cvae": (55.86, 53.19, 51.46), "dcgan": (54.82, 53.80, 53.95),
                           "fedprox": (55.10, 54.95, 52.71)}),
    # Not in the GeFL paper: the many-class check. CIFAR-10's FE, heads with 100 outputs and CIFAR-10's
    # rounds. frac = 0.22 gives the standard CIFAR-100-LT shape at IF = 100 (about 480 images in the
    # largest class, 5 in the smallest, about 10.5k in total) with the total data budget fixed.
    "cifar100": dict(in_ch=3, norm=((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
                     header_bn=False, fe="cifar10", frac=0.22, B=128, T_FE=50, T_KA=200, T_TN=100,
                     latent=50, augment=True, C=100, paper=None),
}
HEADER_WIDTHS = [(16,), (16, 32), (20, 40), (10, 20), (16, 32, 64), (20, 40, 80),
                 (10, 20, 40), (16, 32, 64, 128), (20, 40, 80, 100), (10, 20, 40, 80)]
NUM_CLASSES = 10  # set per dataset by set_num_classes() (DATASETS[ds]["C"], default 10)


def set_num_classes(ds):
    global NUM_CLASSES
    NUM_CLASSES = DATASETS[ds].get("C", 10)


# Class groups by frequency rank (classes are ordered from most to least frequent under the long tail):
# head = first 30%, medium = next 30%, tail = last 40% (for C = 10: 0-2, 3-5, 6-9).
def HEAD():
    return slice(0, int(round(0.3 * NUM_CLASSES)))


def MED():
    return slice(int(round(0.3 * NUM_CLASSES)), int(round(0.6 * NUM_CLASSES)))


def TAIL():
    return slice(int(round(0.6 * NUM_CLASSES)), NUM_CLASSES)


def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)


def set_seed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- data
def _fetch_hf(root, sub, base):
    """Download <split>-00000-of-00001.parquet (test, train) from a Hugging Face
    dataset into root/sub. The original servers (Stanford, Toronto) are often slow."""
    import urllib.request
    d = os.path.join(root, sub)
    os.makedirs(d, exist_ok=True)
    try:
        for split in ("test", "train"):
            dst = os.path.join(d, f"{split}.parquet")
            if not os.path.exists(dst):
                log(f"downloading {sub} {split} from Hugging Face")
                urllib.request.urlretrieve(base + f"{split}-00000-of-00001.parquet", dst + ".part")
                os.replace(dst + ".part", dst)
        return True
    except Exception as e:  # fall back to torchvision's own download
        log(f"Hugging Face download failed ({e}); trying torchvision")
        return False


def _fetch_svhn_hf(root):
    return _fetch_hf(root, "svhn_hf", "https://huggingface.co/datasets/ufldl-stanford/svhn/resolve/main/cropped_digits/")


def load_dataset(name, root):
    """Whole dataset as preprocessed GPU tensors: x (N,C,32,32) float, y (N,) long.
    Preprocessing follows the reference utils/getData.py (MNIST: none,
    FMNIST: (0.5, 0.5), SVHN / CIFAR-10: dataset mean/std)."""
    cache = os.path.join(root, f"_gefl_tensors_{name}.pt")
    if os.path.exists(cache):
        d = torch.load(cache)
    else:
        import torchvision.datasets as tvd
        os.makedirs(root, exist_ok=True)
        if name in ("mnist", "fmnist"):
            cls = tvd.MNIST if name == "mnist" else tvd.FashionMNIST
            tr, te = cls(root, train=True, download=True), cls(root, train=False, download=True)
            xtr, ytr, xte, yte = tr.data.unsqueeze(1), tr.targets, te.data.unsqueeze(1), te.targets
        elif name == "svhn" and (os.path.isdir(os.path.join(root, "svhn_hf")) or _fetch_svhn_hf(root)):
            sizes = {"test.parquet": 47003111, "train.parquet": 135589380}  # Hugging Face file sizes
            t_wait = time.time()
            while any(not os.path.exists(os.path.join(root, "svhn_hf", f)) or
                      os.path.getsize(os.path.join(root, "svhn_hf", f)) < n for f, n in sizes.items()):
                if time.time() - t_wait > 4 * 3600:
                    raise RuntimeError("SVHN parquet download incomplete after 4 h")
                time.sleep(30)
            # Hugging Face mirror (ufldl-stanford/svhn, cropped_digits), used when
            # the Stanford server is slow. Same images; labels already 0-9.
            import io
            import pyarrow.parquet as pq
            from PIL import Image

            def read(split):
                t = pq.read_table(os.path.join(root, "svhn_hf", f"{split}.parquet")).to_pydict()
                x = np.stack([np.array(Image.open(io.BytesIO(im["bytes"])).convert("RGB")) for im in t["image"]])
                return torch.from_numpy(x).permute(0, 3, 1, 2), torch.tensor(t["label"])
            (xtr, ytr), (xte, yte) = read("train"), read("test")
        elif name == "svhn":
            tr, te = tvd.SVHN(root, split="train", download=True), tvd.SVHN(root, split="test", download=True)
            xtr, ytr = torch.from_numpy(tr.data), torch.from_numpy(tr.labels).long()
            xte, yte = torch.from_numpy(te.data), torch.from_numpy(te.labels).long()
        elif name == "cifar10" and not os.path.isdir(os.path.join(root, "cifar-10-batches-py")) and                 _fetch_hf(root, "cifar10_hf", "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/"):
            # Hugging Face mirror (~70x faster than the Toronto server); verified
            # identical to torchvision's copy: same images, same order, same labels.
            import io
            import pyarrow.parquet as pq
            from PIL import Image

            def read(split):
                t = pq.read_table(os.path.join(root, "cifar10_hf", f"{split}.parquet")).to_pydict()
                x = np.stack([np.array(Image.open(io.BytesIO(im["bytes"])).convert("RGB")) for im in t["img"]])
                return torch.from_numpy(x).permute(0, 3, 1, 2), torch.tensor(t["label"])
            (xtr, ytr), (xte, yte) = read("train"), read("test")
        elif name == "cifar100" and not os.path.isdir(os.path.join(root, "cifar-100-python")) and \
                _fetch_hf(root, "cifar100_hf", "https://huggingface.co/datasets/uoft-cs/cifar100/resolve/main/cifar100/"):
            import io
            import pyarrow.parquet as pq
            from PIL import Image

            def read(split):
                t = pq.read_table(os.path.join(root, "cifar100_hf", f"{split}.parquet")).to_pydict()
                x = np.stack([np.array(Image.open(io.BytesIO(im["bytes"])).convert("RGB")) for im in t["img"]])
                return torch.from_numpy(x).permute(0, 3, 1, 2), torch.tensor(t["fine_label"])
            (xtr, ytr), (xte, yte) = read("train"), read("test")
        elif name == "cifar100":
            tr, te = tvd.CIFAR100(root, train=True, download=True), tvd.CIFAR100(root, train=False, download=True)
            xtr, ytr = torch.from_numpy(tr.data).permute(0, 3, 1, 2), torch.tensor(tr.targets)
            xte, yte = torch.from_numpy(te.data).permute(0, 3, 1, 2), torch.tensor(te.targets)
        elif name == "cifar10":
            tr, te = tvd.CIFAR10(root, train=True, download=True), tvd.CIFAR10(root, train=False, download=True)
            xtr, ytr = torch.from_numpy(tr.data).permute(0, 3, 1, 2), torch.tensor(tr.targets)
            xte, yte = torch.from_numpy(te.data).permute(0, 3, 1, 2), torch.tensor(te.targets)
        else:
            raise ValueError(name)
        d = dict(xtr=xtr.contiguous(), ytr=ytr.long(), xte=xte.contiguous(), yte=yte.long())
        torch.save(d, cache)
    spec = DATASETS[name]

    def prep(x):
        if x.shape[-1] != 32:  # MNIST / FMNIST 28 -> 32 (Table XVI); in chunks to bound peak RAM
            x = torch.cat([F.interpolate(x[i:i + 8192].float().div(255.0), size=32, mode="bilinear",
                                         align_corners=False, antialias=True) for i in range(0, len(x), 8192)])
        else:
            x = x.float().div(255.0)
        if spec["norm"] == "half":
            x.sub_(0.5).div_(0.5)  # in place: no second full-size copy
        elif spec["norm"] != "none":
            m = torch.tensor(spec["norm"][0]).view(1, -1, 1, 1)
            s = torch.tensor(spec["norm"][1]).view(1, -1, 1, 1)
            x.sub_(m).div_(s)
        return x

    # training images stay on the CPU; localize() moves only the ones in use
    return dict(xtr=prep(d["xtr"]), ytr=d["ytr"], xte=prep(d["xte"]).to(DEV), yte=d["yte"].to(DEV))


def localize(data, part):
    """GPU copy of just the federation's images (+ held-out set), with the
    partition's indices remapped into it."""
    used = torch.cat(list(part["parts"]) + [part["heldout"]]).cpu()
    local = dict(xtr=data["xtr"][used].to(DEV), ytr=data["ytr"][used].to(DEV), xte=data["xte"], yte=data["yte"])
    pos, out = 0, []
    for p in part["parts"]:
        out.append(torch.arange(pos, pos + len(p), device=DEV))
        pos += len(p)
    part = dict(part, parts=out, heldout=torch.arange(pos, pos + len(part["heldout"]), device=DEV))
    return local, part


def make_partition(ytr, K, frac, IF, alpha, seed, min_size=10):
    """Client index lists. Total size fixed at frac * N (paper footnote 5);
    the long-tail profile n_c ~ IF^(c/(C-1)) is rescaled to keep that total,
    capped at what each class has (plan section 2.2). alpha=None -> IID equal
    shards; otherwise Dirichlet(alpha) per class. Also returns 2000 held-out
    images (200/class; 2000 // C per class when C > 10) never given to any client, for the referee and MND."""
    rng = np.random.RandomState(seed)
    y = ytr.cpu().numpy()
    C = NUM_CLASSES
    n_total = int(round(frac * len(y)))
    mu = IF ** (1.0 / (C - 1))
    prof = mu ** np.arange(C)
    want = np.floor(n_total * prof / prof.sum()).astype(int)
    by_class = [rng.permutation(np.where(y == c)[0]) for c in range(C)]
    nh = min(200, 2000 // C)
    heldout = np.concatenate([bc[:nh] for bc in by_class])
    pools = [bc[nh:nh + min(want[c], len(bc) - nh)] for c, bc in enumerate(by_class)]
    if alpha is None:
        allidx = rng.permutation(np.concatenate(pools))
        parts = [p.tolist() for p in np.array_split(allidx, K)]
    else:
        for _ in range(100):
            parts = [[] for _ in range(K)]
            for c in range(C):
                idx = pools[c]
                prop = rng.dirichlet([alpha] * K)
                cuts = (np.cumsum(prop) * len(idx)).astype(int)[:-1]
                for k, chunk in enumerate(np.split(idx, cuts)):
                    parts[k].extend(chunk.tolist())
            if min(len(p) for p in parts) >= min_size:
                break
        # Rare at K=100: if no draw gave every client min_size samples, top the
        # small clients up with random samples taken from the largest one.
        while min(len(p) for p in parts) < min_size:
            small = int(np.argmin([len(p) for p in parts]))
            big = int(np.argmax([len(p) for p in parts]))
            j = rng.randint(len(parts[big]))
            parts[small].append(parts[big].pop(j))
        parts = [rng.permutation(p).tolist() for p in parts]
    counts = np.array([[int((y[p] == c).sum()) for c in range(C)] for p in parts])
    return dict(parts=[torch.tensor(p, dtype=torch.long) for p in parts],
                counts=counts, heldout=torch.tensor(heldout, dtype=torch.long), class_counts=counts.sum(0))


def assign_arch(K, M=10):
    """Reference GeFL_CVAE-F.py:35: contiguous blocks."""
    blk = max(K // M, 1)
    return [min(k // blk, M - 1) for k in range(K)]


def gpu_augment(x):
    """RandomCrop(32, padding=4) + RandomHorizontalFlip on a GPU batch."""
    B, C, H, W = x.shape
    xp = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (B,), device=x.device)
    j = torch.randint(0, 9, (B,), device=x.device)
    ar = torch.arange(H, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None].expand(B, C, H, W)
    cols = (j[:, None] + ar)[:, None, None, :].expand(B, C, H, W)
    out = xp[torch.arange(B, device=x.device)[:, None, None, None],
             torch.arange(C, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(B, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], out.flip(3), out)


def batches(n, B):
    perm = torch.randperm(n, device=DEV)
    for s in range(0, n, B):
        yield perm[s:s + B]


# ---------------------------------------------------------------- networks
class PaperFE(nn.Module):
    def __init__(self, in_ch, variant):
        super().__init__()
        if variant == "mnist":
            layers = [nn.Conv2d(in_ch, 3, 3, 1, 1), nn.BatchNorm2d(3), nn.ReLU(inplace=True)]
            self.out_ch = 3
        else:
            layers = [nn.Conv2d(in_ch, 3, 3, 1, 1), nn.BatchNorm2d(3), nn.ReLU(inplace=True),
                      nn.Conv2d(3, 10, 3, 1, 1), nn.BatchNorm2d(10), nn.ReLU(inplace=True)]
            self.out_ch = 10
        self.net = nn.Sequential(*layers, nn.MaxPool2d(2, 2))

    def forward(self, x):
        return self.net(x)


class PaperHeader(nn.Module):
    def __init__(self, widths, in_ch, bn, C=None, S=16):
        super().__init__()
        C = C or NUM_CLASSES
        layers, ci, s = [], in_ch, S
        for w in widths:
            layers.append(nn.Conv2d(ci, w, 3, 1, 1))
            if bn:
                layers.append(nn.BatchNorm2d(w))
            layers += [nn.ReLU(inplace=True), nn.MaxPool2d(2, 2)]
            ci, s = w, s // 2
        self.features = nn.Sequential(*layers)
        self.fc = nn.Linear(ci * s * s, C)

    def embed(self, h):
        return self.features(h).flatten(1)

    def forward(self, h):
        return self.fc(self.embed(h))


class PaperCVAE(nn.Module):
    """Table XX. The decoder's Linear(l+C -> 512) is stored as dec_z (z
    columns + bias) and label_proj (C x 512, row r = class r's column), the
    same layer split so that row r is the only class-r-specific parameter."""
    kind = "cvae"

    def __init__(self, in_ch, latent, C=None, S=16):
        super().__init__()
        C = C or NUM_CLASSES
        n = int(round(math.log2(S)))
        widths = [min(64 * 2 ** i, 512) for i in range(n)]
        enc, ci = [], in_ch + 1
        for w in widths:
            enc += [nn.Conv2d(ci, w, 4, 2, 1), nn.BatchNorm2d(w), nn.ReLU(inplace=True)]
            ci = w
        self.enc = nn.Sequential(*enc)
        self.flat = widths[-1]
        self.latent = latent
        self.mu, self.logvar = nn.Linear(self.flat, latent), nn.Linear(self.flat, latent)
        self.dec_z = nn.Linear(latent, self.flat)
        self.label_proj = nn.Parameter(torch.empty(C, self.flat))
        b = 1.0 / math.sqrt(latent + C)
        for p in (self.dec_z.weight, self.dec_z.bias, self.label_proj):
            nn.init.uniform_(p, -b, b)
        dec, rev = [], list(reversed(widths))
        for i, w in enumerate(rev):
            co = rev[i + 1] if i + 1 < len(rev) else in_ch
            dec.append(nn.ConvTranspose2d(w, co, 4, 2, 1))
            if i + 1 < len(rev):
                dec += [nn.BatchNorm2d(co), nn.ReLU(inplace=True)]
        self.dec = nn.Sequential(*dec)

    # name -> dimension that indexes the class
    cond = {"label_proj": 0}

    def decode(self, z, y):
        h = F.relu(self.dec_z(z) + self.label_proj[y]).view(-1, self.flat, 1, 1)
        return F.relu(self.dec(h))

    def loss(self, x, y):
        ymap = y.float().view(-1, 1, 1, 1).expand(-1, 1, x.size(2), x.size(3))
        h = self.enc(torch.cat([x, ymap], 1)).flatten(1)
        mu, logvar = self.mu(h), self.logvar(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        rec = self.decode(z, y)
        return F.mse_loss(rec, x, reduction="sum") - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    @torch.no_grad()
    def sample(self, y):
        return self.decode(torch.randn(y.size(0), self.latent, device=y.device), y)


class ProtoCVAE(nn.Module):
    """PC-VAE (new): the CVAE-F of Table XX with its per-class label rows removed.
    Class identity comes from the EXACT federated class mean of the frozen-FE
    features, mu_c = sum_k S_kc / sum_k n_kc (secure-aggregated sums, the same
    disclosure as FSG), fed through a class-SHARED projection; the decoder models
    only the within-class residual:  x = ReLU(mu_y + dec(z, mu_y)).
    No class-specific parameter exists, so nothing can be diluted or collapse
    under FedAvg at any K; tail classes borrow within-class variation from all
    classes through the shared decoder. A linear decoder recovers FSG.
    Optional ex-post latent prior (zpost): z ~ N(m, S) fitted to the encoder means
    of all real features (exact sums, latent x latent), closing the prior hole."""
    kind = "cvae"
    cond = {}

    def __init__(self, in_ch, latent, proto, S=16, rows=False):
        super().__init__()
        n = int(round(math.log2(S)))
        widths = [min(64 * 2 ** i, 512) for i in range(n)]
        enc, ci = [], 2 * in_ch
        for w in widths:
            enc += [nn.Conv2d(ci, w, 4, 2, 1), nn.BatchNorm2d(w), nn.ReLU(inplace=True)]
            ci = w
        self.enc = nn.Sequential(*enc)
        self.flat, self.latent = widths[-1], latent
        self.mu, self.logvar = nn.Linear(self.flat, latent), nn.Linear(self.flat, latent)
        self.dec_z = nn.Linear(latent, self.flat)
        self.proto_proj = nn.Linear(proto[0].numel(), self.flat)
        dec, rev = [], list(reversed(widths))
        for i, w in enumerate(rev):
            co = rev[i + 1] if i + 1 < len(rev) else in_ch
            dec.append(nn.ConvTranspose2d(w, co, 4, 2, 1))
            if i + 1 < len(rev):
                dec += [nn.BatchNorm2d(co), nn.ReLU(inplace=True)]
        self.dec = nn.Sequential(*dec)
        self.register_buffer("proto", proto.clone())
        self.register_buffer("zm", torch.zeros(latent))
        self.register_buffer("zL", torch.eye(latent))
        # PCR (hybrid): learned class rows as in CVAE-F, aggregated with HWA, in addition
        # to the exact-mean anchor. Rows carry class identity where class means do not
        # (SVHN: nearest-class-mean accuracy at chance); the mean anchors where it does.
        self.rows = rows
        if rows:
            self.label_proj = nn.Parameter(torch.empty(NUM_CLASSES, self.flat))
            nn.init.uniform_(self.label_proj, -1.0 / math.sqrt(latent + NUM_CLASSES), 1.0 / math.sqrt(latent + NUM_CLASSES))
            self.cond = {"label_proj": 0}

    def decode(self, z, y):
        p = self.proto[y]
        a = self.dec_z(z) + self.proto_proj(p.flatten(1))
        if self.rows:
            a = a + self.label_proj[y]
        h = F.relu(a).view(-1, self.flat, 1, 1)
        return F.relu(p + self.dec(h))

    def encode(self, x, y):
        h = self.enc(torch.cat([x, self.proto[y]], 1)).flatten(1)
        return self.mu(h), self.logvar(h)

    def loss(self, x, y):
        mu, logvar = self.encode(x, y)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        rec = self.decode(z, y)
        return F.mse_loss(rec, x, reduction="sum") - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    @torch.no_grad()
    def fit_zpost(self, feats, ys_list):
        """Ex-post latent Gaussian from exact client sums of encoder means."""
        was = self.training
        self.eval()
        n, s1, s2 = 0, torch.zeros(self.latent, device=self.zm.device), torch.zeros(self.latent, self.latent, device=self.zm.device)
        for X, Y in zip(feats, ys_list):
            for i in range(0, len(Y), 2048):
                m, _ = self.encode(X[i:i + 2048], Y[i:i + 2048])
                n, s1, s2 = n + len(m), s1 + m.sum(0), s2 + m.T @ m
        mean = s1 / n
        cov = s2 / n - torch.outer(mean, mean) + 1e-4 * torch.eye(self.latent, device=mean.device)
        ev, V = torch.linalg.eigh(cov)
        self.zm.copy_(mean)
        self.zL.copy_(V * ev.clamp(min=1e-6).sqrt())
        self.train(was)

    @torch.no_grad()
    def sample(self, y):
        z = self.zm + torch.randn(y.size(0), self.latent, device=y.device) @ self.zL.T
        return self.decode(z, y)


class PaperDCGAN(nn.Module):
    """Table XVIII (d_g = d_d = 128 for MNIST/FMNIST/SVHN). D outputs logits
    (sigmoid + BCE in the paper == BCE-with-logits)."""
    kind = "dcgan"

    def __init__(self, in_ch, dg=128, dd=128, C=None, S=16):
        super().__init__()
        C = C or NUM_CLASSES
        self.C, self.S = C, S
        self.g_z = nn.ConvTranspose2d(100, 2 * dg, 4, 1, 0)
        self.g_z_bn = nn.BatchNorm2d(2 * dg)
        self.g_label = nn.ConvTranspose2d(C, 2 * dg, 4, 1, 0)
        self.g_label_bn = nn.BatchNorm2d(2 * dg)
        self.g_body = nn.Sequential(nn.ConvTranspose2d(4 * dg, 2 * dg, 4, 2, 1), nn.BatchNorm2d(2 * dg),
                                    nn.LeakyReLU(0.2, inplace=True), nn.ConvTranspose2d(2 * dg, in_ch, 4, 2, 1))
        self.d_x = nn.Conv2d(in_ch, dd // 2, 4, 2, 1)
        self.d_label = nn.Conv2d(C, dd // 2, 4, 2, 1)
        self.d_body = nn.Sequential(nn.Conv2d(dd, 2 * dd, 4, 2, 1), nn.BatchNorm2d(2 * dd),
                                    nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(2 * dd, 4 * dd, 4, 2, 1),
                                    nn.BatchNorm2d(4 * dd), nn.LeakyReLU(0.2, inplace=True),
                                    nn.Conv2d(4 * dd, 1, 2, 1, 0))
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.normal_(m.weight, 0.0, 0.02)

    cond = {"g_label.weight": 0, "d_label.weight": 1}

    def g_params(self):
        return [p for n, p in self.named_parameters() if n.startswith("g_")]

    def d_params(self):
        return [p for n, p in self.named_parameters() if n.startswith("d_")]

    def generate(self, z, y):
        oh = F.one_hot(y, self.C).float()[:, :, None, None]
        a = F.leaky_relu(self.g_z_bn(self.g_z(z[:, :, None, None])), 0.2)
        b = F.leaky_relu(self.g_label_bn(self.g_label(oh)), 0.2)
        return F.relu(self.g_body(torch.cat([a, b], 1)))

    def discriminate(self, x, y):
        fill = F.one_hot(y, self.C).float()[:, :, None, None].expand(-1, -1, x.size(2), x.size(3))
        a = F.leaky_relu(self.d_x(x), 0.2)
        b = F.leaky_relu(self.d_label(fill), 0.2)
        return self.d_body(torch.cat([a, b], 1)).flatten()

    @torch.no_grad()
    def sample(self, y):
        return self.generate(torch.randn(y.size(0), 100, device=y.device), y)


# ---------------------------------------------------------------- CUDA graphs
# These small networks are launch-bound on a laptop GPU (each kernel costs
# more CPU time than GPU time). Recording forward+backward once as a CUDA
# graph and replaying it gives the same maths at ~2.6x the speed. Only
# full-size batches are graphed; other shapes run eagerly.
USE_GRAPHS = DEV == "cuda"


class _Call(nn.Module):
    def __init__(self, fn, owner):
        super().__init__()
        self.owner = owner
        self.fn = fn

    def forward(self, *a):
        return self.fn(*a)


class GraphCache:
    def __init__(self):
        self.store = {}

    def __call__(self, key, module, fn, *args):
        if not USE_GRAPHS or not module.training:
            return fn(*args)
        g = self.store.get(key)
        if g is None:
            saved = {k: v.detach().clone() for k, v in module.state_dict().items()}
            g = torch.cuda.make_graphed_callables(_Call(fn, module), tuple(a.detach().clone() for a in args))
            module.load_state_dict(saved)  # undo BN-statistics updates from the warm-up passes
            self.store[key] = g
        return g(*args)


def sd_clone(m):
    return {k: v.detach().clone() for k, v in m.state_dict().items()}


class Averager:
    """Streaming average of client state dicts; optional holder-weighted
    rows for class-conditioning parameters (HWA)."""

    def __init__(self, cond=None, prev=None):
        self.sum, self.n = None, 0
        self.cond, self.prev = cond or {}, prev
        self.csum, self.cw = {}, {}

    def add(self, sd, class_w=None):
        if self.sum is None:
            self.sum = {k: (v.detach().clone().float() if v.is_floating_point() else v.clone())
                        for k, v in sd.items()}
        else:
            for k, v in sd.items():
                if v.is_floating_point():
                    self.sum[k] += v
                else:
                    self.sum[k] = v.clone()
        self.n += 1
        if class_w is not None:
            w = torch.as_tensor(class_w, dtype=torch.float32, device=DEV)
            for k, dim in self.cond.items():
                rows = sd[k].movedim(dim, 0)
                shape = (-1,) + (1,) * (rows.dim() - 1)
                if k not in self.csum:
                    self.csum[k] = torch.zeros_like(rows)
                    self.cw[k] = torch.zeros_like(w)
                self.csum[k] += w.view(shape) * rows
                self.cw[k] += w

    def result(self):
        out = {k: (v / self.n if v.is_floating_point() else v) for k, v in self.sum.items()}
        for k, dim in self.cond.items():
            if k not in self.csum:
                continue
            wsum = self.cw[k]
            shape = (-1,) + (1,) * (self.csum[k].dim() - 1)
            rows = self.csum[k] / wsum.clamp(min=1e-12).view(shape)
            prev_rows = (self.prev[k] if self.prev is not None else out[k]).movedim(dim, 0)
            rows = torch.where((wsum > 0).view(shape), rows, prev_rows)
            out[k] = rows.movedim(0, dim).contiguous()
        return out


def opt_to(opt, device):
    for st in opt.state.values():
        for k, v in st.items():
            if torch.is_tensor(v) and v.device.type != torch.device(device).type:
                st[k] = v.to(device, non_blocking=True)


# ---------------------------------------------------------------- method spec
FE_DEFAULTS = dict(la=False, T_FE=None, oracle=False)  # BALFE adds bal > 0 (class-balanced loss, clip)
GEN_DEFAULTS = dict(type="cvae", wd=1e-3, agg="flat", lazy=False, beta=0.999, weight="en",
                    dp_eps=None, oracle=False, T_KA=None, opt="adam", rowwd0=False)
HEAD_DEFAULTS = dict(sampler="uniform", order="syn_first", ts=1, tr=5, mix=None, mix_ratio=0.2, vct=0.0, mixup=0.0,
                     loss="ce", tau=1.0, bcr=0, bcr_lr=0.05, gen_update=False, csl=0.0, csl_mix=False,
                     sed=0, sed_lr=0.05, sed_beta=0.5, sed_bs=128, csl_mode="mix", mc=False, rsa=0.0, kh=False,
                     erl=0, erl_t=1.0, kh_mode="kh", kh_mult=4, kh_D=6144, mc_mode="both", sgate=0.0,
                     scsl=False)


def method(pipeline="gefl_f", fe=None, gen=None, head=None):
    return dict(pipeline=pipeline, fe={**FE_DEFAULTS, **(fe or {})},
                gen=None if gen == "none" else {**GEN_DEFAULTS, **(gen or {})},
                head={**HEAD_DEFAULTS, **(head or {})})


# Named components; compose("HWA+LCD+PCM+LA") builds a method. Method descriptions: docs/paper_draft.md.
COMPONENTS = {
    "HWA": dict(gen=dict(agg="A")),             # holder-weighted conditioning aggregation
    "LCD": dict(gen=dict(lazy=True)),           # lazy conditioning decay
    "NOWD": dict(gen=dict(wd=0.0)),             # generator without weight decay
    "ADAMW": dict(gen=dict(opt="adamw", wd=1e-2)),  # decoupled decay (Loshchilov & Hutter), PyTorch default 1e-2
    "ADAMW3": dict(gen=dict(opt="adamw", wd=1e-3)),  # decoupled decay at the paper's coefficient (matched to GeFL-F)
    "ROWNOWD": dict(gen=dict(rowwd0=True)),     # no decay on the class rows only; trunk keeps coupled decay 1e-3
    "GF": dict(head=dict(sampler="gapfill")),   # gap-filling synthetic labels (water-filling)
    "PCM": dict(head=dict(mix="pcm")),          # prior-completing mixed batches
    "LA": dict(head=dict(loss="la")),           # logit adjustment by the training prior
    "BCR": dict(head=dict(bcr=30)),             # server-side generative classifier re-calibration
    "LAFE": dict(fe=dict(la=True)),             # logit adjustment during FE warm-up too
    "BALFE": dict(fe=dict(bal=20.0)),           # P2: class-balanced FE loss N/(C n_y) from the exact global counts
    "RF": dict(head=dict(order="real_first")),  # real phase first, synthetic last
    "GENUPD": dict(head=dict(gen_update=True)),  # keep training G_F during stage (iii)
    "DCGAN": dict(gen=dict(type="dcgan", wd=0.0)),
    "DDPM": dict(gen=dict(type="ddpm", wd=0.0)),   # the authors' DDPM-F (feature diffusion, w = 0)
    "PC": dict(gen=dict(type="pcvae")),              # prototype-conditioned residual CVAE (no class rows)
    "PCR": dict(gen=dict(type="pcvae", rows=True, agg="A")),  # hybrid: exact-mean anchor + HWA-aggregated class rows
    "MCS": dict(head=dict(mcs=True)),                # MC-S: exact class mean + class covariance (Gelbrich map)
    "MCB": dict(head=dict(mcs=True, mcs_prior="gen")),  # MC-B: exact covariance where the class's sample size determines it,
                                                     # the generator's own shape where it does not (empirical Bayes)
    "VCT": dict(head=dict(vct=1.0)),                 # vicinal consensus transfer: real x consensus-labelled synthetic partners
    "MIXUP": dict(head=dict(mixup=1.0)),             # control: standard MixUp on the client's own real features
    "ZP": dict(gen=dict(zpost=True)),                # ex-post latent prior from exact sums
    "MC": dict(head=dict(mc=True)),                  # moment-calibrated sampling (exact mean + spread)
    "GATE": dict(gen=dict(gate_F=0.05)),             # runtime switch on the released class-separation F (training
                                                     # sums only): F < gate_F -> CVAE class rows + HWA, else PC-VAE
    "KME": dict(gen=dict(type="kme")),               # server-trained generator from exact kernel mean embeddings
    "KH": dict(head=dict(kh=True)),                  # kernel herding of samples toward exact class embeddings
    "SGATE": dict(head=dict(sgate=0.1)),             # P1: synthetic labels only for classes whose estimated signal
                                                     # fraction s_c (exact-sum LDA agreement) is >= 0.1
    "SCSL": dict(head=dict(scsl=True, csl=0.5)),     # P1 variant: per-class CSL weight beta_c = 1 - s_c
    "RSA": dict(head=dict(rsa=1.0)),                 # held-class real-synthetic alignment (shift transfer)
    "CSL": dict(head=dict(csl=0.5)),            # consensus soft labels for synthetic features
    "REAL": dict(head=dict(ts=0)),              # heads on real features only (no synthetic phase): plain FL heads
    "FEDLC": dict(head=dict(loss="lc")),        # FedLC logit calibration by the client's class counts (published)
    "FEDRS": dict(head=dict(loss="rs")),        # FedRS restricted softmax for classes the client lacks (published)
    "ERL": dict(head=dict(erl=1)),              # exact readout in the loop: every round the server sets each head's
                                                # last layer to the class-balanced LDA readout from exact sums of all
                                                # clients' embeddings; clients train only the body under it
    "ERL5": dict(head=dict(erl=5)),
    # C2 mechanism controls (action plan): is kernel herding's SELECTION what helps, and which MC moment matters?
    "KHRAND": dict(head=dict(kh_mode="random")),     # same candidate pool, a random n of the 4n instead of herding
    "KHD1536": dict(head=dict(kh_D=1536)),           # random-feature dimension 1536 (default 6144)
    "KHD12288": dict(head=dict(kh_D=12288)),
    "KHM2": dict(head=dict(kh_mult=2)),              # candidate pool 2n (default 4n)
    "KHM8": dict(head=dict(kh_mult=8)),
    "MCMEAN": dict(head=dict(mc_mode="mean")),       # MC moves only the class mean (no spread rescaling)
    "MCSPREAD": dict(head=dict(mc_mode="spread")),   # MC rescales only the spread (keeps the generator's mean)
    "FEDDF": dict(head=dict(sed=20, sed_beta=1.0)),  # FedDF-style server ensemble distillation on generated features             # the same, recomputed every 5 rounds (5x less extra communication)
    "BCSL": dict(head=dict(csl=0.5, csl_mode="bayes")),  # Bayesian CSL: label prior x ensemble, rho estimated
    "CSLM": dict(head=dict(csl=0.5, csl_mix=True)),
    "SED": dict(head=dict(sed=20)),             # server-side ensemble distillation on generated features  # ...interleaved into every real batch (no forgetting)
    "GAUSS": dict(gen=dict(type="gauss", wd=0.0)),   # federated sufficient-statistics Gaussian (FSG)
    "HYB": dict(gen=dict(type="hybrid")),            # CVAE-F for common classes, FSG for rare ones (n < 100)
    "RHYB": dict(gen=dict(type="hybrid", n_min_rel=0.5)),  # FSG for classes below half the average count
    "MIX": dict(gen=dict(type="mix", n_min_rel=1.0)),     # every class mixes FSG (w_c) and CVAE-F (1 - w_c)
}
OURS = "HWA+LCD+PCM+LA"
OURS_PRIVATE = "LCD+PCM+LA"  # no class histograms leave any client

# The final method, chosen from the quick-pass evidence (E03, E12-E14) and
# used by the full-run experiments F01-F08. Set once here.
FINAL_LABEL = "Ours"
# Full-scale F01 (3 seeds): +HWA+LA beat the hybrid on MNIST (87.9 vs 87.1)
# and FMNIST (76.0 vs 72.4), so the generator stays CVAE-F with HWA. F10
# (3 seeds) then showed consensus soft labels add on top of HWA + LA in both
# regimes (MNIST LT 89.9 vs 87.8, FMNIST LT 76.4 vs 76.0, FMNIST IID 83.2 vs
# 82.6 for GeFL-F), so the final method is HWA + LA + CSL.
FINAL_PARTS = "HWA+LA+CSL"
FINAL_OVER = {}


def final_method(**over):
    o = {k: dict(v) for k, v in FINAL_OVER.items()}
    for k, v in over.items():
        o.setdefault(k, {}).update(v)
    return compose(FINAL_PARTS, **o)


def compose(parts="", **over):
    """compose("HWA+LCD", head=dict(tau=1.5)) -> method dict."""
    fe, gen, head = {}, {}, {}
    for p in [q for q in parts.split("+") if q]:
        c = COMPONENTS[p]
        fe.update(c.get("fe", {}))
        gen.update(c.get("gen", {}))
        head.update(c.get("head", {}))
    fe.update(over.get("fe", {}))
    gen.update(over.get("gen", {}))
    head.update(over.get("head", {}))
    return method("gefl_f", fe, gen, head)


def baseline_method(name, la=False):
    pipe = {"LG-FedAvg": "lg_fedavg", "FedAvg": "fedavg_grouped"}[name]
    return method(pipe, gen="none", head=dict(loss="la" if la else "ce"))


BASELINE = method()


def run_spec(label, dataset, seed, m, IF=1.0, alpha=None, K=10, **extra):
    return dict(label=label, dataset=dataset, seed=seed, IF=IF, alpha=alpha, K=K, m=m, **extra)


# ---------------------------------------------------------------- training pieces
def rsa_loss(er, es, y, C):
    """Held-class real-synthetic alignment: per class present in the real batch,
    squared distance of the synthetic embedding mean to the (stop-gradient) real
    one, relative to the real mean's norm (scale-free)."""
    M = F.one_hot(y, C).float()
    cnt = M.sum(0)
    mr = (M.T @ er.detach()) / cnt.clamp(min=1)[:, None]
    ms = (M.T @ es) / cnt.clamp(min=1)[:, None]
    present = cnt > 0
    rel = (ms - mr).pow(2).sum(1) / mr.pow(2).sum(1).clamp(min=1e-6)
    return rel[present].mean()


def ce_or_la(logits, y, log_prior, tau):
    if log_prior is not None and tau:
        logits = logits + tau * log_prior
    return F.cross_entropy(logits, y)


def client_loss_adj(hspec, counts_k):
    """Per-client constant of the published long-tail FL losses (None for CE / LA)."""
    if hspec["loss"] == "lc":   # FedLC (Zhang et al., ICML 2022): z_c - tau * n_{k,c}^(-1/4)
        return torch.tensor(hspec["tau"] * (counts_k + 1.0) ** -0.25, dtype=torch.float32, device=DEV)
    if hspec["loss"] == "rs":   # FedRS (Li & Zhan, KDD 2021): logits of classes the client lacks scaled by alpha
        return torch.tensor(counts_k == 0, device=DEV)
    return None


def head_loss(logits, y, log_prior, hspec, adj=None):
    if hspec["loss"] == "lc":
        return F.cross_entropy(logits - adj, y)
    if hspec["loss"] == "rs":
        return F.cross_entropy(torch.where(adj[None, :], logits * hspec.get("rs_alpha", 0.5), logits), y)
    return ce_or_la(logits, y, log_prior, hspec["tau"])


def local_log_prior(counts, extra=None):
    p = counts.astype(float) + 1.0
    if extra is not None:
        p = p / p.sum() + extra
    p = p / p.sum()
    return torch.tensor(np.log(p), dtype=torch.float32, device=DEV)


def waterfill(real_counts, budget):
    """p over classes minimising the spread of real + synthetic exposure:
    p_c = [lam - a_c]_+ / budget with sum p = 1 (a_c = real exposure)."""
    a = np.asarray(real_counts, dtype=float)
    if budget <= 0:
        return np.full(len(a), 1.0 / len(a))
    order = np.sort(a)
    lam = None
    for i in range(1, len(a) + 1):
        lam = (budget + order[:i].sum()) / i
        if i == len(a) or lam <= order[i]:
            break
    p = np.clip(lam - a, 0, None)
    return p / p.sum()


class Ctx:
    """Everything one (dataset, partition, seed) needs."""

    def __init__(self, cfg, data, ds, part, K, seed):
        self.cfg, self.data, self.ds, self.part, self.K, self.seed = cfg, data, ds, part, K, seed
        self.spec = DATASETS[ds]
        self.arch = assign_arch(K)
        self.groups = sorted(set(self.arch))
        self.B = self.spec["B"]
        sizes = [len(p) for p in part["parts"]]
        self.sizes = sizes
        self.pfrac = 1.0  # fraction of clients that take part in each round (X11 partial participation)
        self.inst = None  # dict when the run is instrumented (X6): per-round diagnostics, see instrument_* below

    def active(self, r, stage):
        """Clients taking part in round r of a stage: all of them, or a seeded random subset of
        max(1, round(pfrac * K)) clients (the same subset for every method with the same seed)."""
        if self.pfrac >= 1.0:
            return range(self.K)
        m = max(1, int(round(self.pfrac * self.K)))
        rng = np.random.RandomState((self.seed * 1000003 + {"i": 1, "ii": 2, "iii": 3}[stage] * 10007 + r) % 2**32)
        return sorted(rng.choice(self.K, m, replace=False).tolist())


def build_headers(ctx, in_ch):
    return [PaperHeader(HEADER_WIDTHS[g], in_ch, ctx.spec["header_bn"]).to(DEV) for g in range(10)]


def _ece(prob, yte, bins=15):
    """Expected calibration error of the top-label confidence (Guo et al., 2017), equal-width bins."""
    conf, pred = prob.max(1)
    acc = (pred == yte).float()
    edges = torch.linspace(0, 1, bins + 1, device=prob.device)
    e = torch.zeros((), device=prob.device)
    for i in range(bins):
        sel = (conf > edges[i]) & (conf <= edges[i + 1])
        if sel.any():
            e += sel.float().mean() * (conf[sel].mean() - acc[sel].mean()).abs()
    return float(e)


def _class_metrics(logits, yte, C, prior=None):
    """Everything computed from one head's test logits. recall per class; macro-F1; G-mean of the
    class recalls (each floored at 0.01 so one missed class does not zero it); ECE, NLL (true-class
    probability floored at 1e-6) and Brier score of the softmax; and, when the head's own clients' label prior is given, the accuracy on
    test data re-weighted to that prior (what the head's own clients would see)."""
    prob = logits.float().softmax(1)
    pred = prob.argmax(1)
    correct = pred == yte
    ny = torch.bincount(yte, minlength=C).float()
    tp = torch.bincount(yte[correct], minlength=C).float()
    npred = torch.bincount(pred, minlength=C).float()
    rec_t = tp / ny.clamp(min=1)
    f1 = (2 * tp / (ny + npred).clamp(min=1)).mean()
    oh = F.one_hot(yte, C).float()
    out = dict(acc=float(correct.float().mean()), recall=rec_t.cpu().numpy(), f1=float(f1),
               gmean=float(torch.exp(torch.log(rec_t.clamp(min=0.01)).mean())),
               ece=_ece(prob, yte), nll=float(-prob.gather(1, yte[:, None]).clamp(min=1e-6).log().mean()),
               brier=float((prob - oh).pow(2).sum(1).mean()))
    if prior is not None:
        out["local"] = float((torch.as_tensor(prior, device=rec_t.device, dtype=torch.float32) * rec_t).sum())
    return out


@torch.no_grad()
def eval_headers(ctx, headers, feats_te, yte):
    """Per-architecture accuracy, per-class recall and the metrics of _class_metrics (test sets of
    MNIST/FMNIST/CIFAR-10/100 are balanced; SVHN's is not, hence both accuracy and balanced accuracy)."""
    out = {}
    C = NUM_CLASSES
    prob_sum = None
    counts = ctx.part["counts"] if hasattr(ctx, "part") else None  # _Sub (one group) carries no partition
    for g in ctx.groups:
        h = headers[g]
        h.eval()
        logits = torch.cat([h(feats_te[s:s + 2000]) for s in range(0, len(yte), 2000)])
        prob_sum = logits.softmax(1) if prob_sum is None else prob_sum + logits.softmax(1)
        prior = None
        if counts is not None:
            own = counts[[k for k in range(ctx.K) if ctx.arch[k] == g]].sum(0).astype(np.float64)
            prior = own / max(own.sum(), 1.0)
        out[g] = _class_metrics(logits, yte, C, prior=prior)
        h.train()
    if len(ctx.groups) > 1:
        # diagnostic only: the averaged-softmax ensemble of all architectures,
        # i.e. how much the headers know jointly that none knows alone
        correct = (prob_sum.argmax(1) == yte)
        rec = torch.stack([correct[yte == c].float().mean() for c in range(C)]).cpu().numpy()
        out["_ens"] = dict(acc=float(correct.float().mean()), recall=rec)
    return out


EXTRA_METRICS = ("f1", "gmean", "ece", "nll", "brier", "local")


def summarize_eval(ev):
    """Means over architectures (each architecture's head counts once, as in the paper's mean accuracy).
    worst = the worst class of the mean recall; worst_arch = the balanced accuracy of the weakest
    architecture; rec_std = spread of the class recalls (lower = fairer across classes)."""
    ens = ev.get("_ens")
    ev = {g: v for g, v in ev.items() if g != "_ens"}
    accs = np.array([v["acc"] for v in ev.values()])
    rec = np.stack([v["recall"] for v in ev.values()]).mean(0)
    out = dict(acc=float(accs.mean()), bal=float(rec.mean()), head=float(rec[HEAD()].mean()),
               medium=float(rec[MED()].mean()), tail=float(rec[TAIL()].mean()), worst=float(rec.min()),
               worst_arch=float(min(np.mean(v["recall"]) for v in ev.values())), rec_std=float(rec.std()),
               recall=rec.tolist())
    for k in EXTRA_METRICS:
        xs = [v[k] for v in ev.values() if k in v]
        if xs:
            out[k] = float(np.mean(xs))
    if ens is not None:
        out.update(ens_acc=ens["acc"], ens_bal=float(np.mean(ens["recall"])))
    return out


class Tracker:
    def __init__(self):
        self.best = {}
        self.hist = []

    def update(self, stage, rnd, ev):
        for g, v in ev.items():
            if g != "_ens":
                self.best[g] = max(self.best.get(g, 0.0), v["acc"])
        s = summarize_eval(ev)
        s.update(stage=stage, round=rnd, best_mean_acc=float(np.mean(list(self.best.values()))))
        self.hist.append(s)
        return s


# ----- stage (i): FE + header joint training (also LG-FedAvg / FedAvg baselines)
# ----- X6 instrumentation (math report §7, M1, M2, M4, M7, M8); active only when ctx.inst is a dict
def instrument_fe_grads(ctx, fe, headers, fe_sd, hd_sd, r):
    """M1: heterogeneity of the clients' full-batch FE gradients at the current global FE (Corollary 2):
    zeta^2 = sum_k n_k/N |g_k - g|^2, |g| (pooled gradient) and max_k |g_k|."""
    d = ctx.data
    grads, w = [], []
    for k in range(ctx.K):
        g = ctx.arch[k]
        fe.load_state_dict(fe_sd)
        headers[g].load_state_dict(hd_sd[g])
        fe.train(), headers[g].train()
        fe.zero_grad(set_to_none=True)
        idx = ctx.part["parts"][k]
        for b in batches(len(idx), 512):
            x, y = d["xtr"][idx[b]], d["ytr"][idx[b]]
            (F.cross_entropy(headers[g](fe(x)), y, reduction="sum") / len(idx)).backward()
        grads.append(torch.cat([p.grad.flatten() for p in fe.parameters() if p.grad is not None]).detach())
        w.append(len(idx))
    fe.load_state_dict(fe_sd)
    G_ = torch.stack(grads)
    w = torch.tensor(w, dtype=torch.float32, device=G_.device) / sum(w)
    gbar = (w[:, None] * G_).sum(0)
    zeta2 = float((w * (G_ - gbar).pow(2).sum(1)).sum())
    ctx.inst.setdefault("fe_grad", []).append(dict(round=r, zeta2=round(zeta2, 6), pooled=round(float(gbar.norm()), 6),
                                                   max_client=round(float(G_.norm(dim=1).max()), 6)))


@torch.no_grad()
def instrument_bn(ctx, fe):
    """M2: averaged BN running variance vs the exact pooled variance of the pre-BN activations, per channel."""
    fe.eval()
    mods = list(fe.net)
    out = []
    for i, m in enumerate(mods):
        if isinstance(m, nn.BatchNorm2d):
            s1, s2, cnt = 0, 0, 0
            for p in ctx.part["parts"]:
                for b in batches(len(p), 2048):
                    u = nn.Sequential(*mods[:i])(ctx.data["xtr"][p[b]])
                    s1 = s1 + u.sum((0, 2, 3)).double()
                    s2 = s2 + u.pow(2).sum((0, 2, 3)).double()
                    cnt += u.shape[0] * u.shape[2] * u.shape[3]
            var = s2 / cnt - (s1 / cnt) ** 2
            out.append((m.running_var.double() / var.clamp(min=1e-12)).round(decimals=3).tolist())
    ctx.inst["bn_var_ratio"] = out


def instrument_rows(ctx, r, k, nk, n0):
    """M4: per round and class, mean change of the class-row norm on holders (gain) and non-holders (loss)."""
    rec = ctx.inst.setdefault("rows", {})
    cur = rec.setdefault(r, dict(hg=np.zeros(NUM_CLASSES), hn=np.zeros(NUM_CLASSES), ng=np.zeros(NUM_CLASSES),
                                 nn=np.zeros(NUM_CLASSES)))
    hold = ctx.part["counts"][k] > 0
    dlt = np.asarray(nk) - np.asarray(n0)
    cur["hg"][hold] += dlt[hold]
    cur["hn"][hold] += 1
    cur["ng"][~hold] -= dlt[~hold]
    cur["nn"][~hold] += 1


def instrument_phases(ctx, r, k, h, G, syn_phase, real_phase):
    """M7 + M8 on one client: logit spread on generated features before / after the synthetic phase (Prop. 9), and
    the counterfactual retention of the synthetic phase's change through the real phase (Prop. 8), per last-layer row
    (absent vs held classes) and for the body: R = <w_after - w_cf, w_syn - w_before> / |w_syn - w_before|^2, where
    w_cf is the real phase run from the pre-synthetic head."""
    C = NUM_CLASSES
    if "probe" not in ctx.inst:
        with torch.no_grad():
            ctx.inst["probe"] = G.sample(torch.arange(C, device=DEV).repeat_interleave(32))

    def spread():
        was = h.training
        h.eval()
        with torch.no_grad():
            f = h(ctx.inst["probe"])
        h.train(was)
        return float((f - f.mean(1, keepdim=True)).pow(2).mean())

    def flat(sd, fc):
        return torch.cat([v.flatten().float() for n_, v in sd.items() if n_.startswith("fc.") == fc
                          and v.is_floating_point()])
    w0 = sd_clone(h)
    s0 = spread()
    syn_phase()
    w1, s1 = sd_clone(h), spread()
    real_phase()
    w2 = sd_clone(h)
    h.load_state_dict(w0)
    real_phase()
    wcf = sd_clone(h)
    h.load_state_dict(w2)
    dS = (w1["fc.weight"] - w0["fc.weight"]).float()
    dA = (w2["fc.weight"] - wcf["fc.weight"]).float()
    Rc = ((dA * dS).sum(1) / dS.pow(2).sum(1).clamp(min=1e-12)).cpu().numpy()
    absent = ctx.part["counts"][k] == 0
    db = flat(w1, False) - flat(w0, False)
    Rb = float((flat(w2, False) - flat(wcf, False)) @ db / db.pow(2).sum().clamp(min=1e-12))
    ctx.inst.setdefault("phases", []).append(dict(
        round=r, client=k, spread_before=round(s0, 4), spread_after_syn=round(s1, 4),
        retain_absent=round(float(Rc[absent].mean()), 4) if absent.any() else None,
        retain_held=round(float(Rc[~absent].mean()), 4), retain_body=round(Rb, 4)))


def stage_i(ctx, rounds, fe_mode="all", la=False, tracker=None, eval_every=10, bal=0.0):
    """fe_mode: 'all' (GeFL-F / LG-FedAvg: FE averaged over all clients),
    'group' (FedAvg grouped by architecture: whole model averaged per group)."""
    d, spec = ctx.data, ctx.spec
    fe = PaperFE(spec["in_ch"], spec["fe"]).to(DEV)
    headers = build_headers(ctx, fe.out_ch)
    fe_sd = {g: sd_clone(fe) for g in range(10)} if fe_mode == "group" else sd_clone(fe)
    hd_sd = [sd_clone(h) for h in headers]
    priors = [local_log_prior(ctx.part["counts"][k]) for k in range(ctx.K)] if la else None
    cw = None
    if bal:  # P2: weights N / (C n_c) from the exact global counts (one secure-aggregated count vector), clipped
        n_c = torch.tensor(ctx.part["counts"].sum(0), dtype=torch.float32, device=DEV).clamp(min=1)
        cw = (n_c.sum() / (len(n_c) * n_c)).clamp(max=bal)
    for r in range(rounds):
        if ctx.inst is not None and fe_mode == "all" and (r % 5 == 0 or r == rounds - 1):
            instrument_fe_grads(ctx, fe, headers, fe_sd, hd_sd, r)
        fe_avg = {g: Averager() for g in ctx.groups} if fe_mode == "group" else Averager()
        hd_avg = {g: Averager() for g in ctx.groups}
        for k in ctx.active(r, "i"):
            g = ctx.arch[k]
            fe.load_state_dict(fe_sd[g] if fe_mode == "group" else fe_sd)
            headers[g].load_state_dict(hd_sd[g])
            fe.train(), headers[g].train()
            opt = torch.optim.SGD(list(fe.parameters()) + list(headers[g].parameters()), lr=0.1, momentum=0.0)
            idx = ctx.part["parts"][k]
            for _ in range(5):  # T_w
                for b in batches(len(idx), ctx.B):
                    if b.numel() < 2:
                        continue
                    x, y = d["xtr"][idx[b]], d["ytr"][idx[b]]
                    if spec["augment"]:
                        x = gpu_augment(x)
                    if cw is not None:
                        loss = (F.cross_entropy(headers[g](fe(x)), y, reduction="none") * cw[y]).mean()
                    else:
                        loss = ce_or_la(headers[g](fe(x)), y, priors[k] if la else None, 1.0)
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
            (fe_avg[g] if fe_mode == "group" else fe_avg).add(fe.state_dict())
            hd_avg[g].add(headers[g].state_dict())
        if fe_mode == "group":
            for g in ctx.groups:
                if fe_avg[g].n:  # an architecture with no participating client keeps its model
                    fe_sd[g] = fe_avg[g].result()
        else:
            fe_sd = fe_avg.result()
        for g in ctx.groups:
            if hd_avg[g].n:
                hd_sd[g] = hd_avg[g].result()
        if tracker is not None and (r == 0 or (r + 1) % eval_every == 0 or r == rounds - 1):
            ev = {}
            for g in ctx.groups:
                fe.load_state_dict(fe_sd[g] if fe_mode == "group" else fe_sd)
                fe.eval()
                headers[g].load_state_dict(hd_sd[g])
                with torch.no_grad():
                    fte = torch.cat([fe(d["xte"][s:s + 2000]) for s in range(0, len(d["yte"]), 2000)])
                ev[g] = eval_headers(_Sub([g]), headers, fte, d["yte"])[g]
            tracker.update("fe_warmup", r + 1, ev)
    return fe_sd, hd_sd, fe


class _Sub:
    """Evaluate a subset of architectures (FedAvg-grouped has one FE per group)."""

    def __init__(self, groups):
        self.groups = groups


@torch.no_grad()
def compute_features(fe, x):
    fe.eval()
    return torch.cat([fe(x[s:s + 4000]) for s in range(0, len(x), 4000)])


def stage_i_oracle(ctx, epochs=30):
    """E10(a): FE (and all ten headers) trained centrally on the pooled
    federation data with class-balanced sampling."""
    d, spec = ctx.data, ctx.spec
    fe = PaperFE(spec["in_ch"], spec["fe"]).to(DEV)
    headers = build_headers(ctx, fe.out_ch)
    allidx = torch.cat(ctx.part["parts"])
    y = d["ytr"][allidx]
    w = 1.0 / torch.bincount(y, minlength=NUM_CLASSES).float().clamp(min=1)
    params = list(fe.parameters()) + [p for g in ctx.groups for p in headers[g].parameters()]
    opt = torch.optim.SGD(params, lr=0.1, momentum=0.0)
    n = len(allidx)
    for _ in range(epochs):
        draw = torch.multinomial(w[y], n, replacement=True)
        for s in range(0, n, ctx.B):
            b = draw[s:s + ctx.B]
            if b.numel() < 2:
                continue
            x = d["xtr"][allidx[b]]
            if spec["augment"]:
                x = gpu_augment(x)
            h = fe(x)
            loss = sum(F.cross_entropy(headers[g](h), y[b]) for g in ctx.groups)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
    return sd_clone(fe), [sd_clone(h) for h in headers], fe



# ----- DDPM-F: the authors' feature diffusion model (DDPM/ddpm16.py), ported
class _ResConv(nn.Module):
    def __init__(self, cin, cout, is_res=False):
        super().__init__()
        self.same, self.is_res = cin == cout, is_res
        self.conv1 = nn.Sequential(nn.Conv2d(cin, cout, 3, 1, 1), nn.BatchNorm2d(cout), nn.GELU())
        self.conv2 = nn.Sequential(nn.Conv2d(cout, cout, 3, 1, 1), nn.BatchNorm2d(cout), nn.GELU())

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        if self.is_res:
            return ((x if self.same else x1) + x2) / 1.414
        return x2


class _EmbedFC(nn.Module):
    def __init__(self, din, demb):
        super().__init__()
        self.din = din
        self.model = nn.Sequential(nn.Linear(din, demb), nn.GELU(), nn.Linear(demb, demb))

    def forward(self, x):
        return self.model(x.view(-1, self.din))


class _ContextUnet(nn.Module):
    """ContextUnet of the reference ddpm16.py (16x16 feature maps)."""

    def __init__(self, cin, n_feat=128, n_classes=10):
        super().__init__()
        self.n_feat, self.n_classes = n_feat, n_classes
        self.init_conv = _ResConv(cin, n_feat, is_res=True)
        self.down1 = nn.Sequential(_ResConv(n_feat, n_feat), nn.MaxPool2d(2))
        self.down2 = nn.Sequential(_ResConv(n_feat, 2 * n_feat), nn.MaxPool2d(2))
        self.to_vec = nn.Sequential(nn.AvgPool2d(4), nn.GELU())
        self.timeembed1, self.timeembed2 = _EmbedFC(1, 2 * n_feat), _EmbedFC(1, n_feat)
        self.contextembed1, self.contextembed2 = _EmbedFC(n_classes, 2 * n_feat), _EmbedFC(n_classes, n_feat)
        self.up0 = nn.Sequential(nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 4, 4), nn.GroupNorm(4, 2 * n_feat), nn.ReLU())
        self.up1 = nn.Sequential(nn.ConvTranspose2d(4 * n_feat, n_feat, 2, 2), _ResConv(n_feat, n_feat), _ResConv(n_feat, n_feat))
        self.up2 = nn.Sequential(nn.ConvTranspose2d(2 * n_feat, n_feat, 2, 2), _ResConv(n_feat, n_feat), _ResConv(n_feat, n_feat))
        self.out = nn.Sequential(nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1), nn.GroupNorm(8, n_feat), nn.ReLU(),
                                 nn.Conv2d(n_feat, cin, 3, 1, 1))

    def forward(self, x, c, t, context_mask):
        x = self.init_conv(x)
        d1 = self.down1(x)
        d2 = self.down2(d1)
        hv = self.to_vec(d2)
        c = F.one_hot(c, num_classes=self.n_classes).float() * (1 - context_mask[:, None])
        cemb1 = self.contextembed1(c).view(-1, 2 * self.n_feat, 1, 1)
        temb1 = self.timeembed1(t).view(-1, 2 * self.n_feat, 1, 1)
        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)
        u1 = self.up0(hv)
        u2 = self.up1(torch.cat([cemb1 * u1 + temb1, d2], 1))
        u3 = self.up2(torch.cat([cemb2 * u2 + temb2, d1], 1))
        return self.out(torch.cat([u3, x], 1))


class PaperDDPMF(nn.Module):
    """Reference DDPM-F: classifier-free-guidance DDPM on FE features
    (n_T = 200, betas 1e-4..0.02, context dropout 0.1, Adam 1e-4 with linear
    decay, guidance w = 0). With w = 0 the guided noise estimate equals the
    conditional one, so sampling runs only the conditional half of the
    reference's doubled batch (identical output, half the cost). The class
    enters through the first Linear of each context embedding: its column c
    is class c's conditioning row (HWA averages it over holders)."""
    kind = "ddpm"
    cond = {"net.contextembed1.model.0.weight": 1, "net.contextembed2.model.0.weight": 1}

    def __init__(self, in_ch, shape, n_feat=128, n_T=200, drop_prob=0.1):
        super().__init__()
        self.net = _ContextUnet(in_ch, n_feat, NUM_CLASSES)
        self.n_T, self.drop_prob, self.shape = n_T, drop_prob, tuple(shape)
        beta = (0.02 - 1e-4) * torch.arange(0, n_T + 1, dtype=torch.float32) / n_T + 1e-4
        alpha = 1 - beta
        abar = torch.cumsum(torch.log(alpha), 0).exp()
        self.register_buffer("sqrtab", abar.sqrt())
        self.register_buffer("sqrtmab", (1 - abar).sqrt())
        self.register_buffer("oneover_sqrta", 1 / alpha.sqrt())
        self.register_buffer("mab_over_sqrtmab", (1 - alpha) / (1 - abar).sqrt())
        self.register_buffer("sqrt_beta", beta.sqrt())

    def loss(self, x, y):
        t = torch.randint(1, self.n_T + 1, (len(x),), device=x.device)
        noise = torch.randn_like(x)
        xt = self.sqrtab[t, None, None, None] * x + self.sqrtmab[t, None, None, None] * noise
        mask = torch.bernoulli(torch.full((len(y),), self.drop_prob, device=x.device))
        return F.mse_loss(self.net(xt, y, t.float().div(self.n_T)[:, None], mask), noise)

    @torch.no_grad()
    def sample(self, y):
        if len(y) > 768:  # bound GPU memory: 200 UNet passes per sample
            return torch.cat([self.sample(y[i:i + 768]) for i in range(0, len(y), 768)])
        was = self.training
        self.eval()
        x = torch.randn(len(y), *self.shape, device=y.device)
        mask = torch.zeros(len(y), device=y.device)
        for i in range(self.n_T, 0, -1):
            t = torch.full((len(y), 1), i / self.n_T, device=y.device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=x.is_cuda):  # inference only
                eps = self.net(x, y, t, mask).float()
            z = torch.randn_like(x) if i > 1 else 0
            x = self.oneover_sqrta[i] * (x - eps * self.mab_over_sqrtmab[i]) + self.sqrt_beta[i] * z
        self.train(was)
        return x


# ----- stage (ii): feature-generator training
def make_gen(ctx, gspec, in_ch, shape=None, proto=None):
    if gspec["type"] == "pcvae":
        G = ProtoCVAE(in_ch, ctx.spec["latent"], proto, rows=gspec.get("rows", False)).to(DEV)
    elif gspec["type"] == "ddpm":
        G = PaperDDPMF(in_ch, shape).to(DEV)
    else:
        G = PaperCVAE(in_ch, ctx.spec["latent"]).to(DEV) if gspec["type"] == "cvae" else PaperDCGAN(in_ch).to(DEV)
    G.graphs = GraphCache()
    return G


class GenClient:
    """Per-client persistent Adam state(s) bound to the shared generator
    module (reference keeps one optimizer state per client across rounds)."""

    def __init__(self, G, gspec, offload):
        self.offload = offload
        if G.kind == "cvae":
            cond = [G.label_proj] if (gspec["lazy"] or gspec.get("rowwd0")) else []
            ids = {id(p) for p in cond}
            trunk = [p for p in G.parameters() if id(p) not in ids]
            groups = [{"params": trunk, "weight_decay": gspec["wd"]}]
            if cond:
                groups.append({"params": cond, "weight_decay": 0.0})
            # "adamw": decoupled decay (no gradient term), so a non-holder's row shrinks only by
            # (1 - lr * wd) per step instead of by a step of order lr (Theorem 1 / Lemma 2)
            opt_cls = torch.optim.AdamW if gspec.get("opt", "adam") == "adamw" else torch.optim.Adam
            self.opts = [opt_cls(groups, lr=1e-3, fused=DEV == "cuda")]
        elif G.kind == "ddpm":
            self.opts = [torch.optim.Adam(G.parameters(), lr=1e-4)]
        else:
            self.opts = [torch.optim.Adam(G.g_params(), lr=2e-4, betas=(0.5, 0.999)),
                         torch.optim.Adam(G.d_params(), lr=2e-4, betas=(0.5, 0.999))]

    def acquire(self):
        if self.offload:
            for o in self.opts:
                opt_to(o, DEV)

    def release(self):
        if self.offload:
            for o in self.opts:
                opt_to(o, "cpu")


def gen_local_update(G, gc, gspec, feats, ys, B=64, epochs=5):
    G.train()
    gc.acquire()
    n = len(ys)
    for _ in range(epochs):
        for b in batches(n, B):
            if b.numel() < 2:
                continue
            x, y = feats[b], ys[b]
            if G.kind == "ddpm":
                opt = gc.opts[0]
                loss = G.loss(x, y)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                continue
            if G.kind == "cvae":
                opt = gc.opts[0]
                loss = G.graphs(("loss", len(y)), G, G.loss, x, y) if len(y) == B else G.loss(x, y)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                if gspec["lazy"] and gspec["wd"] > 0:
                    # LCD: decay only the rows of classes present in this batch,
                    # and freeze absent rows (no leftover Adam momentum either).
                    present = torch.zeros(NUM_CLASSES, 1, device=DEV)
                    present[y] = 1.0
                    W = G.label_proj
                    saved = W.detach().clone()
                    if W.grad is not None:
                        W.grad.add_(gspec["wd"] * present * W.detach())
                    opt.step()
                    with torch.no_grad():
                        W.copy_(torch.where(present > 0, W, saved))
                else:
                    opt.step()
            else:
                og, od = gc.opts
                z = torch.randn(len(y), 100, device=DEV)
                fake = G.generate(z, y)
                d_real, d_fake = G.discriminate(x, y), G.discriminate(fake.detach(), y)
                ld = F.binary_cross_entropy_with_logits(d_real, torch.ones_like(d_real)) + \
                    F.binary_cross_entropy_with_logits(d_fake, torch.zeros_like(d_fake))
                od.zero_grad(set_to_none=True)
                ld.backward()
                od.step()
                fake = G.generate(torch.randn(len(y), 100, device=DEV), y)
                d_out = G.discriminate(fake, y)
                lg = F.binary_cross_entropy_with_logits(d_out, torch.ones_like(d_out))
                og.zero_grad(set_to_none=True)
                lg.backward()
                og.step()
    gc.release()


def class_weights(counts, gspec, rng):
    """HWA weights E(n) = (1 - beta^n) / (1 - beta) (or n), optionally from
    Laplace-noised histograms (E12, epsilon-DP per client histogram)."""
    c = counts.astype(float)
    if gspec["dp_eps"]:
        c = np.clip(c + rng.laplace(0.0, 1.0 / gspec["dp_eps"], size=c.shape), 0, None)
    if gspec["weight"] == "linear":
        return c
    if gspec["weight"] == "inverse":  # E08: the opposite direction (1/E(n) over holders)
        e = (1 - gspec["beta"] ** c) / (1 - gspec["beta"])
        return np.where(c > 0, 1.0 / np.maximum(e, 1e-12), 0.0)
    return (1 - gspec["beta"] ** c) / (1 - gspec["beta"])


@torch.no_grad()
def cond_row_norms(G):
    out = torch.zeros(NUM_CLASSES, device=DEV)
    for k, dim in G.cond.items():
        W = dict(G.named_parameters())[k].detach().movedim(dim, 0)
        out += W.reshape(NUM_CLASSES, -1).pow(2).sum(1)
    return out.sqrt().cpu().numpy()


# Differential privacy for the anchoring statistics (E26). None = exact release.
# When set: dict(clip, sig1, seed). Features are clipped to L2 norm <= clip; the
# four releases - class counts (sensitivity 1), class sums of h (clip), class sums
# of ||h||^2 (clip^2) and class sums of phi(h) (sqrt 2) - each receive Gaussian
# noise sig1 * sensitivity, with sig1 = sqrt(2 / rho) for a total rho-zCDP budget
# split equally (rho / 4 each), rho from the target (eps, delta).
_STAT_DP = None
SNR_GATE = 0.5  # DP mode: MC / KH used for a class only if noise norm <= SNR_GATE x signal scale


def zcdp_rho(eps, delta):
    L = math.log(1.0 / delta)
    return (math.sqrt(L + eps) - math.sqrt(L)) ** 2


def anchor_stats(feats, ys_list):
    """Per-class count n_c, sum of h and sum of ||h||^2 over all clients - what
    secure aggregation returns - exact, or clipped and Gaussian-noised under _STAT_DP.
    Deterministic given the DP seed, so every consumer sees the same release."""
    X, Y = torch.cat(feats).flatten(1), torch.cat(ys_list)
    dp = _STAT_DP
    if dp:
        X = X * (dp["clip"] / X.norm(dim=1, keepdim=True).clamp(min=1e-12)).clamp(max=1.0)
    S1 = torch.zeros(NUM_CLASSES, X.shape[1], device=DEV).index_add_(0, Y, X)
    S2 = torch.zeros(NUM_CLASSES, device=DEV).index_add_(0, Y, X.pow(2).sum(1))
    n = torch.bincount(Y, minlength=NUM_CLASSES).float()
    if dp:
        g = torch.Generator(device=DEV).manual_seed(dp["seed"])
        sg = dp["sig1"]
        n = n + sg * torch.randn(NUM_CLASSES, device=DEV, generator=g)
        S1 = S1 + sg * dp["clip"] * torch.randn(S1.shape, device=DEV, generator=g)
        S2 = S2 + sg * dp["clip"] ** 2 * torch.randn(NUM_CLASSES, device=DEV, generator=g)
    return n.clamp(min=1.0), S1, S2


def anchor_rff(feats, ys_list, phi):
    """Class sums of the bounded random features phi(h) (||phi|| <= sqrt 2)."""
    K_ = torch.zeros(NUM_CLASSES, phi.W.shape[0], device=DEV)
    for X_k, Y_k in zip(feats, ys_list):
        for i in range(0, len(Y_k), 4096):
            K_.index_add_(0, Y_k[i:i + 4096], phi(X_k[i:i + 4096]))
    if _STAT_DP:
        g = torch.Generator(device=DEV).manual_seed(_STAT_DP["seed"] + 1)
        K_ = K_ + _STAT_DP["sig1"] * math.sqrt(2.0) * torch.randn(K_.shape, device=DEV, generator=g)
    return K_


def dp_class_gate(n, dim, scale_norm):
    """Post-processing rule: a class's noisy statistic is used only if its expected
    noise norm sig1 * sens * sqrt(dim) / n_c is at most SNR_GATE x the signal scale
    (sens / scale_norm cancel to the ratio used here). All classes when not DP."""
    if not _STAT_DP:
        return torch.ones(NUM_CLASSES, dtype=torch.bool, device=DEV)
    return (_STAT_DP["sig1"] * math.sqrt(dim) * scale_norm / n) <= SNR_GATE


def class_prototypes(feats, ys_list):
    """Federated class means of FE features from the per-class sums and counts
    secure aggregation returns (exact, or DP under _STAT_DP)."""
    n, S1, _ = anchor_stats(feats, ys_list)
    return (S1 / n[:, None]).view((NUM_CLASSES,) + tuple(feats[0].shape[1:]))


# ----- KME-Gen (new): a federated generator that is never trained federatedly
class RFFMap(nn.Module):
    """Random Fourier features for a mixture of Gaussian kernels (bandwidths
    s0 * {0.5, 1, 2}); phi(x) = sqrt(2/D) cos(W x + b), ||phi(x)||_2 <= sqrt(2),
    so a client's class sums have L2 sensitivity sqrt(2) (DP-ready)."""

    def __init__(self, d, D, s0, seed=1234):
        super().__init__()
        g = torch.Generator(device="cpu").manual_seed(seed)
        parts = []
        for f in (0.5, 1.0, 2.0):
            parts.append(torch.randn(D // 3, d, generator=g) / (f * s0))
        W = torch.cat(parts)
        self.register_buffer("W", W)
        self.register_buffer("b", torch.rand(W.shape[0], generator=g) * 2 * math.pi)
        self.scale = math.sqrt(2.0 / W.shape[0])

    def forward(self, x):
        return self.scale * torch.cos(x.flatten(1) @ self.W.T + self.b)


class KMEGen(nn.Module):
    """Class-conditional feature generator trained AT THE SERVER by matching the
    exact federated kernel mean embedding of every class:
        min_G  sum_c || E_z phi(G(z, c)) - mu_c ||^2 = sum_c MMD_k(G_c, P_c)^2,
    mu_c = sum_k sum_{i in k, y_i = c} phi(h_i) / sum_k n_kc  (one secure-aggregation
    round). The objective equals the centralised one for ANY partition of the data
    (partition invariance): no FedAvg of generator weights, hence no collapse,
    dilution or client drift, and no generator training on clients.
    Decoder: the prototype-anchored residual of PC-VAE, x = ReLU(m_c + dec(z, m_c))."""
    kind = "cvae"
    cond = {}

    def __init__(self, in_ch, latent, proto, S=16):
        super().__init__()
        n = int(round(math.log2(S)))
        widths = [min(64 * 2 ** i, 512) for i in range(n)]
        self.flat, self.latent = widths[-1], latent
        self.dec_z = nn.Linear(latent, self.flat)
        self.proto_proj = nn.Linear(proto[0].numel(), self.flat)
        dec, rev = [], list(reversed(widths))
        for i, w in enumerate(rev):
            co = rev[i + 1] if i + 1 < len(rev) else in_ch
            dec.append(nn.ConvTranspose2d(w, co, 4, 2, 1))
            if i + 1 < len(rev):
                dec += [nn.BatchNorm2d(co), nn.ReLU(inplace=True)]
        self.dec = nn.Sequential(*dec)
        self.register_buffer("proto", proto.clone())

    def decode(self, z, y):
        p = self.proto[y]
        h = F.relu(self.dec_z(z) + self.proto_proj(p.flatten(1))).view(-1, self.flat, 1, 1)
        return F.relu(p + self.dec(h))

    @torch.no_grad()
    def sample(self, y):
        was = self.training
        self.eval()
        out = self.decode(torch.randn(y.size(0), self.latent, device=y.device), y)
        self.train(was)
        return out


def stage_ii_kme(ctx, gspec, feats, ys_list):
    """One upload per client (class sums of phi and of h, counts; secure-aggregated,
    optionally Gaussian-noised for (eps, delta)-DP), then server-side training."""
    in_ch = feats[0].shape[1]
    d = feats[0][0].numel()
    cnt, S1, S2 = anchor_stats(feats, ys_list)
    proto = (S1 / cnt[:, None]).view((NUM_CLASSES,) + tuple(feats[0].shape[1:]))
    # bandwidth from the released statistics: within-class spread V_c = E||h - mu_c||^2
    V = (S2 / cnt - proto.flatten(1).pow(2).sum(1)).clamp(min=1e-6)
    s0 = float(torch.sqrt(2 * V.mean()))
    phi = RFFMap(d, gspec.get("kme_D", 6144), s0, seed=1234 + ctx.seed).to(DEV)
    K_ = anchor_rff(feats, ys_list, phi)          # each client: its class sums of phi
    if gspec.get("kme_dp_eps"):
        eps, delta = gspec["kme_dp_eps"], 1e-5
        sig = math.sqrt(2.0) * math.sqrt(2 * math.log(1.25 / delta)) / eps
        K_ = K_ + sig * torch.randn_like(K_)
    mu = K_ / cnt[:, None]
    G = KMEGen(in_ch, ctx.spec["latent"], proto).to(DEV)
    opt = torch.optim.Adam(G.parameters(), lr=gspec.get("kme_lr", 1e-3))
    steps, bc = gspec.get("kme_steps", 4000), gspec.get("kme_bc", 128)
    y = torch.arange(NUM_CLASSES, device=DEV).repeat_interleave(bc)
    G.train()
    for t in range(steps):
        for gr in opt.param_groups:
            gr["lr"] = gspec.get("kme_lr", 1e-3) * (1 - t / steps)
        x = G.decode(torch.randn(len(y), G.latent, device=DEV), y)
        m = phi(x).view(NUM_CLASSES, bc, -1).mean(1)
        loss = (m - mu).pow(2).sum(1).sum()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    G.eval()
    return sd_clone(G), np.zeros((1, NUM_CLASSES)), G, None, None


def stage_ii(ctx, gspec, feats, ys_list, rounds):
    """Federated feature-generator training on frozen-FE features."""
    if gspec["type"] == "kme":
        return stage_ii_kme(ctx, gspec, feats, ys_list)
    in_ch = feats[0].shape[1]
    proto = class_prototypes(feats, ys_list) if gspec["type"] == "pcvae" else None
    G = make_gen(ctx, gspec, in_ch, feats[0].shape[1:], proto)
    g_sd = sd_clone(G)
    offload = ctx.K > 20
    clients = [GenClient(G, gspec, offload) for _ in range(ctx.K)]
    rng = np.random.RandomState(ctx.seed + 7)
    W = class_weights(ctx.part["counts"], gspec, rng)
    norms = [cond_row_norms(G)]
    for r in range(rounds):
        avg = Averager(G.cond if gspec["agg"] == "A" else None, prev=g_sd)
        for k in ctx.active(r, "ii"):
            G.load_state_dict(g_sd)
            if G.kind == "ddpm":  # reference: lr = 1e-4 * (1 - round / T_KA)
                for gr in clients[k].opts[0].param_groups:
                    gr["lr"] = 1e-4 * (1 - (r + 1) / rounds)
            gen_local_update(G, clients[k], gspec, feats[k], ys_list[k])
            if ctx.inst is not None and G.cond:
                instrument_rows(ctx, r, k, cond_row_norms(G), norms[-1])
            avg.add(G.state_dict(), W[k] if gspec["agg"] == "A" else None)
        g_sd = avg.result()
        G.load_state_dict(g_sd)
        norms.append(cond_row_norms(G))
    if gspec.get("zpost") and hasattr(G, "fit_zpost"):
        G.fit_zpost(feats, ys_list)  # one extra round of exact sums (latent stats only)
        g_sd = sd_clone(G)
    return g_sd, np.array(norms), G, clients, W


def stage_ii_oracle(ctx, gspec, feats, ys_list, rounds):
    """E10(b): the same generator trained centrally on the pooled federation
    features with class-balanced sampling, same number of optimizer steps."""
    in_ch = feats[0].shape[1]
    G = make_gen(ctx, gspec, in_ch, feats[0].shape[1:])
    gc = GenClient(G, gspec, False)
    X, Y = torch.cat(feats), torch.cat(ys_list)
    w = 1.0 / torch.bincount(Y, minlength=NUM_CLASSES).float().clamp(min=1)
    steps_per_round = sum(5 * math.ceil(len(y) / 64) for y in ys_list)
    for _ in range(rounds):
        draw = torch.multinomial(w[Y], steps_per_round * 64 // 5, replacement=True)
        gen_local_update(G, gc, gspec, X[draw], Y[draw], epochs=5)
    return sd_clone(G), np.array([cond_row_norms(G)]), G


# ----- FSG: federated sufficient-statistics Gaussian generator (new)
class GaussFeatureGen(nn.Module):
    """Class-conditional Gaussian in the common FE feature space with a
    shared (pooled within-class) covariance:  h | y ~ N(mu_y, Sigma),
    samples clipped at 0 (FE features are post-ReLU). Built in ONE round
    from client SUMS (see stage_ii_gauss), so the federated model equals
    the centralised one exactly - no averaging of weights, hence none of
    the dilution / collapse of a learned conditional generator."""
    kind = "gauss"

    def __init__(self, mu, chol, shape):
        super().__init__()
        self.mu = nn.Parameter(mu, requires_grad=False)
        self.chol = nn.Parameter(chol, requires_grad=False)
        self.shape = shape

    cond = {"mu": 0}

    @torch.no_grad()
    def sample(self, y):
        eps = torch.randn(len(y), self.mu.shape[1], device=self.mu.device)
        return (self.mu[y] + eps @ self.chol.T).clamp(min=0).view(len(y), *self.shape)


class HybridFeatureGen(nn.Module):
    """CVAE-F for classes with at least n_min samples federation-wide, the
    Gaussian for rarer ones (bias-variance: a low-capacity model is the
    better estimator when a class has few samples)."""
    kind = "hybrid"

    def __init__(self, cvae, gauss, use_gauss):
        super().__init__()
        self.cvae, self.gauss = cvae, gauss
        self.register_buffer("use_gauss", use_gauss)
        self.cond = {"gauss.mu": 0}

    @torch.no_grad()
    def sample(self, y):
        out = self.cvae.sample(y)
        m = self.use_gauss[y]
        if m.any():
            out[m] = self.gauss.sample(y[m])
        return out


class MixFeatureGen(nn.Module):
    """Every class draws from BOTH generators: FSG with probability
    w_c = nbar / (nbar + n_c) (nbar = rel * average class size), CVAE-F
    otherwise. Unlike HybridFeatureGen, the generator's style carries no
    information about which classes are rare, so a header cannot use it as
    a shortcut; and mixing two estimators with different errors (bias for
    the Gaussian, variance for the CVAE on rare classes) reduces error."""
    kind = "mix"

    def __init__(self, cvae, gauss, w):
        super().__init__()
        self.cvae, self.gauss = cvae, gauss
        self.register_buffer("w", w)
        self.cond = {"gauss.mu": 0}

    @torch.no_grad()
    def sample(self, y):
        out = self.cvae.sample(y)
        m = torch.rand(len(y), device=y.device) < self.w[y]
        if m.any():
            out[m] = self.gauss.sample(y[m])
        return out


def stage_ii_gauss(ctx, gspec, feats, ys_list, shrink=0.1, rng=None):
    """Each client uploads, for its features h (flattened, D dims):
        n_kc = #samples of class c,  S_kc = sum_{h in c} h,  M_k = sum_h h h^T.
    Under secure aggregation the server sees only the sums, which give the
    exact pooled estimates
        mu_c = sum_k S_kc / sum_k n_kc,
        W    = sum_k M_k - sum_c n_c mu_c mu_c^T   (within-class scatter),
        Sigma = (1 - shrink) W/(N - C) + shrink * tr(.)/D * I.
    Optional epsilon-DP: Laplace noise on counts (gspec['dp_eps'])."""
    D = feats[0][0].numel()
    shape = tuple(feats[0].shape[1:])
    C = NUM_CLASSES
    n = torch.zeros(C, device=DEV, dtype=torch.float64)
    S = torch.zeros(C, D, device=DEV, dtype=torch.float64)
    M = torch.zeros(D, D, device=DEV, dtype=torch.float64)
    for X, Y in zip(feats, ys_list):
        Xf = X.reshape(len(X), -1).double()
        n += torch.bincount(Y, minlength=C).double()
        S.index_add_(0, Y, Xf)
        M += Xf.T @ Xf
    if gspec.get("dp_eps"):
        r = rng or np.random.RandomState(ctx.seed + 11)
        n = (n + torch.from_numpy(r.laplace(0, 1.0 / gspec["dp_eps"], C)).to(DEV)).clamp(min=1.0)
    mu = S / n.clamp(min=1.0)[:, None]
    W = M - (mu.T * n) @ mu
    N = float(n.sum())
    Sig = W / max(N - C, 1.0)
    Sig = (Sig + Sig.T) / 2
    Sig = (1 - shrink) * Sig + shrink * (torch.trace(Sig) / D) * torch.eye(D, device=DEV, dtype=torch.float64)
    # Square root via eigendecomposition with a positive floor: W can lose
    # positive-definiteness when the counts carry DP noise.
    evals, evecs = torch.linalg.eigh(Sig)
    evals = evals.clamp(min=1e-6 * float(torch.trace(Sig)) / D)
    L = evecs * evals.sqrt()
    G = GaussFeatureGen(mu.float(), L.float(), shape).to(DEV)
    G.graphs = GraphCache()
    return G, n


# ----- referee (per-class generator fidelity) and feature-space MND
def train_referee(ctx, fe):
    d = ctx.data
    idx = ctx.part["heldout"]
    X, Y = compute_features(fe, d["xtr"][idx]), d["ytr"][idx]
    ref = PaperHeader(HEADER_WIDTHS[4], X.shape[1], True).to(DEV)
    opt = torch.optim.Adam(ref.parameters(), lr=1e-3)
    for _ in range(40):
        for b in batches(len(Y), 64):
            loss = F.cross_entropy(ref(X[b]), Y[b])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
    ref.eval()
    return ref


@torch.no_grad()
def referee_fidelity(ref, G, n=300):
    G.eval()
    fid = []
    for c in range(NUM_CLASSES):
        y = torch.full((n,), c, device=DEV, dtype=torch.long)
        fid.append(float((ref(G.sample(y)).argmax(1) == c).float().mean()))
    return np.array(fid)


def federated_kme(ctx, feats, ys_list, D=6144):
    """Exact pooled class kernel mean embeddings (one secure-aggregation upload of
    per-class phi sums and counts) and the RFF map that defines them."""
    cnt, S1, S2 = anchor_stats(feats, ys_list)
    V = (S2 / cnt - (S1 / cnt[:, None]).pow(2).sum(1)).clamp(min=1e-6)
    phi = RFFMap(feats[0][0].numel(), D, float(torch.sqrt(2 * V.mean())), seed=1234 + ctx.seed).to(DEV)
    K_ = anchor_rff(feats, ys_list, phi)
    # relative noise of mu_c: sig1 * sqrt(2) * sqrt(D) / n_c over the bound sqrt(2) on ||mu_c||
    return phi, K_ / cnt[:, None], dp_class_gate(cnt, phi.W.shape[0], 1.0)


class HerdedGen(nn.Module):
    """KH (new): sampling-time kernel herding of generated features toward the exact
    federated class embedding mu_c. For each class, draw M = mult * n candidates
    from the wrapped generator and greedily select n of them by Frank-Wolfe on
    MMD(selected, P_c):  w <- w + mu_c - phi(x_t),  x_{t+1} = argmax_i <phi_i, w>
    (without replacement). Lowers the MMD term of the head's risk bound for ANY
    generator; uses only the federated statistics already uploaded."""

    def __init__(self, G, phi, mu, mult=4, use=None, mode="kh"):
        super().__init__()
        self.G, self.kind, self.phi, self.mult, self.mode = G, G.kind, phi, mult, mode
        self.register_buffer("mu", mu)
        self.register_buffer("use", torch.ones(NUM_CLASSES, dtype=torch.bool, device=mu.device) if use is None else use)

    @torch.no_grad()
    def sample(self, y):
        if self.mode == "kh" and os.environ.get("GEFL_KH_LOOP") != "1":
            return self.sample_batched(y)
        return self.sample_loop(y)

    @staticmethod
    @torch.no_grad()
    def greedy(P, mu, n, valid):
        """Herding selection for a block of classes at once, without a host sync per step.
        P: (b, M, D) candidate features, mu: (b, D), n: (b,) picks per class, valid: (b, M) real candidates.
        Same picks as the per-class loop (incremental scores: s <- s + P mu - P P_j)."""
        b, M, _ = P.shape
        sc = torch.bmm(P, mu[:, :, None]).squeeze(2)            # <P_i, w_0>, w_0 = mu
        Pmu = sc.clone()
        sc = sc.masked_fill(~valid, -float("inf"))
        T = int(n.max())
        picks = torch.zeros(b, T, dtype=torch.long, device=P.device)
        ar = torch.arange(b, device=P.device)
        for t in range(T):
            j = sc.argmax(1)
            picks[:, t] = j
            sc[ar, j] = -float("inf")
            sc = sc + Pmu - torch.bmm(P, P[ar, j][:, :, None]).squeeze(2)
        return picks

    @torch.no_grad()
    def sample_batched(self, y):
        out = None
        cls = torch.unique(y)
        hard = [c for c in cls.tolist() if not bool(self.use[c])]
        for c in hard:  # DP gate: release too noisy for this class
            idx = (y == c).nonzero().flatten()
            sel = self.G.sample(y[idx])
            if out is None:
                out = torch.empty((len(y),) + tuple(sel.shape[1:]), device=y.device)
            out[idx] = sel
        cls = [c for c in cls.tolist() if bool(self.use[c])]
        if not cls:
            return out
        ns = {c: int((y == c).sum()) for c in cls}
        Mmax = self.mult * max(ns.values())
        D = self.mu.shape[1]
        per = max(1, int(2.5e8 // (4 * Mmax * D)))  # classes per block: about 250 MB of candidate features
        for s0 in range(0, len(cls), per):
            blk = cls[s0:s0 + per]
            lab = torch.cat([torch.full((self.mult * ns[c],), c, device=y.device, dtype=torch.long) for c in blk])
            cand = torch.cat([self.G.sample(lab[i:i + 4096]) for i in range(0, len(lab), 4096)])
            Pall = self.phi(cand)
            P = torch.zeros(len(blk), Mmax, D, device=y.device)
            valid = torch.zeros(len(blk), Mmax, dtype=torch.bool, device=y.device)
            off, starts = 0, []
            for i, c in enumerate(blk):
                m = self.mult * ns[c]
                P[i, :m] = Pall[off:off + m]
                valid[i, :m] = True
                starts.append(off)
                off += m
            n = torch.tensor([ns[c] for c in blk], device=y.device)
            picks = self.greedy(P, self.mu[torch.tensor(blk, device=y.device)], n, valid)
            for i, c in enumerate(blk):
                sel = cand[starts[i] + picks[i, :ns[c]]]
                idx = (y == c).nonzero().flatten()
                if out is None:
                    out = torch.empty((len(y),) + tuple(sel.shape[1:]), device=y.device)
                out[idx] = sel
        return out

    @torch.no_grad()
    def sample_loop(self, y):
        out = None
        for c in torch.unique(y):
            idx = (y == c).nonzero().flatten()
            n = len(idx)
            if not bool(self.use[c]):  # DP gate: release too noisy for this class
                sel = self.G.sample(y[idx])
                if out is None:
                    out = torch.empty((len(y),) + tuple(sel.shape[1:]), device=y.device)
                out[idx] = sel
                continue
            cand = torch.cat([self.G.sample(torch.full((min(4096, self.mult * n - i),), int(c), device=y.device,
                                                       dtype=torch.long)) for i in range(0, self.mult * n, 4096)])
            if self.mode == "random":  # control: same pool, uniform random subset
                sel = cand[torch.randperm(len(cand), device=y.device)[:n]]
                if out is None:
                    out = torch.empty((len(y),) + tuple(sel.shape[1:]), device=y.device)
                out[idx] = sel
                continue
            P = self.phi(cand)
            w = self.mu[c].clone()
            taken = torch.zeros(len(cand), dtype=torch.bool, device=y.device)
            pick = []
            for _ in range(n):
                sc = P @ w
                sc[taken] = -float("inf")
                j = int(sc.argmax())
                taken[j] = True
                pick.append(j)
                w += self.mu[c] - P[j]
            sel = cand[torch.tensor(pick, device=y.device)]
            if out is None:
                out = torch.empty((len(y),) + tuple(sel.shape[1:]), device=y.device)
            out[idx] = sel
        return out


class MomentCalibratedGen(nn.Module):
    """MC (new): sampling-time W2 projection of each class-conditional generator
    distribution onto the exact federated first moment and total spread:
        x' = ReLU(mu_c + s_c (x - m_c)),   s_c = sqrt(V_c / V~_c),
    mu_c, V_c = E||h - mu_c||^2 from exact client sums (sum h, sum ||h||^2, n per
    class: what secure aggregation returns); m_c, V~_c from the generator's own
    samples. Fixes VAE under-dispersion and tail-mean bias; keeps the learned shape."""

    def __init__(self, G, feats, ys_list, n=500, mode="both"):
        super().__init__()
        self.G, self.kind = G, G.kind
        cnt, S1, S2 = anchor_stats(feats, ys_list)
        D = S1.shape[1]
        mu = S1 / cnt[:, None]
        V = S2 / cnt - mu.pow(2).sum(1)
        # DP gate: relative noise of mu_c is sig1 * clip * sqrt(D) / n_c over the feature scale clip
        self.use = dp_class_gate(cnt, D, 1.0)
        G.eval()
        m, Vg = [], []
        with torch.no_grad():
            for c in range(NUM_CLASSES):
                g = G.sample(torch.full((n,), c, device=DEV, dtype=torch.long)).flatten(1)
                m.append(g.mean(0))
                Vg.append((g - g.mean(0)).pow(2).sum(1).mean())
        m, Vg = torch.stack(m), torch.stack(Vg)
        scale = (V.clamp(min=0) / Vg.clamp(min=1e-8)).sqrt().clamp(0.1, 10.0)
        if mode == "mean":      # control: move the mean only
            scale = torch.ones_like(scale)
        elif mode == "spread":  # control: rescale the spread only, keep the generator's own mean
            mu = m.clone()
        mu = torch.where(self.use[:, None], mu, m)            # gated classes: identity map
        scale = torch.where(self.use, scale, torch.ones_like(scale))
        self.register_buffer("mu", mu)
        self.register_buffer("m", m)
        self.register_buffer("scale", scale)
        self.shape = None

    @torch.no_grad()
    def sample(self, y):
        x = self.G.sample(y)
        shp = x.shape[1:]
        z = self.mu[y] + self.scale[y, None] * (x.flatten(1) - self.m[y])
        return F.relu(z).view(-1, *shp)


def _psd_pow(S, p, floor=1e-10):
    e, V = torch.linalg.eigh((S + S.T) / 2)
    e = e.clamp(min=floor * float(e.abs().max()))
    return (V * e.pow(p)) @ V.T


@torch.no_grad()
def anchor_cov(feats, ys_list, k=256, raw=False):
    """Exact federated class second moments for MC-S, in the top-k principal subspace
    of the pooled within-class covariance. Releases, each a secure-aggregated sum:
    the MC statistics (class counts, class sums of h and of ||h||^2), the pooled
    second moment sum h h^T (D x D, the same release as FSG), and, in the broadcast
    basis B (D x k), class sums of z = B^T h and of z z^T (C x k x k).
    Tail classes: the k x k covariance is shrunk toward the pooled one with prior
    strength k (an inverse-Wishart prior), then rescaled to the class's exact
    in-subspace variance, so the exact scalar moments are kept and only the shape is
    borrowed. Returns mu (C x D), B (D x k), Sig (C x k x k), Vperp (C,)."""
    cnt, S1, S2 = anchor_stats(feats, ys_list)
    mu = (S1 / cnt[:, None]).double()
    D = mu.shape[1]
    k = min(k, D)
    M = torch.zeros(D, D, device=DEV, dtype=torch.float64)
    for X_k in feats:                                   # per client: sum h h^T
        Xf = X_k.flatten(1)
        for i in range(0, len(Xf), 4096):
            xb = Xf[i:i + 4096].double()
            M += xb.T @ xb
    n = cnt.double()
    Sw = (M - (mu.T * n) @ mu) / max(float(n.sum()) - NUM_CLASSES, 1.0)
    Sw = (Sw + Sw.T) / 2
    ev, U = torch.linalg.eigh(Sw)
    B = U[:, -k:].contiguous()
    Spk = B.T @ Sw @ B
    Sz = torch.zeros(NUM_CLASSES, k, device=DEV, dtype=torch.float64)
    Szz = torch.zeros(NUM_CLASSES, k, k, device=DEV, dtype=torch.float64)
    for X_k, Y_k in zip(feats, ys_list):                # per client: class sums of z and z z^T
        Z = X_k.flatten(1).double() @ B
        Sz.index_add_(0, Y_k, Z)
        for c in torch.unique(Y_k):
            Zc = Z[Y_k == c]
            Szz[c] += Zc.T @ Zc
    mz = Sz / n[:, None]
    Sig, Vperp = [], []
    V = (S2.double() / n - mu.pow(2).sum(1)).clamp(min=1e-8)
    for c in range(NUM_CLASSES):
        Sc = (Szz[c] - n[c] * torch.outer(mz[c], mz[c])) / max(float(n[c]) - 1, 1.0)
        Sc = (Sc + Sc.T) / 2
        tin = torch.trace(Sc).clamp(min=1e-8)
        if raw:  # unshrunk: the caller chooses the prior (MC-B: the generator's own covariance)
            Sig.append(Sc)
            Vperp.append((V[c] - tin).clamp(min=1e-6 * float(V[c])))
            continue
        Sh = (n[c] * Sc + k * Spk) / (n[c] + k)
        Sh = Sh * (tin / torch.trace(Sh).clamp(min=1e-12))
        Sig.append(Sh)
        Vperp.append((V[c] - tin).clamp(min=1e-6 * float(V[c])))
    if raw:
        return mu, B, torch.stack(Sig), torch.stack(Vperp), n
    return mu, B, torch.stack(Sig), torch.stack(Vperp)


class CovCalibratedGen(nn.Module):
    """MC-S (new): sampling-time correction of each class-conditional generator
    distribution to the exact federated class mean AND class covariance (in the
    top-k principal subspace; exact spread outside it):
        x' = ReLU(mu_c + B A_c B^T (x - m_c) + s_c (I - B B^T)(x - m_c)),
        A_c = S~^-1/2 (S~^1/2 Sig_c S~^1/2)^1/2 S~^-1/2,   s_c = sqrt(Vperp_c / V~perp_c),
    with m_c, S~_c, V~perp_c from the generator's own samples. A_c is the Gelbrich
    (W2-optimal) linear map: among ALL maps achieving the target mean and covariance
    it displaces the samples least (Proposition 6). Unlike MC it gives class identity
    where the class MEANS carry none but the covariances do (SVHN: QDA 54 %,
    nearest-mean 10 %). Eigenvalues of A_c are clipped to [0.1, 10] as MC's scale."""

    def __init__(self, G, feats, ys_list, k=256, n=2048, prior="pooled"):
        super().__init__()
        self.G, self.kind = G, G.kind
        # prior = "pooled": tail-class covariances shrink toward the pooled one (MC-S).
        # prior = "gen" (MC-B): they shrink toward the GENERATOR's own class covariance,
        # rescaled to the class's exact variance - empirical Bayes with the generator as the
        # prior, weights n_c and k. A class with few samples keeps the generator's learned
        # shape (the Gelbrich map then reduces to MC's scalar map); a class with many gets
        # its exact covariance. One operator interpolating MC and MC-S by sample size.
        if prior == "gen":
            mu, B, Sig, Vperp, ncnt = anchor_cov(feats, ys_list, k, raw=True)
        else:
            mu, B, Sig, Vperp = anchor_cov(feats, ys_list, k)
        k = B.shape[1]
        G.eval()
        m, A, s = [], [], []
        with torch.no_grad():
            for c in range(NUM_CLASSES):
                g = torch.cat([G.sample(torch.full((min(n, 512),), c, device=DEV, dtype=torch.long)).flatten(1)
                               for _ in range((n + 511) // 512)]).double()
                mc = g.mean(0)
                r = g - mc
                z = r @ B
                Sg = z.T @ z / max(len(g) - 1, 1)
                Sg = (Sg + Sg.T) / 2 + 1e-4 * float(torch.trace(Sg)) / k * torch.eye(k, device=DEV, dtype=torch.float64)
                Sg_h, Sg_ih = _psd_pow(Sg, 0.5), _psd_pow(Sg, -0.5)
                target = Sig[c]
                if prior == "gen":
                    tin = torch.trace(Sig[c]).clamp(min=1e-12)
                    nc = float(ncnt[c])
                    target = (nc * Sig[c] + k * Sg * (tin / torch.trace(Sg).clamp(min=1e-12))) / (nc + k)
                Ac = Sg_ih @ _psd_pow(Sg_h @ target @ Sg_h, 0.5) @ Sg_ih
                e, Vv = torch.linalg.eigh((Ac + Ac.T) / 2)
                Ac = (Vv * e.clamp(0.1, 10.0)) @ Vv.T
                vperp_g = (r.pow(2).sum(1).mean() - z.pow(2).sum(1).mean()).clamp(min=1e-10)
                m.append(mc)
                A.append(Ac)
                s.append((Vperp[c] / vperp_g).sqrt().clamp(0.1, 10.0))
        self.register_buffer("mu", mu.float())
        self.register_buffer("m", torch.stack(m).float())
        self.register_buffer("B", B.float())
        self.register_buffer("A", torch.stack(A).float())
        self.register_buffer("s_perp", torch.stack(s).float())
        self.k = k

    @torch.no_grad()
    def sample(self, y):
        x = self.G.sample(y)
        shp = x.shape[1:]
        r = x.flatten(1) - self.m[y]
        z = r @ self.B
        out = torch.empty_like(r)
        for c in torch.unique(y):
            i = (y == c).nonzero().flatten()
            zc = z[i] @ self.A[c].T
            out[i] = self.mu[c] + zc @ self.B.T + self.s_perp[c] * (r[i] - z[i] @ self.B.T)
        return F.relu(out).view(-1, *shp)


@torch.no_grad()
def sample_spread(G, feats, ys_list, n=300):
    """Diversity diagnostic: per class, mean squared distance of generated
    samples to their own mean over the same for real features of that class
    (1 = realistic spread, 0 = prototype copies, > 1 = over-dispersed)."""
    X, Y = torch.cat(feats).flatten(1), torch.cat(ys_list)
    G.eval()
    out = []
    for c in range(NUM_CLASSES):
        R = X[Y == c]
        g = G.sample(torch.full((n,), c, device=DEV, dtype=torch.long)).flatten(1)
        vr = (R - R.mean(0)).pow(2).sum(1).mean() if len(R) > 1 else torch.tensor(float("nan"), device=DEV)
        out.append(float((g - g.mean(0)).pow(2).sum(1).mean() / vr))
    return np.array(out)


@torch.no_grad()
def stage1_agreement(ctx, s1, G, n=300):
    """H-G probe: per-class agreement a_c = E p_ens(c | x~), x~ ~ G(.|c), where the
    ensemble is the stage-(i) heads - trained on REAL local data only, before any
    synthetic data exists, and held by the server. A label-free stand-in for the
    held-out referee's fidelity; recorded next to it to test the proxy."""
    hs = build_headers(ctx, s1["feats"][0].shape[1])
    for g in ctx.groups:
        hs[g].load_state_dict(s1["hd_sd"][g])
        hs[g].eval()
    G.eval()
    out = []
    for c in range(NUM_CLASSES):
        x = G.sample(torch.full((n,), c, device=DEV, dtype=torch.long))
        p = torch.stack([hs[g](x).softmax(1) for g in ctx.groups]).mean(0)
        out.append(float(p[:, c].mean()))
    return np.array(out)


@torch.no_grad()
def feature_mnd(ctx, G, fe, n_query=1000, n_ref=600):
    """Paper Eq. (1) in feature space: rho_i = min_V d / min_S d over training
    points; > 1 means training points sit closer to synthetic than to unseen
    data (memorisation signal). Not the paper's LPIPS image-space number."""
    d = ctx.data
    allidx = torch.cat(ctx.part["parts"])
    q = compute_features(fe, d["xtr"][allidx[torch.randperm(len(allidx), device=DEV)[:n_query]]]).flatten(1)
    v = compute_features(fe, d["xtr"][ctx.part["heldout"][torch.randperm(len(ctx.part["heldout"]),
                                                                        device=DEV)[:n_ref]]]).flatten(1)
    G.eval()
    ys = torch.randint(0, NUM_CLASSES, (n_ref,), device=DEV)
    s = G.sample(ys).flatten(1)
    dv = torch.cdist(q, v).min(1).values
    ds_ = torch.cdist(q, s).min(1).values.clamp(min=1e-8)
    return float((dv / ds_).mean())


# ----- stage (iii): header training
def stage_iii(ctx, hspec, fe, hd_sd, G, feats, ys_list, feats_te, rounds, tracker, eval_every,
              gen_state=None):
    headers = build_headers(ctx, feats[0].shape[1])
    hd_sd = [dict(s) for s in hd_sd]
    C, B = NUM_CLASSES, ctx.B
    counts = ctx.part["counts"]
    graphs = GraphCache()
    full_sizes = {B, B + max(1, int(round(hspec["mix_ratio"] * B)))}

    def fwd(g, x):
        h = headers[g]
        return graphs((g, len(x)), h, h, x) if len(x) in full_sizes else h(x)
    glob = counts.sum(0)
    la = hspec["loss"] == "la"
    pool = None
    erl = int(hspec.get("erl", 0) or 0)

    def set_erl_readouts():
        # ERL: the server sets every architecture's last layer to the class-balanced LDA readout of the current
        # aggregated body, from exact sums of every client's real embeddings (each body is broadcast for this)
        # The LDA weights S^-1 mu are much larger than an SGD-trained last layer; training the body under them at
        # the usual learning rate collapses it (pilot, 16 Oct). They are therefore rescaled to the Frobenius norm of
        # the head's current last layer: same predictions (positive scaling), usual gradient size.
        for g in ctx.groups:
            headers[g].load_state_dict(hd_sd[g])
            W, b = lda_readout(headers[g], feats, ys_list, list(range(ctx.K)))
            sc = float(hd_sd[g]["fc.weight"].norm() / W.norm().clamp(min=1e-12)) / hspec.get("erl_t", 1.0)
            b = torch.where(b <= -1e4, b, b * sc)
            hd_sd[g]["fc.weight"], hd_sd[g]["fc.bias"] = W * sc, b

    def train_params(h):
        return [p for n_, p in h.named_parameters() if not (erl and n_.startswith("fc."))]
    # P1: per-class signal fraction of the synthetic features, estimated at the server without held-out data
    s_hat, allowed = None, torch.arange(C, device=DEV)
    if G is not None and hspec["ts"] > 0 and (hspec.get("sgate") or hspec.get("scsl")):
        s_hat = signal_fraction(G, feats, ys_list)
        ctx.sfrac_hat = s_hat.round(decimals=3).tolist()
        if hspec.get("sgate"):
            allowed = torch.nonzero(s_hat >= hspec["sgate"]).flatten()
            ctx.sgate_kept = int(len(allowed))
    no_syn = len(allowed) == 0
    for r in range(rounds):
        if erl and r % erl == 0:
            set_erl_readouts()
        if gen_state is not None and hspec["gen_update"]:
            # ET.2: keep training the generator during stage (iii)
            g_sd, clients, gspec, W = gen_state
            avg = Averager(G.cond if gspec["agg"] == "A" else None, prev=g_sd)
            for k in range(ctx.K):
                G.load_state_dict(g_sd)
                gen_local_update(G, clients[k], gspec, feats[k], ys_list[k])
                avg.add(G.state_dict(), W[k] if gspec["agg"] == "A" else None)
            g_sd = avg.result()
            G.load_state_dict(g_sd)
            gen_state = (g_sd, clients, gspec, W)
        if G is not None:
            G.eval()
        pool = None
        if G is not None and (hspec["csl"] or G.kind == "ddpm") and hspec["ts"] > 0 and not no_syn:
            # CSL: one shared synthetic batch per round, labelled by the consensus
            # of every architecture's current header (clients would upload only
            # these probabilities; summable under secure aggregation)
            need = [hspec["ts"] * (sz // B if sz >= B else 1) * (B if sz >= B else sz) for sz in ctx.sizes]
            if ctx.K == len(ctx.groups):
                # one client per architecture: clients may share samples without
                # correlating two heads of the same architecture before averaging
                P, pool_off = max(need), [0] * ctx.K
            else:
                # several clients per architecture: disjoint fresh slices, so the
                # synthetic diversity per architecture matches per-client sampling
                pool_off = list(np.cumsum([0] + need[:-1]))
                P = int(sum(need))
            py = allowed[torch.randint(0, len(allowed), (P,), device=DEV)]
            px = torch.cat([G.sample(py[s:s + 4096]) for s in range(0, P, 4096)])
            # (DDPM-F without CSL: the same shared pool with one-hot targets. Each
            # head still sees fresh uniform-label samples every round - exactly the
            # reference's per-client draw in distribution - at 1/K of the cost.)
            with torch.no_grad():
                cons = torch.zeros(P, C, device=DEV)
                for g in (ctx.groups if hspec["csl"] else []):
                    headers[g].load_state_dict(hd_sd[g])
                    headers[g].eval()
                    cons += torch.cat([headers[g](px[s:s + 4096]).softmax(1) for s in range(0, P, 4096)])
                    headers[g].train()
                cons /= len(ctx.groups)
            beta = hspec["csl"]
            if hspec["csl"] and hspec["csl_mode"] == "bayes":
                # Bayesian CSL: posterior of a generated feature's class given both the
                # ensemble p(c|x) and the conditioning label y under symmetric generator
                # label noise q(c|y) = rho 1{c=y} + (1-rho)/C. rho is estimated without
                # real data: for a calibrated ensemble of confidence kappa = E max_c p(c|x),
                # a sample of class y has E p(y|x) = kappa and a uniformly drawn class has
                # E p(y|x) = 1/C, so the agreement a = E p(y|x) = rho kappa + (1 - rho)/C:
                # rho = (a - 1/C) / (kappa - 1/C).
                a = cons[torch.arange(P, device=DEV), py].mean()
                kappa = cons.max(1).values.mean()
                rho = ((a - 1.0 / C) / (kappa - 1.0 / C).clamp(min=1e-6)).clamp(0.0, 1.0)
                tgt = cons * (rho * F.one_hot(py, C).float() + (1 - rho) / C)
                pool = (px, py, tgt / tgt.sum(1, keepdim=True).clamp(min=1e-12))
                ctx.__dict__.setdefault("rho_hist", []).append(round(float(rho), 4))
                ctx.__dict__.setdefault("kappa_hist", []).append(round(float(kappa), 4))
            elif s_hat is not None and hspec.get("scsl"):
                bc = (1.0 - s_hat)[py][:, None]  # per-class consensus weight: pure ensemble label where s_c = 0
                pool = (px, py, (1 - bc) * F.one_hot(py, C).float() + bc * cons)
            else:
                pool = (px, py, (1 - beta) * F.one_hot(py, C).float() + beta * cons)
        hd_avg = {g: Averager() for g in ctx.groups}
        for k in ctx.active(r, "iii"):
            g = ctx.arch[k]
            h = headers[g]
            h.load_state_dict(hd_sd[g])
            h.train()
            h.fc.requires_grad_(not erl)  # ERL: the last layer is fixed by the server, only the body trains
            Xk, Yk = feats[k], ys_list[k]
            n = len(Yk)
            nb, bs = (n // B, B) if n >= B else (1, n)
            # synthetic label distribution for this client
            if hspec["sampler"] == "gapfill":
                p_syn = waterfill(hspec["tr"] * counts[k], hspec["ts"] * nb * bs)
            elif hspec["sampler"] == "gapfill_global":
                p_syn = waterfill(hspec["tr"] * n * glob / glob.sum(), hspec["ts"] * nb * bs)
            else:
                p_syn = np.zeros(C)
                p_syn[allowed.cpu().numpy()] = 1.0 / len(allowed) if len(allowed) else 0.0
                if not len(allowed):
                    p_syn[:] = 1.0 / C
            nsyn_k = hspec["ts"] * nb * bs
            # LA prior = the client's actual training mixture
            prior = None
            if la:
                if G is not None and hspec["mix"] == "pcm":
                    S = hspec["mix_ratio"]
                    q = counts[k] / max(n, 1)
                    p_mix = waterfill(q, S)  # expected per-batch completion
                    prior = torch.tensor(np.log((q + S * p_mix) / (1 + S) + 1e-3), dtype=torch.float32, device=DEV)
                else:
                    prior = local_log_prior(counts[k])
            adj = client_loss_adj(hspec, counts[k])

            def syn_phase():
                if G is None or hspec["ts"] <= 0 or no_syn:
                    return
                opt = torch.optim.SGD(train_params(h), lr=0.1, momentum=0.0)
                if pool is not None:
                    nsyn = hspec["ts"] * nb * bs
                    o = int(pool_off[k])
                    xs, tgt = pool[0][o:o + nsyn], pool[2][o:o + nsyn]
                    for s in range(0, nsyn, bs):
                        logp = fwd(g, xs[s:s + bs]).log_softmax(1)
                        loss = -(tgt[s:s + bs] * logp).sum(1).mean()
                        opt.zero_grad(set_to_none=True)
                        loss.backward()
                        opt.step()
                    return
                ylab = torch.from_numpy(np.random.choice(C, size=hspec["ts"] * nb * bs, p=p_syn)).long().to(DEV)
                xs = torch.cat([G.sample(ylab[s:s + 4096]) for s in range(0, len(ylab), 4096)])
                for s in range(0, len(ylab), bs):
                    loss = F.cross_entropy(fwd(g, xs[s:s + bs]), ylab[s:s + bs])
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()

            def real_phase():
                opt = torch.optim.SGD(train_params(h), lr=0.1, momentum=0.0)
                for _ in range(hspec["tr"]):
                    for b in batches(n, B):
                        if b.numel() < 2:
                            continue
                        x, y = Xk[b], Yk[b]
                        if hspec["vct"] and pool is not None:
                            # VCT (new): vicinal consensus transfer. Every real feature is mixed
                            # with a partner from this round's synthetic pool (the client's own
                            # slice, exact-moment-corrected generator), and the partner's share
                            # of the target is its CONSENSUS soft label (all architectures).
                            # lambda = max(l, 1 - l), l ~ Beta(a, a): each sample stays mostly real.
                            # To second order, MixUp depends on its partners only through the class
                            # prior, means and covariances (Proposition 7), so these partners give
                            # the global-data MixUp regulariser; the consensus labels add ensemble
                            # distillation along the path from real data into the federation's
                            # knowledge. No real feature or head leaves the client.
                            ab = torch.full((len(y),), float(hspec["vct"]), device=DEV)
                            lam = torch.distributions.Beta(ab, ab).sample()
                            lam = torch.maximum(lam, 1 - lam)
                            j = int(pool_off[k]) + torch.randint(0, max(1, nsyn_k), (len(y),), device=DEV)
                            lv = lam.view(-1, *([1] * (x.dim() - 1)))
                            xm = lv * x + (1 - lv) * pool[0][j]
                            tgt = lam[:, None] * F.one_hot(y, C).float() + (1 - lam)[:, None] * pool[2][j]
                            logits = fwd(g, xm)
                            if prior is not None and hspec["tau"]:
                                # LA with the mixed sample's own label distribution:
                                # lambda * client prior + (1 - lambda) * uniform (the pool's)
                                pk = prior.exp() / prior.exp().sum()
                                logits = logits + hspec["tau"] * torch.log(lam[:, None] * pk + (1 - lam)[:, None] / C)
                            loss = -(tgt * logits.log_softmax(1)).sum(1).mean()
                            opt.zero_grad(set_to_none=True)
                            loss.backward()
                            opt.step()
                            continue
                        if hspec["mixup"]:
                            # Control: standard MixUp (Zhang et al., 2018) on the client's own
                            # real features, l ~ Beta(a, a) per sample, partner = shuffled batch.
                            ab = torch.full((len(y),), float(hspec["mixup"]), device=DEV)
                            lam = torch.distributions.Beta(ab, ab).sample()
                            perm = torch.randperm(len(y), device=DEV)
                            lv = lam.view(-1, *([1] * (x.dim() - 1)))
                            xm = lv * x + (1 - lv) * x[perm]
                            tgt = lam[:, None] * F.one_hot(y, C).float() + (1 - lam)[:, None] * F.one_hot(y[perm], C).float()
                            logits = fwd(g, xm)
                            if prior is not None and hspec["tau"]:
                                logits = logits + hspec["tau"] * prior
                            loss = -(tgt * logits.log_softmax(1)).sum(1).mean()
                            opt.zero_grad(set_to_none=True)
                            loss.backward()
                            opt.step()
                            continue
                        if pool is not None and hspec["csl_mix"]:
                            # CSLM: a budget-matched share of consensus-labelled synthetic
                            # features in every real batch: L = CE(real) + soft-CE(synthetic)
                            S = max(1, int(round(hspec["mix_ratio"] * len(y))))
                            j = torch.randint(0, len(pool[1]), (S,), device=DEV)
                            logits = fwd(g, torch.cat([x, pool[0][j]]))
                            if prior is not None and hspec["tau"]:
                                logits = logits + hspec["tau"] * prior
                            logp = logits.log_softmax(1)
                            tgt = torch.cat([F.one_hot(y, C).float(), pool[2][j]])
                            loss = -(tgt * logp).sum(1).mean()
                            opt.zero_grad(set_to_none=True)
                            loss.backward()
                            opt.step()
                            continue
                        if G is not None and hspec["mix"] == "pcm":
                            # PCM: complete this batch's class prior with synthetic features
                            S = max(1, int(round(hspec["mix_ratio"] * len(y))))
                            bc = torch.bincount(y, minlength=C).cpu().numpy()
                            p = waterfill(bc, S)
                            ys_ = torch.from_numpy(np.random.choice(C, size=S, p=p)).long().to(DEV)
                            x = torch.cat([x, G.sample(ys_)])
                            y = torch.cat([y, ys_])
                        if hspec["rsa"] and G is not None and G.kind != "ddpm":
                            # RSA: class-matched synthetic batch, one body pass for both
                            xs_ = G.sample(y)
                            e = h.embed(torch.cat([x, xs_]))
                            er, es = e[:len(y)], e[len(y):]
                            loss = ce_or_la(h.fc(er), y, prior, hspec["tau"]) + hspec["rsa"] * rsa_loss(er, es, y, C)
                        else:
                            loss = head_loss(fwd(g, x), y, prior, hspec, adj)
                        opt.zero_grad(set_to_none=True)
                        loss.backward()
                        opt.step()

            if hspec["mix"] == "pcm" or (pool is not None and hspec["csl_mix"]):
                real_phase()
            elif hspec["order"] == "real_first":
                real_phase()
                syn_phase()
            elif ctx.inst is not None and G is not None and hspec["ts"] > 0 and not no_syn and \
                    (r % 10 == 0 or r == rounds - 1) and k == list(ctx.active(r, "iii"))[0]:
                instrument_phases(ctx, r, k, h, G, syn_phase, real_phase)
            else:
                syn_phase()
                real_phase()
            hd_avg[g].add(h.state_dict())
        for g in ctx.groups:
            if hd_avg[g].n:  # an architecture with no participating client keeps its head
                hd_sd[g] = hd_avg[g].result()
        if hspec["bcr"] and G is not None:
            # BCR: server re-fits each header's fc on class-balanced synthetic features
            if pool is None or hspec["gen_update"]:
                py = torch.arange(C, device=DEV).repeat(500)
                pool = (G.sample(py), py)
            for g in ctx.groups:
                h = headers[g]
                h.load_state_dict(hd_sd[g])
                h.eval()
                with torch.no_grad():
                    emb = h.embed(pool[0])
                opt = torch.optim.SGD(h.fc.parameters(), lr=hspec["bcr_lr"], momentum=0.0)
                for _ in range(hspec["bcr"]):
                    b = torch.randint(0, len(pool[1]), (128,), device=DEV)
                    loss = F.cross_entropy(h.fc(emb[b]), pool[1][b])
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
                hd_sd[g] = sd_clone(h)
        if hspec["sed"] and G is not None:
            # SED: server-side ensemble distillation on generated features. The
            # server already holds every architecture's aggregated header and
            # G_F; each header takes sed steps toward the CSL target
            # (1 - b) onehot(y) + b * mean_g softmax(h_g(x)) on fresh synthetic
            # features. No client computation or message is added.
            n_sed = hspec["sed"] * hspec["sed_bs"]
            sy = torch.randint(0, C, (n_sed,), device=DEV)
            with torch.no_grad():
                sx = torch.cat([G.sample(sy[s:s + 4096]) for s in range(0, n_sed, 4096)])
                cons = torch.zeros(n_sed, C, device=DEV)
                for g in ctx.groups:
                    headers[g].load_state_dict(hd_sd[g])
                    headers[g].eval()
                    cons += torch.cat([headers[g](sx[s:s + 4096]).softmax(1) for s in range(0, n_sed, 4096)])
                cons /= len(ctx.groups)
                tgt = (1 - hspec["sed_beta"]) * F.one_hot(sy, C).float() + hspec["sed_beta"] * cons
            for g in ctx.groups:
                h = headers[g]
                h.train()  # BN normalises synthetic batches by their own statistics, as on clients
                opt = torch.optim.SGD(h.parameters(), lr=hspec["sed_lr"], momentum=0.0)
                ok = True
                for s in range(0, n_sed, hspec["sed_bs"]):
                    loss = -(tgt[s:s + hspec["sed_bs"]] * h(sx[s:s + hspec["sed_bs"]]).log_softmax(1)).sum(1).mean()
                    if not torch.isfinite(loss):
                        ok = False
                        break
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
                if ok:
                    new_sd = sd_clone(h)
                    for k, v in hd_sd[g].items():  # BN running statistics stay those of real features
                        if "running_" in k or "num_batches" in k:
                            new_sd[k] = v
                    hd_sd[g] = new_sd
        if erl and r == rounds - 1:
            set_erl_readouts()  # the final last layer matches the final body
        if r == 0 or (r + 1) % eval_every == 0 or r == rounds - 1:
            for g in ctx.groups:
                headers[g].load_state_dict(hd_sd[g])
            tracker.update("header", r + 1, eval_headers(ctx, headers, feats_te, ctx.data["yte"]))
    for g in ctx.groups:
        headers[g].load_state_dict(hd_sd[g])
        headers[g].fc.requires_grad_(True)  # ERL froze it during training; later readouts re-fit it
    return headers


class _BiasedHead(nn.Module):
    def __init__(self, h, b):
        super().__init__()
        self.h = h
        self.register_buffer("b", b)

    def forward(self, x):
        return self.h(x) + self.b


@torch.no_grad()
def balanced_bias_calibration(ctx, headers, G, feats_te, n=500, iters=50):
    """BBC diagnostic: per head, the C logit offsets b with which the head predicts
    every class equally often on class-balanced synthetic features from G
    (prior correction a la Saerens et al. 2002, fixed point b <- b - log(C q(b))).
    Uses only what the server holds (heads, generator); evaluated on the test set."""
    return summarize_eval(eval_headers(ctx, bbc_heads(ctx, headers, G, n, iters), feats_te, ctx.data["yte"]))


@torch.no_grad()
def bbc_heads(ctx, headers, G, n=500, iters=50):
    G.eval()
    y = torch.arange(NUM_CLASSES, device=DEV).repeat_interleave(n)
    x = torch.cat([G.sample(y[i:i + 4096]) for i in range(0, len(y), 4096)])
    out = []
    for g in range(len(headers)):
        h = headers[g]
        if g not in ctx.groups:
            out.append(h)
            continue
        h.eval()
        logits = torch.cat([h(x[i:i + 4096]) for i in range(0, len(x), 4096)])
        b = torch.zeros(NUM_CLASSES, device=DEV)
        for _ in range(iters):
            q = (logits + b).softmax(1).mean(0)
            b = b - torch.log(q * NUM_CLASSES)
        out.append(_BiasedHead(h, b - b.mean()))
    return out


@torch.no_grad()
def synreal_gap(ctx, headers, G, feats_te, n=300, held_min=5):
    """Mechanism probe for RSA: per head g (its client's counts define held =
    count >= held_min, unheld = count == 0), the relative distance between the
    head's embedding mean of synthetic class-c features and of REAL test
    features of class c. Returns mean over heads of (held gap, unheld gap)."""
    yte = ctx.data["yte"]
    G.eval()
    syn = {c: G.sample(torch.full((n,), c, device=DEV, dtype=torch.long)) for c in range(NUM_CLASSES)}
    held, unheld = [], []
    for k in range(ctx.K):
        g = ctx.arch[k]
        h = headers[g]
        h.eval()
        cnt = ctx.part["counts"][k]
        for c in range(NUM_CLASSES):
            if cnt[c] >= held_min or cnt[c] == 0:
                er = h.embed(feats_te[yte == c][:1000]).mean(0)
                es = h.embed(syn[c]).mean(0)
                gap = float((es - er).pow(2).sum() / er.pow(2).sum().clamp(min=1e-6))
                (held if cnt[c] >= held_min else unheld).append(gap)
        if ctx.K > 10 and k >= 9:
            break
    return (float(np.mean(held)) if held else float("nan"), float(np.mean(unheld)) if unheld else float("nan"))


@torch.no_grad()
def exact_readout(ctx, headers, feats, ys_list, feats_te, gamma=0.1):
    """ER (T03): per architecture, every client embeds its real features with that head's body and
    contributes class sums, counts and sum e e^T (secure-aggregatable sums). The server sets the
    uniform-prior LDA readout w_c = S^-1 mu_c, b_c = -mu_c^T S^-1 mu_c / 2, with S the pooled
    within-class covariance shrunk by gamma toward tr(S)/d I. Closes the readout gap without real
    data leaving any client; compare with classifier_oracle (pooled-real CE refit)."""
    C = NUM_CLASSES
    import copy
    hs = {}
    for g in ctx.groups:
        h = copy.deepcopy(headers[g])
        h.eval()
        S1, n, M = None, torch.zeros(C, dtype=torch.float64, device=DEV), None
        for X, Y in zip(feats, ys_list):
            e = torch.cat([h.embed(X[i:i + 4096]) for i in range(0, len(X), 4096)]).double()
            if S1 is None:
                S1 = torch.zeros(C, e.shape[1], dtype=torch.float64, device=DEV)
                M = torch.zeros(e.shape[1], e.shape[1], dtype=torch.float64, device=DEV)
            S1.index_add_(0, Y, e)
            n += torch.bincount(Y, minlength=C).double()
            M += e.T @ e
        mu = S1 / n[:, None].clamp(min=1)
        d = mu.shape[1]
        Sw = (M - (mu.T * n) @ mu) / max(float(n.sum()) - C, 1.0)
        Sw = (Sw + Sw.T) / 2
        Ss = (1 - gamma) * Sw + gamma * torch.trace(Sw) / d * torch.eye(d, dtype=torch.float64, device=DEV)
        W = mu @ torch.linalg.inv(Ss)
        h.fc.weight.copy_(W.float())
        h.fc.bias.copy_((-0.5 * (W * mu).sum(1)).float())
        hs[g] = h
    return summarize_eval(eval_headers(ctx, hs, feats_te, ctx.data["yte"]))


def classifier_oracle(ctx, headers, feats, ys_list, feats_te, steps=300):
    """Diagnostic run on every method: re-fit each header's fc on the pooled
    federation real features with class-balanced sampling. The gap between
    this and the method's own final accuracy is what the classifier loses."""
    X, Y = torch.cat(feats), torch.cat(ys_list)
    w = 1.0 / torch.bincount(Y, minlength=NUM_CLASSES).float().clamp(min=1)
    import copy
    hs = {}
    for g in ctx.groups:
        h = copy.deepcopy(headers[g])
        h.eval()
        with torch.no_grad():
            emb = h.embed(X)
        opt = torch.optim.SGD(h.fc.parameters(), lr=0.05)
        for _ in range(steps):
            b = torch.multinomial(w[Y], 128, replacement=True)
            loss = F.cross_entropy(h.fc(emb[b]), Y[b])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        hs[g] = h
    return summarize_eval(eval_headers(ctx, hs, feats_te, ctx.data["yte"]))


# ----- post-hoc readouts: new last layers for the trained heads, computed from what a server can hold
class _FnHead(nn.Module):
    """A trained head whose last layer is replaced by fn(embedding) -> logits."""

    def __init__(self, h, fn):
        super().__init__()
        self.h, self.fn = h, fn

    def embed(self, x):
        return self.h.embed(x)

    def forward(self, x):
        return self.fn(self.h.embed(x))


@torch.no_grad()
def _embeds(h, feats, ys_list, idx):
    E = torch.cat([torch.cat([h.embed(feats[k][i:i + 4096]) for i in range(0, len(feats[k]), 4096)]) for k in idx])
    return E, torch.cat([ys_list[k] for k in idx])


@torch.no_grad()
@torch.no_grad()
def signal_fraction(G, feats, ys_list, n=200, gamma=0.1):
    """P1 (math report §6): per-class signal fraction of the synthetic features, s_c = (a_c - 1/C) / (r_c - 1/C),
    where a_c is the agreement of the class-balanced LDA on the frozen features (from exact per-class sums over all
    clients, Theorem 10) with the conditioning label of n generated class-c samples, and r_c is the same LDA's recall
    on the real training features (aggregated confusion counts), which corrects for the LDA's own error. Clipped to
    [0, 1]. No held-out data are used."""
    C = NUM_CLASSES
    X = torch.cat(feats).flatten(1).to(DEV).double()
    Y = torch.cat(ys_list).to(DEV)
    d = X.shape[1]
    cnt = torch.bincount(Y, minlength=C).double()
    mu = torch.zeros(C, d, dtype=torch.float64, device=DEV).index_add_(0, Y, X) / cnt[:, None].clamp(min=1)
    Sw = (X.T @ X - (mu.T * cnt) @ mu) / max(float(cnt.sum()) - C, 1.0)
    Ss = (1 - gamma) * Sw + (gamma * torch.trace(Sw) / d + 1e-6) * torch.eye(d, dtype=torch.float64, device=DEV)
    W = mu @ torch.linalg.pinv((Ss + Ss.T) / 2, hermitian=True)
    b = -0.5 * (W * mu).sum(1)
    del Sw, Ss

    def recall(Xs, Ys):
        pred = torch.cat([(Xs[s:s + 4096].double() @ W.T + b).argmax(1) for s in range(0, len(Ys), 4096)])
        hit = torch.zeros(C, dtype=torch.float64, device=DEV).index_add_(0, Ys, (pred == Ys).double())
        return hit / torch.bincount(Ys, minlength=C).double().clamp(min=1)
    r = recall(X, Y)
    del X
    yg = torch.arange(C, device=DEV).repeat_interleave(n)
    xg = torch.cat([G.sample(yg[s:s + 4096]) for s in range(0, len(yg), 4096)]).flatten(1)
    a = recall(xg, yg)
    s = ((a - 1.0 / C) / (r - 1.0 / C).clamp(min=1e-3)).clamp(0.0, 1.0)
    return s.float()


def lda_readout(h, feats, ys_list, idx, gamma=0.1):
    """Class-balanced LDA last layer for head h's body, from per-class sums over clients idx
    (counts, sums of e and of e e^T: secure-aggregatable). Returns (W, b); absent classes get b = -1e4."""
    C = NUM_CLASSES
    was = h.training
    h.eval()
    E, Y = _embeds(h, feats, ys_list, idx)
    h.train(was)
    d = E.shape[1]
    n = torch.bincount(Y, minlength=C).double()
    Ed = E.double()
    mu = torch.zeros(C, d, dtype=torch.float64, device=DEV).index_add_(0, Y, Ed) / n[:, None].clamp(min=1)
    absent = n == 0
    Sw = (Ed.T @ Ed - (mu.T * n) @ mu) / max(float(n.sum()) - int((~absent).sum()), 1.0)
    Sw = (Sw + Sw.T) / 2
    Ss = (1 - gamma) * Sw + (gamma * torch.trace(Sw) / d + 1e-6) * torch.eye(d, dtype=torch.float64, device=DEV)
    W = mu @ torch.linalg.pinv(Ss, hermitian=True)
    b = -0.5 * (W * mu).sum(1)
    b[absent] = -1e4
    return W.float(), b.float()


def readout_heads(ctx, headers, feats, ys_list, kind, scope):
    """Per architecture g, a new last layer for head g from the real features of its own clients
    (scope "own": the protocol of CCVR / Fed3R / FedNCM applied per architecture) or of every client
    passed through head g's body (scope "all": needs head g's body broadcast to every client, as in ER).
    Every statistic is a per-class sum (counts, sums of e, sums of e e^T), so it can be secure-aggregated;
    here the sums are formed directly.
      lda   : ER, uniform-prior LDA with the pooled within-class covariance shrunk by 0.1 (T03)
      ridge : Fed3R (Fanì et al., 2024): closed-form ridge regression onto one-hot labels
      ncm   : FedNCM (Legate et al., 2023): cosine similarity to the normalised class means
      ccvr  : CCVR (Luo et al., 2021): per-class Gaussians N(mu_c, Sigma_c) of the embeddings, an equal
              number of virtual features per class, the last layer fine-tuned on them by cross-entropy
    A class with no sample in scope gets no logit mass (lda / ncm) or no virtual features (ccvr)."""
    import copy
    C = NUM_CLASSES
    hs = {}
    for g in ctx.groups:
        h = copy.deepcopy(headers[g])
        h.eval()
        idx = list(range(ctx.K)) if scope == "all" else [k for k in range(ctx.K) if ctx.arch[k] == g]
        E, Y = _embeds(h, feats, ys_list, idx)
        d = E.shape[1]
        n = torch.bincount(Y, minlength=C).double()
        Ed = E.double()
        mu = torch.zeros(C, d, dtype=torch.float64, device=DEV).index_add_(0, Y, Ed) / n[:, None].clamp(min=1)
        absent = n == 0
        if kind == "lda":
            W, b = lda_readout(h, feats, ys_list, idx)
            with torch.no_grad():
                h.fc.weight.copy_(W)
                h.fc.bias.copy_(b)
            hs[g] = h
        elif kind == "ridge":
            Ea = torch.cat([Ed, torch.ones(len(Ed), 1, dtype=torch.float64, device=DEV)], 1)
            A = Ea.T @ Ea
            lam = 1e-2 * torch.trace(A) / A.shape[0]
            Bm = Ea.T @ F.one_hot(Y, C).double()
            W = torch.linalg.solve(A + lam * torch.eye(A.shape[0], dtype=torch.float64, device=DEV), Bm)
            with torch.no_grad():
                h.fc.weight.copy_(W[:d].T.float())
                h.fc.bias.copy_(W[d].float())
            hs[g] = h
        elif kind == "ncm":
            mn = F.normalize(mu.float(), dim=1)
            mask = torch.where(absent, torch.full((C,), -1e4, device=DEV), torch.zeros(C, device=DEV))
            hs[g] = _FnHead(h, lambda e, mn=mn, mask=mask: 10.0 * F.normalize(e, dim=1) @ mn.T + mask)
        elif kind == "ccvr":
            nv = max(100, 20000 // C)
            xs, ys = [], []
            for c in range(C):
                Ec = Ed[Y == c]
                if len(Ec) == 0:
                    continue
                z = torch.randn(nv, d, dtype=torch.float64, device=DEV)
                if len(Ec) > 1:
                    S = torch.cov(Ec.T) if d > 1 else Ec.var().view(1, 1)
                    S = (S + S.T) / 2 + 1e-5 * (torch.trace(S) / d + 1e-8) * torch.eye(d, dtype=torch.float64, device=DEV)
                    ev_, V = torch.linalg.eigh(S)
                    xs.append(mu[c] + (z * ev_.clamp(min=0).sqrt()) @ V.T)
                else:
                    xs.append(mu[c].expand(nv, d))
                ys.append(torch.full((nv,), c, dtype=torch.long, device=DEV))
            X, Yv = torch.cat(xs).float(), torch.cat(ys)
            opt = torch.optim.SGD(h.fc.parameters(), lr=0.01, momentum=0.9)
            for _ in range(10):
                for bidx in batches(len(X), 256):
                    loss = F.cross_entropy(h.fc(X[bidx]), Yv[bidx])
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
            hs[g] = h
        else:
            raise ValueError(kind)
    return hs


READOUTS = {"er": ("lda", "all"), "er_own": ("lda", "own"), "fed3r": ("ridge", "all"), "fed3r_own": ("ridge", "own"),
            "ncm": ("ncm", "all"), "ncm_own": ("ncm", "own"), "ccvr": ("ccvr", "all"), "ccvr_own": ("ccvr", "own")}
EVAL_KEYS = ("bal", "acc", "head", "medium", "tail", "worst", "worst_arch", "rec_std") + EXTRA_METRICS + ("ens_bal",)


def flat_eval(prefix, ev, recall=True):
    """{prefix}_{metric} for every scalar of summarize_eval (fractions; ece / nll / brier as they are)."""
    out = {f"{prefix}_{k}": ev[k] for k in EVAL_KEYS if ev.get(k) is not None}
    if recall:
        out[f"{prefix}_recall"] = [round(v, 4) for v in ev["recall"]]
    return out


def evaluate_trained(ctx, cfg, m, s1, headers, G, res):
    """Readouts and diagnostics on trained heads (shared by run_one and the --from_models re-evaluation).
    Always: the pooled-real oracle and ER; cfg["readouts"] adds any of READOUTS; BBC when G exists."""
    yte = ctx.data["yte"]
    # Validation set: the 2,000 held-out training images no client holds (used here as a server-side labelled
    # validation set, so settings can be chosen without the test set). Every readout is scored on it as val_*.
    hold = ctx.part["heldout"]
    if s1.get("feats_val") is None:
        s1["feats_val"] = compute_features(s1["fe"], ctx.data["xtr"][hold])
    Xv, yv = s1["feats_val"], ctx.data["ytr"][hold]

    def both(prefix, hs):
        res.update(flat_eval(prefix, summarize_eval(eval_headers(ctx, hs, s1["feats_te"], yte))))
        res.update(flat_eval("val_" + prefix, summarize_eval(eval_headers(ctx, hs, Xv, yv)), recall=False))
    res.update(flat_eval("val_final", summarize_eval(eval_headers(ctx, headers, Xv, yv)), recall=False))
    orc = classifier_oracle(ctx, headers, s1["feats"], s1["ys"], s1["feats_te"])
    res.update(flat_eval("oracle", orc))
    for nm in ["er"] + [r for r in (cfg.get("readouts") or []) if r != "er"]:
        try:  # diagnostics, never allowed to break a run
            set_seed(ctx.seed + 3000)
            hs = readout_heads(ctx, headers, s1["feats"], s1["ys"], *READOUTS[nm])
            both(nm, hs)
            if nm == "er":
                # label-free readout-gap estimate (idea 3): how often each head and its exact readout disagree on
                # UNLABELLED held-out features; needs no labels at the server
                with torch.no_grad():
                    dis = []
                    for g in ctx.groups:
                        headers[g].eval()
                        hs[g].eval()
                        dis.append(float((headers[g](Xv).argmax(1) != hs[g](Xv).argmax(1)).float().mean()))
                        headers[g].train()
                res["ro_disagree"] = float(np.mean(dis))
        except Exception as e:  # pragma: no cover
            res[f"{nm}_error"] = repr(e)[:200]
    if G is not None:
        both("bbc", bbc_heads(ctx, headers, G))
        # gate (no test data): apply BBC only when the federation's GLOBAL label
        # distribution is imbalanced, read off the exact aggregated class counts
        glob_n = ctx.part["counts"].sum(0)
        res["global_imbalance"] = float(glob_n.max() / max(glob_n.min(), 1))
        res["bbc_applied"] = bool(res["global_imbalance"] > 2.0)
        gh, gu = synreal_gap(ctx, headers, G, s1["feats_te"])
        res.update(gap_held=gh, gap_unheld=gu)


def wrap_sampler(ctx, m, G, s1, res):
    """The stage-(iii) sampler: MC-S / MC moment correction and kernel herding on top of the trained G."""
    if G is not None and m["head"].get("mcs") and not _STAT_DP:
        G = CovCalibratedGen(G, s1["feats"], s1["ys"], k=m["head"].get("mcs_k", 256), prior=m["head"].get("mcs_prior", "pooled"))
        res["mcs_k"] = G.k
        if s1.get("ref") is not None:
            fmc = referee_fidelity(s1["ref"], G)
            res["fidelity_tail_mc"], res["fidelity_mc"] = float(fmc[TAIL()].mean()), fmc.round(3).tolist()
        smc = sample_spread(G, s1["feats"], s1["ys"])
        res["spread_tail_mc"] = float(np.nanmean(smc[TAIL()]))
    elif G is not None and (m["head"].get("mc") or m["head"].get("mcs")):
        G = MomentCalibratedGen(G, s1["feats"], s1["ys"], mode=m["head"].get("mc_mode", "both"))
        if s1.get("ref") is not None:
            fmc = referee_fidelity(s1["ref"], G)
            res["fidelity_tail_mc"], res["fidelity_mc"] = float(fmc[TAIL()].mean()), fmc.round(3).tolist()
        smc = sample_spread(G, s1["feats"], s1["ys"])
        res["spread_tail_mc"], res["mc_scale"] = float(np.nanmean(smc[TAIL()])), G.scale.cpu().numpy().round(3).tolist()
        res["mc_classes_used"] = int(G.use.sum())
    if G is not None and m["head"].get("kh"):
        phi_, mu_, use_ = federated_kme(ctx, s1["feats"], s1["ys"], D=m["head"].get("kh_D", 6144))
        G = HerdedGen(G, phi_, mu_, use=use_, mult=m["head"].get("kh_mult", 4), mode=m["head"].get("kh_mode", "kh"))
        res["kh_classes_used"] = int(use_.sum())
        if s1.get("ref") is not None:
            fkh = referee_fidelity(s1["ref"], G)
            res["fidelity_tail_kh"], res["fidelity_kh"] = float(fkh[TAIL()].mean()), fkh.round(3).tolist()
        res["spread_tail_kh"] = float(np.nanmean(sample_spread(G, s1["feats"], s1["ys"])[TAIL()]))
    return G


# ---------------------------------------------------------------- experiment runner
def default_cfg():
    return dict(quick=False, seeds=[0, 1, 2], data_root="./data", out_dir="./results",
                datasets=None, only=None)


def rounds_for(cfg, ds, m):
    spec = DATASETS[ds]
    T = dict(T_FE=spec["T_FE"], T_KA=spec["T_KA"], T_TN=spec["T_TN"])
    if m["fe"]["T_FE"]:
        T["T_FE"] = m["fe"]["T_FE"]
    if m["gen"] and m["gen"]["T_KA"]:
        T["T_KA"] = m["gen"]["T_KA"]
    if cfg["quick"]:
        T = {k: max(2, int(math.ceil(v * cfg["quick_scale"]))) for k, v in T.items()}
    return T


def key_of(run, cfg, upto):
    m = run["m"]
    T = rounds_for(cfg, run["dataset"], m)
    k = [run["dataset"], run["IF"], run["alpha"], run["K"], run["seed"], m["pipeline"],
         json.dumps(m["fe"], sort_keys=True), T["T_FE"]]
    if run.get("tail_perm") or run.get("pfrac", 1.0) < 1.0:  # appended only when set: old keys are unchanged
        k += [bool(run.get("tail_perm")), run.get("pfrac", 1.0)]
    if upto >= 2:
        k += [json.dumps(m["gen"], sort_keys=True), T["T_KA"]]
    return json.dumps(k)


def cvae_part_key(run, cfg):
    """Stage-(ii) key of the plain CVAE-F that a hybrid generator contains,
    so the hybrid reuses the CVAE an ordinary run in the same setting trained."""
    g = run["m"]["gen"]
    if not g or g["type"] not in ("hybrid", "mix"):
        return None
    cv = {k: v for k, v in g.items() if k in GEN_DEFAULTS}
    cv["type"] = "cvae"
    return key_of(dict(run, m=dict(run["m"], gen=cv)), cfg, 2)


def stage_keys(run, cfg):
    keys = [key_of(run, cfg, 1)]
    if run["m"]["gen"]:
        keys.append(key_of(run, cfg, 2))
    kc = cvae_part_key(run, cfg)
    if kc:
        keys.append(kc)
    return keys


def run_one(cfg, run, cache, data_cache):
    ds, seed, K = run["dataset"], run["seed"], run["K"]
    m = run["m"]
    spec = DATASETS[ds]
    set_num_classes(ds)
    if ds not in data_cache:
        log(f"loading {ds}")
        data_cache.clear()
        data_cache[ds] = load_dataset(ds, cfg["data_root"])
    data = permute_labels(data_cache[ds], run, seed)
    frac = spec["frac"] * (cfg["quick_frac"].get(ds, 1.0) if cfg["quick"] else 1.0)
    part = make_partition(data["ytr"], K, frac, run["IF"], run["alpha"], seed)
    data, part = localize(data, part)
    ctx = Ctx(cfg, data, ds, part, K, seed)
    ctx.pfrac = run.get("pfrac", 1.0)
    ctx.inst = {} if cfg.get("instrument") else None
    T = rounds_for(cfg, ds, m)
    ev_every = 4 if cfg["quick"] else 10
    t0 = time.time()
    tracker = Tracker()
    res = dict(n_train=int(sum(ctx.sizes)), min_client=int(min(ctx.sizes)),
               tail_holders=int((part["counts"][:, TAIL()] > 0).sum(0).min()),
               client_counts=part["counts"].astype(int).tolist())  # per-client class counts (count-only collapse predictor)

    # ---- baselines without a generator: LG-FedAvg / FedAvg (grouped)
    if m["pipeline"] in ("lg_fedavg", "fedavg_grouped"):
        set_seed(seed)
        fe_sd, hd_sd, fe = stage_i(ctx, T["T_FE"] + T["T_TN"], "all" if m["pipeline"] == "lg_fedavg" else "group",
                                   la=m["head"]["loss"] == "la", tracker=tracker, eval_every=ev_every)
        res["time_s"] = time.time() - t0
        return finish(res, tracker, None)

    # ---- stage (i)
    k1 = key_of(run, cfg, 1)
    if k1 not in cache:
        set_seed(seed)
        tr1 = Tracker()
        if m["fe"]["oracle"]:
            fe_sd, hd_sd, fe = stage_i_oracle(ctx)
        else:
            fe_sd, hd_sd, fe = stage_i(ctx, T["T_FE"], "all", la=m["fe"]["la"], tracker=tr1, eval_every=ev_every,
                                        bal=m["fe"].get("bal", 0.0))
        fe.load_state_dict(fe_sd)
        feats = [compute_features(fe, data["xtr"][p]) for p in part["parts"]]
        ys_list = [data["ytr"][p] for p in part["parts"]]
        feats_te = compute_features(fe, data["xte"])
        ref = train_referee(ctx, fe) if cfg.get("referee", True) else None
        cache[k1] = dict(fe=fe, hd_sd=hd_sd, feats=feats, ys=ys_list, feats_te=feats_te, tr=tr1, ref=ref,
                         t=time.time() - t0)
        if ctx.inst is not None:
            instrument_bn(ctx, fe)
            cache[k1]["inst"] = dict(ctx.inst)
            ctx.inst = {}
    s1 = cache[k1]
    if ctx.inst is not None and s1.get("inst"):
        res["inst_fe_grad"] = s1["inst"].get("fe_grad")
        res["inst_bn_var_ratio"] = s1["inst"].get("bn_var_ratio")
    global _STAT_DP
    _STAT_DP = None
    if "sep" not in s1:
        # How much the class MEAN identifies the class in the shared feature space:
        # F = between-class variance of the exact class means / mean within-class
        # spread (computable at the server from the released sums), and the
        # nearest-class-mean accuracy on the test set (diagnostic only).
        with torch.no_grad():
            n_, S1_, S2_ = anchor_stats(s1["feats"], s1["ys"])
            mu_ = S1_ / n_[:, None]
            Vw = (S2_ / n_ - mu_.pow(2).sum(1)).clamp(min=1e-12)
            Fsep = float((mu_ - mu_.mean(0)).pow(2).sum(1).mean() / Vw.mean())
            Xt = s1["feats_te"].flatten(1)
            dist = Xt.pow(2).sum(1, keepdim=True) - 2 * Xt @ mu_.T + mu_.pow(2).sum(1)[None]
            ncm = float((dist.argmin(1) == ctx.data["yte"]).float().mean())
        s1["sep"] = (Fsep, ncm)
    res["class_sep_F"], res["ncm_acc"] = s1["sep"]
    if m["gen"] is not None and m["gen"].get("stat_dp_eps"):
        # clipping bound from the held-out pool (public data: no client holds it)
        with torch.no_grad():
            hold = compute_features(s1["fe"], ctx.data["xtr"][ctx.part["heldout"]]).flatten(1)
        clip = float(torch.quantile(hold.norm(dim=1), 0.9))
        eps_, delta_ = m["gen"]["stat_dp_eps"], m["gen"].get("stat_dp_delta", 1e-5)
        _STAT_DP = dict(clip=clip, sig1=math.sqrt(2.0 / zcdp_rho(eps_, delta_)), seed=9001 + 97 * seed + int(eps_ * 1000))
        res.update(stat_dp_eps=eps_, stat_dp_delta=delta_, stat_dp_clip=clip, stat_dp_sigma_mult=_STAT_DP["sig1"])
    tracker.best = dict(s1["tr"].best)
    tracker.hist = list(s1["tr"].hist)
    res["fe_time_s"] = s1["t"]

    # ---- generator gate (A13): decided at run time from F, which the server computes from the released class
    # sums (counts, sums of h, sums of |h|^2); no test data. The resolved generator spec equals the plain
    # PC-VAE or CVAE+HWA spec, so cached stage-(ii) generators are shared with those arms.
    if m["gen"] is not None and m["gen"].get("gate_F") is not None:
        use_rows = res["class_sep_F"] < m["gen"]["gate_F"]
        g_ = {k: v for k, v in m["gen"].items() if k != "gate_F"}
        g_.update(dict(type="cvae", agg="A") if use_rows else dict(type="pcvae"))
        m = dict(m, gen=g_)
        run = dict(run, m=m)
        res["gate_branch"] = "rows" if use_rows else "anchor"

    # ---- stage (ii)
    G, gen_state = None, None
    if m["gen"] is not None:
        k2 = key_of(run, cfg, 2)
        if k2 not in cache:
            t2 = time.time()
            n_gauss = None
            set_seed(seed + 1000)
            if m["gen"]["type"] == "gauss":
                G2, _ = stage_ii_gauss(ctx, m["gen"], s1["feats"], s1["ys"])
                g_sd = sd_clone(G2)
                nrm = cond_row_norms(G2)
                norms, clients, W = np.array([nrm, nrm]), None, None
            elif m["gen"]["type"] in ("hybrid", "mix"):
                kc = cvae_part_key(run, cfg)
                if kc not in cache:
                    cv = {k: v for k, v in m["gen"].items() if k in GEN_DEFAULTS}
                    cv["type"] = "cvae"
                    g_sd_c, norms_c, Gc_, cl_c, W_c = stage_ii(ctx, cv, s1["feats"], s1["ys"], T["T_KA"])
                    Gc_.load_state_dict(g_sd_c)
                    fid_c = referee_fidelity(s1["ref"], Gc_) if s1["ref"] is not None else None
                    cache[kc] = dict(g_sd=g_sd_c, norms=norms_c, G=Gc_, clients=cl_c, W=W_c, fid=fid_c, mnd=None,
                                     t=time.time() - t2, n_gauss=None)
                sc = cache[kc]
                Gc, norms, clients, W = sc["G"], sc["norms"], None, None
                Gc.load_state_dict(sc["g_sd"])
                Gg, n_c = stage_ii_gauss(ctx, m["gen"], s1["feats"], s1["ys"])
                # A class gets FSG when it is rare relative to the federation:
                # n_c < rel * N / C (n_min_rel); else an absolute threshold n_min.
                rel = m["gen"].get("n_min_rel")
                thr = rel * float(n_c.sum()) / NUM_CLASSES if rel else m["gen"].get("n_min", 100)
                if m["gen"]["type"] == "mix":
                    nbar = (rel or 1.0) * float(n_c.sum()) / NUM_CLASSES
                    w = (nbar / (nbar + n_c)).float()
                    G2 = MixFeatureGen(Gc, Gg, w).to(DEV)
                    n_gauss = float(w.sum())  # expected number of classes' worth of FSG samples
                else:
                    G2 = HybridFeatureGen(Gc, Gg, n_c < thr).to(DEV)
                    n_gauss = int((n_c < thr).sum())
                g_sd, clients = sd_clone(G2), None
            elif m["gen"]["oracle"]:
                g_sd, norms, G2 = stage_ii_oracle(ctx, m["gen"], s1["feats"], s1["ys"], T["T_KA"])
                clients, W = None, None
            else:
                g_sd, norms, G2, clients, W = stage_ii(ctx, m["gen"], s1["feats"], s1["ys"], T["T_KA"])
            G2.load_state_dict(g_sd)
            fid = referee_fidelity(s1["ref"], G2) if s1["ref"] is not None else None
            agree = stage1_agreement(ctx, s1, G2)
            spread = sample_spread(G2, s1["feats"], s1["ys"])
            mnd = feature_mnd(ctx, G2, s1["fe"]) if cfg.get("mnd", False) else None
            if m["gen"]["type"] == "gauss":
                n_gauss = NUM_CLASSES
            cache[k2] = dict(g_sd=g_sd, norms=norms, G=G2, clients=clients, W=W, fid=fid, mnd=mnd,
                             t=time.time() - t2, n_gauss=n_gauss, agree=agree, spread=spread)
            if ctx.inst is not None and ctx.inst.get("rows"):
                rows = ctx.inst.pop("rows")
                cache[k2]["inst_rows"] = [dict(round=r_, holder_gain=(v["hg"] / np.maximum(v["hn"], 1)).round(5).tolist(),
                                               nonholder_loss=(v["ng"] / np.maximum(v["nn"], 1)).round(5).tolist())
                                          for r_, v in sorted(rows.items())]
        s2 = cache[k2]
        if ctx.inst is not None and s2.get("inst_rows"):
            res["inst_rows"] = s2["inst_rows"]
        G = s2["G"]
        G.load_state_dict(s2["g_sd"])
        res["gen_time_s"] = s2["t"]
        if s2.get("n_gauss") is not None:
            res["n_gauss_classes"] = s2["n_gauss"]
        nr = s2["norms"]
        res["cond_norm_tail_over_head_end"] = float(nr[-1, TAIL()].mean() / max(nr[-1, HEAD()].mean(), 1e-12))
        res["cond_norm_tail_rel_init"] = float(nr[-1, TAIL()].mean() / max(nr[0, TAIL()].mean(), 1e-12))
        res["cond_norms_end"] = nr[-1].round(4).tolist()
        if s2["fid"] is not None:
            res["fidelity_head"] = float(s2["fid"][HEAD()].mean())
            res["fidelity_tail"] = float(s2["fid"][TAIL()].mean())
            res["fidelity"] = s2["fid"].round(3).tolist()
        if s2["mnd"] is not None:
            res["feature_mnd"] = s2["mnd"]
        if s2.get("agree") is not None:
            res["stage1_agree"] = s2["agree"].round(4).tolist()
        if s2.get("spread") is not None:
            res["spread_ratio"] = s2["spread"].round(4).tolist()
            res["spread_tail"] = float(np.nanmean(s2["spread"][TAIL()]))
        if m["head"]["gen_update"] and s2["clients"] is not None:
            gen_state = (dict(s2["g_sd"]), s2["clients"], m["gen"], s2["W"])
            import copy
            opt_snapshot = [[copy.deepcopy(o.state_dict()) for o in c.opts] for c in s2["clients"]]

    # ---- stage (iii)
    t3 = time.time()
    G = wrap_sampler(ctx, m, G, s1, res)
    if G is not None and cfg.get("mnd", False) and (m["head"].get("mc") or m["head"].get("mcs") or m["head"].get("kh")):
        res["feature_mnd_final"] = feature_mnd(ctx, G, s1["fe"])  # memorisation of the sampler heads train on
    set_seed(seed + 2000)
    ctx.rho_hist, ctx.kappa_hist = [], []
    ctx.sfrac_hat, ctx.sgate_kept = None, None
    if ctx.inst is not None:
        ctx.inst = {}
    headers = stage_iii(ctx, m["head"], s1["fe"], s1["hd_sd"], G, s1["feats"], s1["ys"], s1["feats_te"],
                        T["T_TN"], tracker, ev_every, gen_state)
    if ctx.rho_hist:
        res["bcsl_rho"] = [ctx.rho_hist[0], ctx.rho_hist[len(ctx.rho_hist) // 2], ctx.rho_hist[-1]]
        res["bcsl_kappa"] = [ctx.kappa_hist[0], ctx.kappa_hist[len(ctx.kappa_hist) // 2], ctx.kappa_hist[-1]]
    if ctx.inst is not None and ctx.inst.get("phases"):
        res["inst_phases"] = ctx.inst["phases"]
    if ctx.sfrac_hat is not None:
        res["sfrac_hat"] = ctx.sfrac_hat
        res["sfrac_hat_mean"] = float(np.mean(ctx.sfrac_hat))
        if ctx.sgate_kept is not None:
            res["sgate_kept"] = ctx.sgate_kept
    res["head_time_s"] = time.time() - t3
    if gen_state is not None:
        # leave the cached stage-(ii) generator and optimizer states untouched for later runs
        for c, snap in zip(s2["clients"], opt_snapshot):
            for o, sd in zip(c.opts, snap):
                o.load_state_dict(sd)
    evaluate_trained(ctx, cfg, m, s1, headers, G, res)
    if s1["ref"] is not None:
        with torch.no_grad():
            res["referee_test_acc"] = float((s1["ref"](s1["feats_te"]).argmax(1) == data["yte"]).float().mean())
    if cfg.get("save_models"):
        # trained feature extractor and heads (and optionally the generator), so later readouts, baselines and
        # attacks need no re-run; main() writes them to <out_dir>/models/ and removes this key from the record
        cpu = lambda sd: {k: v.detach().cpu() for k, v in sd.items()}
        res["_models"] = dict(fe=cpu(s1["fe"].state_dict()), fe_key=k1, heads=[cpu(h.state_dict()) for h in headers],
                              gen=cpu(s2["g_sd"]) if (cfg.get("save_gen") and m["gen"] is not None) else None,
                              gen_key=key_of(run, cfg, 2) if m["gen"] is not None else None,
                              method=m, gate_branch=res.get("gate_branch"))
    res["time_s"] = time.time() - t0
    if DEV == "cuda":
        res["gpu_peak_mb"] = round(torch.cuda.max_memory_allocated() / 2 ** 20)
        torch.cuda.reset_peak_memory_stats()
    return finish(res, tracker, G)


def finish(res, tracker, G):
    last = tracker.hist[-1]
    res.update(final_acc=last["acc"], final_bal=last["bal"], final_head=last["head"], final_medium=last["medium"],
               final_tail=last["tail"], final_worst=last["worst"], best_mean_acc=last["best_mean_acc"],
               final_recall=[round(v, 4) for v in last["recall"]],
               final_ens_acc=last.get("ens_acc"), final_ens_bal=last.get("ens_bal"))
    for k in EXTRA_METRICS + ("worst_arch", "rec_std"):
        if last.get(k) is not None:
            res["final_" + k] = last[k]
    res["history"] = [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in h.items() if k != "recall"}
                      for h in tracker.hist]
    return res


def reeval_one(cfg, run, mdir, data_cache):
    """--from_models: rebuild the run's data split from its seed, load its saved FE, heads and (if saved)
    generator, and recompute every final metric and readout without training anything."""
    ds, seed, K = run["dataset"], run["seed"], run["K"]
    spec = DATASETS[ds]
    set_num_classes(ds)
    t0 = time.time()
    M = torch.load(os.path.join(mdir, "heads", safe_name(run_id(run)) + ".pt"), map_location=DEV)
    if ds not in data_cache:
        data_cache.clear()
        data_cache[ds] = load_dataset(ds, cfg["data_root"])
    data = permute_labels(data_cache[ds], run, seed)
    frac = spec["frac"] * (cfg["quick_frac"].get(ds, 1.0) if cfg["quick"] else 1.0)
    part = make_partition(data["ytr"], K, frac, run["IF"], run["alpha"], seed)
    data, part = localize(data, part)
    ctx = Ctx(cfg, data, ds, part, K, seed)
    ctx.pfrac = run.get("pfrac", 1.0)
    fe = PaperFE(spec["in_ch"], spec["fe"]).to(DEV)
    fe.load_state_dict(torch.load(os.path.join(mdir, M["fe_file"]), map_location=DEV))
    feats = [compute_features(fe, data["xtr"][p]) for p in part["parts"]]
    ys_list = [data["ytr"][p] for p in part["parts"]]
    s1 = dict(fe=fe, feats=feats, ys=ys_list, feats_te=compute_features(fe, data["xte"]), ref=None)
    headers = build_headers(ctx, feats[0].shape[1])
    for h, sd in zip(headers, M["heads"]):
        h.load_state_dict(sd)
    global _STAT_DP
    _STAT_DP = None
    m, G, res = M["method"], None, dict(reeval_of=M.get("run_id"), gate_branch=M.get("gate_branch"))
    if M.get("gen_file") and m["gen"] is not None and m["gen"]["type"] in ("cvae", "pcvae", "dcgan", "ddpm"):
        proto = class_prototypes(feats, ys_list) if m["gen"]["type"] == "pcvae" else None
        G = make_gen(ctx, m["gen"], feats[0].shape[1], feats[0].shape[1:], proto)
        G.load_state_dict(torch.load(os.path.join(mdir, M["gen_file"]), map_location=DEV))
        set_seed(seed + 2000)
        G = wrap_sampler(ctx, m, G, s1, res)
    res.update(flat_eval("final", summarize_eval(eval_headers(ctx, headers, s1["feats_te"], data["yte"]))))
    evaluate_trained(ctx, cfg, m, s1, headers, G, res)
    res["time_s"] = time.time() - t0
    return res


SCALAR_FIELDS = ["final_f1", "final_gmean", "final_ece", "final_nll", "final_worst_arch", "er_bal", "er_tail", "final_bal", "final_acc", "final_ens_acc", "final_ens_bal", "final_tail", "final_head", "final_medium", "final_worst", "n_gauss_classes",
                 "best_mean_acc", "oracle_bal", "oracle_tail", "fidelity_head", "fidelity_tail",
                 "cond_norm_tail_over_head_end", "cond_norm_tail_rel_init", "feature_mnd", "referee_test_acc",
                 "time_s", "fe_time_s", "gen_time_s", "head_time_s", "n_train", "min_client", "tail_holders"]


def run_id(run):
    extra = ("|tp" if run.get("tail_perm") else "") + (f"|p{run['pfrac']}" if run.get("pfrac", 1.0) < 1.0 else "")
    return f"{run['label']}|{run['dataset']}|IF{run['IF']}|a{run['alpha']}|K{run['K']}|s{run['seed']}" + extra


def permute_labels(data, run, seed):
    """X10: which classes are rare is drawn at random per seed. The labels are relabelled by a seeded permutation
    before the partition, so the long-tail profile (and every head / tail metric, defined on label indices) applies
    to randomly chosen original classes."""
    if not run.get("tail_perm"):
        return data
    C = int(data["ytr"].max()) + 1
    perm = torch.from_numpy(np.random.RandomState(1000 + seed).permutation(C)).long()
    return dict(data, ytr=perm.to(data["ytr"].device)[data["ytr"]], yte=perm.to(data["yte"].device)[data["yte"]])


def safe_name(rid):
    return "".join(ch if ch.isalnum() or ch in "-_.=" else "_" for ch in rid)


def save_models(mroot, rid, models, res):
    """Models go to their own folder tree (default <out_dir>/../models/<exp>/), separate from the results,
    so it can be zipped and re-uploaded on its own (e.g. as a Kaggle dataset) and used with --from_models:
        fe/<dataset>_s<seed>_<hash>.pt     feature extractor (one per stage-(i) setting, shared by arms)
        gen/<dataset>_s<seed>_<hash>.pt    trained generator before MC / KH (one per stage-(ii) setting)
        heads/<run id>.pt                   the run's ten heads + method + the names of its FE and generator files
        index.jsonl                         one line per run: run id -> files"""
    import hashlib
    hk = lambda k: hashlib.sha1(k.encode()).hexdigest()[:12]
    tag = f"{res.get('_ds', 'x')}_s{res.get('_seed', 0)}"
    for sub in ("fe", "gen", "heads"):
        os.makedirs(os.path.join(mroot, sub), exist_ok=True)
    fe_file = f"fe/{tag}_{hk(models['fe_key'])}.pt"
    if not os.path.exists(os.path.join(mroot, fe_file)):
        torch.save(models["fe"], os.path.join(mroot, fe_file))
    gen_file = None
    if models.get("gen") is not None:
        gen_file = f"gen/{tag}_{hk(models['gen_key'])}.pt"
        if not os.path.exists(os.path.join(mroot, gen_file)):
            torch.save(models["gen"], os.path.join(mroot, gen_file))
    heads_file = f"heads/{safe_name(rid)}.pt"
    torch.save(dict(run_id=rid, heads=models["heads"], method=models["method"], gate_branch=models["gate_branch"],
                    fe_file=fe_file, gen_file=gen_file, code_commit=code_commit()), os.path.join(mroot, heads_file))
    with open(os.path.join(mroot, "index.jsonl"), "a", encoding="utf-8") as f:
        f.write(json.dumps(dict(run_id=rid, heads=heads_file, fe=fe_file, gen=gen_file)) + "\n")
    return heads_file


def group_key(run):
    return (run["dataset"], run["IF"], run["alpha"], run["K"])


def paired_t(a, b):
    d = np.asarray(a) - np.asarray(b)
    if len(d) < 2 or d.std(ddof=1) == 0:
        return float("nan")
    t = d.mean() / (d.std(ddof=1) / math.sqrt(len(d)))
    try:
        from scipy import stats
        return float(2 * stats.t.sf(abs(t), len(d) - 1))
    except Exception:
        return float("nan")


def _pct_scale(mtr):
    """Metrics stored as fractions are printed in %; counts, ratios and F as they are."""
    if mtr.endswith(("_nll", "_brier")):
        return 1.0
    return 100.0 if mtr.startswith(("final", "best", "oracle", "fidelity", "referee", "bbc", "method", "ncm", "er_",
                                    "fed3r", "ccvr")) else 1.0


def write_summary(exp, rows, out_dir, cfg):
    """Mean +/- std over seeds per (setting, label); paired t-test (by seed)
    against the experiment's reference label within each setting."""
    by = defaultdict(lambda: defaultdict(dict))
    for r in rows:
        by[(r["dataset"], r["IF"], r["alpha"], r["K"])][r["label"]][r["seed"]] = r
    ref = exp.get("reference")
    metrics = exp.get("metrics", ["final_bal", "final_tail", "final_worst", "best_mean_acc", "oracle_bal"])
    lines = [f"# {exp['name']}: {exp['title']}", "", f"Mode: {'QUICK (reduced rounds; viability only)' if cfg['quick'] else 'full'}",
             "", "Hypothesis: " + exp.get("hypothesis", ""), ""]
    for setting, labels in by.items():
        ds, IF, alpha, K = setting
        lines.append(f"## {ds}  IF={IF}  alpha={alpha}  K={K}")
        paper = (DATASETS[ds]["paper"] or {}).get("cvae")
        if paper and IF == 1.0 and alpha is None and K in (10, 50, 100):
            lines.append(f"Paper Figure 4 GeFL-F (CVAE-F): {paper[[10, 50, 100].index(K)]}%")
        lines.append("")
        lines.append("| label | n | " + " | ".join(metrics) + (f" | p vs {ref} ({metrics[0]})" if ref else "") + " |")
        lines.append("|" + "---|" * (len(metrics) + 2 + (1 if ref else 0)))
        for lab, seeds in labels.items():
            vals = []
            for mtr in metrics:
                xs = [s[mtr] for s in seeds.values() if s.get(mtr) is not None and s.get(mtr) == s.get(mtr)]
                if not xs:
                    vals.append("-")
                    continue
                xs = np.array(xs) * _pct_scale(mtr)
                vals.append(f"{xs.mean():.2f} ± {xs.std(ddof=1) if len(xs) > 1 else 0:.2f}")
            pcol = ""
            if ref:
                if lab == ref or ref not in labels:
                    pcol = " | -"
                else:
                    common = sorted(set(seeds) & set(labels[ref]))
                    p = paired_t([seeds[s][metrics[0]] for s in common], [labels[ref][s][metrics[0]] for s in common])
                    pcol = f" | {p:.3g}" if p == p else " | n<2"
            lines.append(f"| {lab} | {len(seeds)} | " + " | ".join(vals) + pcol + " |")
        lines.append("")
        # per-seed values, so a pasted summary still allows paired tests
        lines.append("| label | seed | " + " | ".join(metrics) + " |")
        lines.append("|" + "---|" * (len(metrics) + 2))
        for lab, seeds in labels.items():
            for sd in sorted(seeds):
                cells = []
                for mtr in metrics:
                    v = seeds[sd].get(mtr)
                    cells.append("-" if v is None or v != v else f"{v * _pct_scale(mtr):.2f}")
                lines.append(f"| {lab} | {sd} | " + " | ".join(cells) + " |")
        lines.append("")
    path = os.path.join(out_dir, "summary.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    return path


def code_commit():
    """Commit of the code that produced a run: BUILD_COMMIT is written into built scripts by build.py
    (Kaggle has no git); otherwise ask git."""
    c = globals().get("BUILD_COMMIT")
    if c:
        return c
    try:
        import subprocess
        here = os.path.dirname(os.path.abspath(__file__))
        h = subprocess.run(["git", "rev-parse", "--short=12", "HEAD"], cwd=here, capture_output=True, text=True).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain", "--", "."], cwd=here, capture_output=True, text=True).stdout.strip()
        return (h + ("+dirty" if dirty else "")) or "unknown"
    except Exception:
        return "unknown"


def main(exp):
    ap = argparse.ArgumentParser(description=exp["title"])
    ap.add_argument("--quick", action="store_true", help="reduced rounds, 1 seed: viability check")
    ap.add_argument("--quick_scale", type=float, default=0.2, help="round multiplier in --quick")
    ap.add_argument("--seeds", type=int, nargs="+", default=None)
    ap.add_argument("--data_root", default="./data")
    ap.add_argument("--out_dir", default="./results")
    ap.add_argument("--datasets", nargs="+", default=None)
    ap.add_argument("--only", nargs="+", default=None, help="run only these labels")
    ap.add_argument("--exclude", nargs="+", default=None, help="skip these labels")
    ap.add_argument("--K", type=int, nargs="+", default=None, help="run only these client counts")
    ap.add_argument("--IF", type=float, nargs="+", default=None, help="run only these imbalance factors (1.0 = balanced)")
    ap.add_argument("--list", action="store_true", help="print the planned runs and exit")
    ap.add_argument("--gpus", type=int, default=None,
                    help="parallel GPU workers (default: all visible GPUs, e.g. 2 on Kaggle T4 x2)")
    ap.add_argument("--worker", type=int, default=None, help=argparse.SUPPRESS)
    ap.add_argument("--deterministic", action="store_true",
                    help="deterministic cuDNN/cuBLAS kernels (slower); same seed then gives the same run")
    ap.add_argument("--no_save_models", action="store_true", help="do not save trained FE + heads per run")
    ap.add_argument("--save_gen", action="store_true", help="also save the stage-(ii) generator (~25 MB, once per setting)")
    ap.add_argument("--models_dir", default=None,
                    help="where models go (default: <out_dir>/../models); kept apart from the results")
    ap.add_argument("--from_models", default=None,
                    help="folder models/<exp> from an earlier run: re-evaluate its saved models, train nothing")
    # labels such as "-CSL" start with a dash; protect them after --only / --exclude so argparse does not read them
    # as options
    argv, in_list = [], False
    for t in sys.argv[1:]:
        if t in ("--only", "--exclude"):
            in_list = True
        elif in_list and t.startswith("-") and t not in ap._option_string_actions:
            t = "LABEL:" + t
        elif t.startswith("-"):
            in_list = False
        argv.append(t)
    a, _unknown = ap.parse_known_args(argv)  # tolerate Jupyter/Kaggle kernel arguments
    for key in ("only", "exclude"):
        if getattr(a, key):
            setattr(a, key, [x[len("LABEL:"):] if x.startswith("LABEL:") else x for x in getattr(a, key)])
    if a.deterministic or os.environ.get("GEFL_DETERMINISTIC") == "1":
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=True)
    cfg = dict(quick=a.quick, quick_scale=a.quick_scale, quick_frac={"cifar10": 0.2, "svhn": 0.5},
               seeds=a.seeds if a.seeds is not None else ([0] if a.quick else exp.get("seeds", [0, 1, 2])),
               data_root=a.data_root, referee=exp.get("referee", True), mnd=exp.get("mnd", False),
               save_models=not a.no_save_models, save_gen=a.save_gen or exp.get("save_gen", False),
               readouts=exp.get("readouts"), instrument=exp.get("instrument", False))
    runs = exp["plan"](cfg)
    if a.datasets:
        runs = [r for r in runs if r["dataset"] in a.datasets]
    if a.only:
        runs = [r for r in runs if r["label"] in a.only]
    if a.exclude:
        runs = [r for r in runs if r["label"] not in a.exclude]
    if a.K:
        runs = [r for r in runs if r["K"] in a.K]
    if a.IF:
        runs = [r for r in runs if r["IF"] in a.IF]
    out_dir = os.path.join(a.out_dir, exp["name"] + ("_quick" if a.quick else "") + ("_reeval" if a.from_models else ""))
    models_dir = a.models_dir or os.path.join(os.path.dirname(os.path.abspath(a.out_dir)), "models")
    mroot = os.path.join(models_dir, exp["name"] + ("_quick" if a.quick else ""))
    os.makedirs(out_dir, exist_ok=True)
    jl = os.path.join(out_dir, "runs.jsonl")
    done = {}
    if os.path.exists(jl):
        for line in open(jl, encoding="utf-8"):
            r = json.loads(line)
            done[r["run_id"]] = r
    if a.list:
        for r in runs:
            print(("done " if run_id(r) in done else "todo ") + run_id(r))
        return
    # order so that cached stages are reused, then free them when unused
    runs.sort(key=lambda r: (key_of(r, cfg, 1), key_of(r, cfg, 2) if r["m"]["gen"] else ""))

    # Multi-GPU: one worker process per GPU, each taking whole (dataset,
    # setting, seed) groups so cached stages are never split across workers.
    n_gpu = a.gpus if a.gpus is not None else (torch.cuda.device_count() if DEV == "cuda" else 1)
    if a.worker is None and n_gpu > 1 and "__file__" in globals():
        import subprocess
        argv = [x for x in sys.argv[1:]]
        # download + cache every dataset ONCE, before workers start (they would
        # otherwise download into the same folder at the same time)
        for ds in sorted({r["dataset"] for r in runs}):
            log(f"preparing dataset {ds} (download + cache, once)")
            load_dataset(ds, cfg["data_root"])
        torch.cuda.empty_cache()
        procs = []
        for w in range(n_gpu):
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(w), PYTHONUNBUFFERED="1")
            logf = open(os.path.join(out_dir, f"worker{w}.log"), "a", encoding="utf-8")
            procs.append((subprocess.Popen([sys.executable, os.path.abspath(__file__)] + argv +
                                           ["--worker", str(w), "--gpus", str(n_gpu)],
                                           env=env, stdout=logf, stderr=subprocess.STDOUT), logf))
            log(f"started worker {w} on GPU {w} (log: {out_dir}/worker{w}.log)")
        # stream each worker's progress (run starts, results, errors) to stdout
        pos = [0] * n_gpu
        keys = ("] ", "bal=", "Traceback", "Error", "error", "downloading", "done in")
        while True:
            alive = any(pr.poll() is None for pr, _ in procs)
            for w in range(n_gpu):
                path = os.path.join(out_dir, f"worker{w}.log")
                try:
                    with open(path, encoding="utf-8", errors="replace") as fh:
                        fh.seek(pos[w])
                        chunk = fh.read()
                        pos[w] = fh.tell()
                except OSError:
                    continue
                for line in chunk.splitlines():
                    if any(k in line for k in keys):
                        print(f"[gpu{w}] {line}", flush=True)
            if not alive:
                break
            time.sleep(20)
        for pr, logf in procs:
            logf.close()
            if pr.returncode != 0:
                log(f"WARNING: a worker exited with code {pr.returncode}; see its log")
        rows = [json.loads(l) for l in open(jl, encoding="utf-8")] if os.path.exists(jl) else []
        ids = {run_id(r) for r in runs}
        rows = [r for r in rows if r["run_id"] in ids]
        path = write_summary(exp, rows, out_dir, cfg)
        log(f"all workers done; summary -> {path}")
        print(open(path, encoding="utf-8").read())
        return
    if a.worker is not None and n_gpu > 1:
        # split_by_gen: runs that train different generators may go to different
        # GPUs (each worker then repeats the cheap FE stage); needed when one
        # seed's generators are hours each (DDPM-F on CIFAR-10).
        gkey = (lambda r: [r["dataset"], r["IF"], r["alpha"], r["K"], r["seed"]] +
                ([key_of(r, cfg, 2) if r["m"]["gen"] else ""] if exp.get("split_by_gen") else []))
        groups = sorted({json.dumps(gkey(r)) for r in runs})
        mine = set(groups[a.worker::n_gpu])
        runs = [r for r in runs if json.dumps(gkey(r)) in mine]
    remaining = defaultdict(int)
    for r in runs:
        if run_id(r) not in done:
            for kk in stage_keys(r, cfg):
                remaining[kk] += 1
    log(f"{exp['name']}: {len(runs)} runs, {len(runs) - sum(run_id(r) in done for r in runs)} to do -> {out_dir}")
    cache, data_cache = {}, {}
    rows = [done[run_id(r)] for r in runs if run_id(r) in done]
    t_exp = time.time()
    deadline = float(os.environ.get("GEFL_DEADLINE", "0") or 0)  # epoch seconds; set by combined Kaggle notebooks
    for i, r in enumerate(runs):
        rid = run_id(r)
        if rid in done:
            continue
        if deadline and time.time() > deadline:
            log(f"[{i + 1}/{len(runs)}] {rid}: SKIPPED (time budget reached; it can be run in a later session)")
            continue
        log(f"[{i + 1}/{len(runs)}] {rid}")
        if a.from_models:
            if not os.path.exists(os.path.join(a.from_models, "heads", safe_name(rid) + ".pt")):
                log("    no saved models for this run; skipped")
                continue
            res = reeval_one(cfg, r, a.from_models, data_cache)
            res.setdefault("best_mean_acc", None)
        else:
            res = run_one(cfg, r, cache, data_cache)
        models = res.pop("_models", None)
        if models is not None:
            res["_ds"], res["_seed"] = r["dataset"], r["seed"]
            res["models_file"] = save_models(mroot, rid, models, res)
            del res["_ds"], res["_seed"]
        row = dict(run_id=rid, exp=exp["name"], label=r["label"], dataset=r["dataset"], IF=r["IF"],
                   alpha=r["alpha"], K=r["K"], seed=r["seed"], method=r["m"], quick=cfg["quick"],
                   code_commit=code_commit(), deterministic=not torch.backends.cudnn.benchmark, **res)
        with open(jl, "a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")
        rows.append(row)
        log(f"    bal={res['final_bal'] * 100:.2f} tail={res['final_tail'] * 100:.2f} "
            f"best_mean_acc={(res['best_mean_acc'] or float('nan')) * 100:.2f} oracle_bal={res.get('oracle_bal', float('nan')) * 100:.2f} "
            f"fid_tail={res.get('fidelity_tail', float('nan')):.3f} norm_t/h={res.get('cond_norm_tail_over_head_end', float('nan')):.3f} "
            f"time={res['time_s']:.0f}s")
        for kk in stage_keys(r, cfg):
            remaining[kk] -= 1
            if remaining[kk] <= 0 and kk in cache:
                del cache[kk]
        torch.cuda.empty_cache()
    with open(os.path.join(out_dir, "runs.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["run_id", "label", "dataset", "IF", "alpha", "K", "seed"] + SCALAR_FIELDS)
        for row in rows:
            w.writerow([row["run_id"], row["label"], row["dataset"], row["IF"], row["alpha"], row["K"], row["seed"]]
                       + [row.get(k, "") for k in SCALAR_FIELDS])
    path = write_summary(exp, rows, out_dir, cfg)
    log(f"done in {time.time() - t_exp:.0f}s; summary -> {path}")
    print(open(path, encoding="utf-8").read())


In [ ]:
%%writefile fedembed.py
"""E40 / E41 - Does the rare-row collapse of a FedAvg-trained generator happen in federated embedding models in general?

The mechanism of paper §3 needs only three things: a parameter row that receives a data gradient only from samples
that touch it (an embedding row), clients that never touch it, and local Adam with coupled (L2) weight decay,
followed by plain FedAvg. A non-holder's gradient on the row is then exactly lambda * r, which Adam normalises to a
step of size ~eta toward zero on every local step; averaging puts those shrunk copies back into the global row.
Approximation 5 predicts collapse from counts alone: kappa_c = sum_{k holds c} tau_k f_kc / sum_{k lacks c} tau_k
(tau_k: local steps; f_kc: probability that a batch of client k contains item c). Nothing here uses GeFL code.

Task "mf" (E40): federated matrix factorisation on MovieLens-1M explicit ratings (FedMF-style): item embeddings and
item biases are shared and aggregated, user embeddings and biases stay on their client. Users are split at random
into K clients (cross-silo groups). Ratings: 90 / 10 train / test per user.

Arms (same split, same initialisation per seed):
  adam_l2       local torch.optim.Adam(weight_decay=lambda) (coupled L2: the common choice), FedAvg by n_k
  adamw         local AdamW, decoupled decay lambda                                         , FedAvg by n_k
  adam          local Adam without decay                                                    , FedAvg by n_k
  adam_l2_hwa   as adam_l2, but each item row averaged over the clients that hold the item, weighted by their count
  sgd_l2        local SGD with coupled L2 (no normalisation)                                , FedAvg by n_k
  adam_l2_reset as adam_l2 with each client's Adam state reset every round (another common implementation)
  central_l2    centralised Adam + L2 on the pooled data (same number of epochs)

Task "lm" (E41): federated next-word prediction on WikiText-2 (raw), clients = random groups of Wikipedia articles,
so rare words live in few clients. Model: the previous 3 words' input embeddings (shared table, the rows we study)
-> hidden layer -> output layer; trained with a sampled softmax (the batch's targets plus 1,024 uniform negatives;
every output row is touched often on every client), evaluated with the full softmax. Vocabulary: the 10,000 most
frequent training words (others -> <unk>). Rows: input embeddings; holder counts = windows whose context contains
the word. Arms as for "mf" (HWA averages each input row over the clients whose windows contain the word).
E1-E4 below apply to E41 with "item" -> "word" and test RMSE -> test perplexity on windows whose context contains a
word with kappa < 1.

PRE-REGISTERED (19 Oct 2026, before any run):
  E1  under adam_l2, every item row with kappa_c < 1 ends at <= 0.15 of its initial norm; Spearman(log kappa, final
      norm) >= 0.8 over items with a non-holder.
  E2  no item row with kappa_c < 1 collapses (<= 0.15) under adamw, adam, adam_l2_hwa or central_l2.
  E3  the collapse costs accuracy where it happens: test RMSE on ratings of tail items (the items with kappa_c < 1)
      is higher under adam_l2 than under adamw / adam_l2_hwa, at every K.
  E4  the fraction of collapsed items grows with K (more non-holders per item).
"""
import argparse, json, math, os, time
import numpy as np
import torch
import torch.nn.functional as F

ROOT = os.environ.get("GEFL_DATA_ROOT") or os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")


def hyper_present(n, m, b):
    """P(a batch of b drawn without replacement from n items contains at least one of m marked items)."""
    if m <= 0:
        return 0.0
    if n - m < b:
        return 1.0
    return 1.0 - math.exp(math.lgamma(n - m + 1) - math.lgamma(n - m - b + 1) - math.lgamma(n + 1) + math.lgamma(n - b + 1))


def kappa(counts, tau, B, with_fbar=False):
    """counts: (K, V) per-client row usage; tau: (K,) local steps. Returns kappa per row (inf if no non-holder) and,
    with with_fbar, fbar per row: the share of all local optimiser steps whose batch carries data for the row (the
    step-frequency predictor that also covers centralised training, K = 1)."""
    K, V = counts.shape
    nk = counts.sum(1)
    out, fbar = np.empty(V), np.empty(V)
    for c in range(V):
        H = counts[:, c] > 0
        f = np.array([hyper_present(int(nk[k]), int(counts[k, c]), min(B, int(nk[k]))) for k in np.nonzero(H)[0]])
        fbar[c] = (tau[H] * f).sum() / tau.sum()
        out[c] = np.inf if H.all() else (tau[H] * f).sum() / tau[~H].sum()
    return (out, fbar) if with_fbar else out


# ------------------------------------------------------------------ MovieLens-1M federated MF
def fetch(path, url):
    """Download a dataset file once (MovieLens from GroupLens, WikiText-2 parquet from Hugging Face)."""
    if os.path.exists(path):
        return
    import urllib.request, zipfile
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = f"{path}.part{os.getpid()}"  # two notebook workers may download at once: private temp file, atomic rename
    urllib.request.urlretrieve(url, tmp)
    if url.endswith(".zip"):
        with zipfile.ZipFile(tmp) as z:
            z.extractall(os.path.dirname(os.path.dirname(path)))
        os.remove(tmp)
    else:
        os.replace(tmp, path)


def load_ml1m(seed):
    path = os.path.join(ROOT, "ml-1m", "ratings.dat")
    fetch(path, "https://files.grouplens.org/datasets/movielens/ml-1m.zip")
    a = np.array([list(map(int, l.split("::")[:3])) for l in open(path)], dtype=np.int64)
    u, i, r = a[:, 0], a[:, 1], a[:, 2].astype(np.float32)
    _, u = np.unique(u, return_inverse=True)
    _, i = np.unique(i, return_inverse=True)
    rng = np.random.RandomState(seed)
    test = np.zeros(len(u), bool)  # 10% of each user's ratings (rounded up) go to the test set, chosen at random
    order = rng.permutation(len(u))
    seen = {}
    us = u[order]
    cnt = np.bincount(u)
    take = np.ceil(0.1 * cnt).astype(int)
    for j, uu in zip(order, us):
        s = seen.get(uu, 0)
        if s < take[uu]:
            test[j] = True
            seen[uu] = s + 1
    return u, i, r, test


def run_mf(arm, K, seed, rounds=60, epochs=1, d=32, B=256, lr=1e-3, lam=1e-4, log_every=5):
    u, i, r, test = load_ml1m(seed)
    U, V = u.max() + 1, i.max() + 1
    mu = float(r[~test].mean())
    rng = np.random.RandomState(seed + 1)
    client_of_user = rng.randint(0, K, size=U)
    tr = ~test
    cl = client_of_user[u]
    torch.manual_seed(seed)
    item0 = 0.1 * torch.randn(V, d)
    user = 0.1 * torch.randn(U, d)
    ib0, ub = torch.zeros(V), torch.zeros(U)
    item, ib = item0.clone(), ib0.clone()
    U_t, I_t, R_t = torch.from_numpy(u), torch.from_numpy(i), torch.from_numpy(r)
    data = [np.nonzero(tr & (cl == k))[0] for k in range(K)]
    counts = np.zeros((K, V), np.int64)
    for k in range(K):
        counts[k] = np.bincount(i[data[k]], minlength=V)
    n_k = np.array([len(x) for x in data], float)
    tau = np.array([epochs * math.ceil(n / B) for n in n_k], float)
    kap, fbar = kappa(counts, tau, B, with_fbar=True)
    init_norm = item0.norm(dim=1).numpy()

    def make_opt(params):
        if arm in ("adam_l2", "adam_l2_hwa", "adam_l2_reset", "central_l2"):
            return torch.optim.Adam(params, lr=lr, weight_decay=lam)
        if arm == "adamw":
            return torch.optim.AdamW(params, lr=lr, weight_decay=lam)
        if arm == "adam":
            return torch.optim.Adam(params, lr=lr)
        if arm == "sgd_l2":
            return torch.optim.SGD(params, lr=0.05, weight_decay=lam)
        raise ValueError(arm)

    def step(ii, uu, rr, it, ibb, us, ubb, opt):
        pred = mu + (it[ii] * us[uu]).sum(1) + ibb[ii] + ubb[uu]
        loss = F.mse_loss(pred, rr)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    hist = []
    t0 = time.time()
    if arm == "central_l2":
        it, ibb = item.clone().requires_grad_(), ib.clone().requires_grad_()
        us, ubb = user.clone().requires_grad_(), ub.clone().requires_grad_()
        opt = make_opt([it, ibb, us, ubb])
        allidx = np.nonzero(tr)[0]
        for rd in range(rounds):
            for _ in range(epochs):
                perm = rng.permutation(allidx)
                for s in range(0, len(perm), B):
                    b = perm[s:s + B]
                    step(I_t[b], U_t[b], R_t[b], it, ibb, us, ubb, opt)
        item, ib, user, ub = it.detach(), ibb.detach(), us.detach(), ubb.detach()
    else:
        states = [None] * K  # persistent per-client optimiser state (except adam_l2_reset)
        for rd in range(rounds):
            acc_i, acc_b, wsum = torch.zeros(V, d), torch.zeros(V), 0.0
            hw_i, hw_b, hw_w = torch.zeros(V, d), torch.zeros(V), torch.zeros(V)
            for k in range(K):
                it, ibb = item.clone().requires_grad_(), ib.clone().requires_grad_()
                mine = client_of_user == k
                us, ubb = user.clone().requires_grad_(), ub.clone().requires_grad_()
                opt = make_opt([it, ibb, us, ubb])
                if states[k] is not None and arm != "adam_l2_reset":
                    opt.load_state_dict(states[k])
                for _ in range(epochs):
                    perm = rng.permutation(data[k])
                    for s in range(0, len(perm), B):
                        b = perm[s:s + B]
                        step(I_t[b], U_t[b], R_t[b], it, ibb, us, ubb, opt)
                states[k] = opt.state_dict()
                # user rows stay local: only this client's users change
                m = torch.from_numpy(mine)
                user[m], ub[m] = us.detach()[m], ubb.detach()[m]
                acc_i += n_k[k] * it.detach()
                acc_b += n_k[k] * ibb.detach()
                wsum += n_k[k]
                w = torch.from_numpy(counts[k]).float()
                hw_i += w[:, None] * it.detach()
                hw_b += w * ibb.detach()
                hw_w += w
            if arm == "adam_l2_hwa":
                held = hw_w > 0
                item = item.clone(); ib = ib.clone()
                item[held] = hw_i[held] / hw_w[held, None]
                ib[held] = hw_b[held] / hw_w[held]
            else:
                item, ib = acc_i / wsum, acc_b / wsum
            if rd % log_every == 0 or rd == rounds - 1:
                hist.append(dict(round=rd, median_norm_ratio=float(np.median(item.norm(dim=1).numpy() / init_norm))))
    # evaluation
    with torch.no_grad():
        te = np.nonzero(test)[0]
        pred = (mu + (item[I_t[te]] * user[U_t[te]]).sum(1) + ib[I_t[te]] + ub[U_t[te]]).clamp(1, 5)
        err = (pred - R_t[te]) ** 2
    norm = item.norm(dim=1).numpy() / init_norm
    ntr = counts.sum(0)
    fin = np.isfinite(kap)
    lo = kap < 1
    te_items = i[te]
    tail_mask = np.isin(te_items, np.nonzero(lo)[0])
    pop = np.isin(te_items, np.nonzero(~lo)[0])
    from scipy.stats import spearmanr
    res = dict(task="mf", arm=arm, K=K, seed=seed, rounds=rounds, lam=lam, lr=lr, d=d, B=B,
               rmse=float(err.mean().sqrt()), rmse_kappa_lt1=float(err[torch.from_numpy(tail_mask)].mean().sqrt())
               if tail_mask.any() else None, rmse_kappa_ge1=float(err[torch.from_numpy(pop)].mean().sqrt()),
               n_test_kappa_lt1=int(tail_mask.sum()), n_items=int(V), n_kappa_lt1=int(lo.sum()),
               collapsed_kappa_lt1=int((norm[lo] <= 0.15).sum()), collapsed_kappa_ge1=int((norm[~lo] <= 0.15).sum()),
               max_norm_kappa_lt1=float(norm[lo].max()) if lo.any() else None,
               median_norm_kappa_lt1=float(np.median(norm[lo])) if lo.any() else None,
               median_norm_kappa_ge1=float(np.median(norm[~lo])),
               spearman_logkappa_norm=float(spearmanr(np.log(kap[fin]), norm[fin])[0]) if fin.sum() > 2 else None,
               time_s=round(time.time() - t0, 1), history=hist,
               kappa=[round(float(x), 4) if np.isfinite(x) else None for x in kap],
               norm_ratio=[round(float(x), 4) for x in norm], train_count=ntr.tolist(),
               fbar=[float(f"{x:.4g}") for x in fbar])
    return res


# ------------------------------------------------------------------ WikiText-2 federated next-word prediction
def load_wt2(V=10000, ctx=3):
    import pyarrow.parquet as pq, re, collections
    def articles(split):
        path = os.path.join(ROOT, "wikitext2", f"{split}.parquet")
        fetch(path, "https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/wikitext-2-raw-v1/"
                    f"{split}-00000-of-00001.parquet")
        t = pq.read_table(path).to_pydict()["text"]
        arts, cur = [], []
        for line in t:
            if re.match(r"^ = [^=].* = \n$", line):
                if cur:
                    arts.append(cur)
                cur = []
            cur += line.split() + ["<eos>"] if line.strip() else []
        arts.append(cur)
        return arts
    tr, te = articles("train"), articles("test")
    cnt = collections.Counter(w for a in tr for w in a)
    vocab = ["<unk>"] + [w for w, _ in cnt.most_common(V - 1)]
    idx = {w: j for j, w in enumerate(vocab)}
    enc = lambda arts: [np.array([idx.get(w, 0) for w in a], np.int64) for a in arts]
    return enc(tr), enc(te)


def windows(art, ctx=3):
    if len(art) <= ctx:
        return np.zeros((0, ctx), np.int64), np.zeros(0, np.int64)
    X = np.stack([art[j:len(art) - ctx + j] for j in range(ctx)], 1)
    return X, art[ctx:]


def run_lm(arm, K, seed, rounds=20, d=64, h=128, B=256, lr=1e-3, lam=1e-4, V=10000, ctx=3, neg=1024, log_every=5):
    tr_arts, te_arts = load_wt2(V, ctx)
    rng = np.random.RandomState(seed)
    owner = rng.randint(0, K, size=len(tr_arts))
    Xs, Ys = [], []
    for k in range(K):
        parts = [windows(a, ctx) for a, o in zip(tr_arts, owner) if o == k]
        Xs.append(np.concatenate([p[0] for p in parts])); Ys.append(np.concatenate([p[1] for p in parts]))
    counts = np.zeros((K, V), np.int64)
    for k in range(K):  # windows whose context contains word w (each window counted once per distinct word)
        Xk = Xs[k]
        for j in range(ctx):
            first = np.ones(len(Xk), bool)
            for jj in range(j):
                first &= Xk[:, jj] != Xk[:, j]
            counts[k] += np.bincount(Xk[first, j], minlength=V)
    n_k = np.array([len(y) for y in Ys], float)
    tau = np.array([math.ceil(n / B) for n in n_k], float)
    kap, fbar = kappa(counts, tau, B, with_fbar=True)
    torch.manual_seed(seed)
    P0 = dict(E=0.1 * torch.randn(V, d), W1=torch.randn(ctx * d, h) / math.sqrt(ctx * d), b1=torch.zeros(h),
              W2=torch.randn(V, h) / math.sqrt(h), b2=torch.zeros(V))
    init_norm = P0["E"].norm(dim=1).numpy()
    P0 = {k_: v.to(DEV) for k_, v in P0.items()}
    Xs = [torch.from_numpy(x).to(DEV) for x in Xs]
    Ys = [torch.from_numpy(y).to(DEV) for y in Ys]
    gen = torch.Generator(device=DEV).manual_seed(seed + 5)
    P = {k_: v.clone() for k_, v in P0.items()}

    def make_opt(params):
        if arm in ("adam_l2", "adam_l2_hwa", "adam_l2_reset", "central_l2"):
            return torch.optim.Adam(params, lr=lr, weight_decay=lam)
        if arm == "adamw":
            return torch.optim.AdamW(params, lr=lr, weight_decay=lam)
        if arm == "adam":
            return torch.optim.Adam(params, lr=lr)
        if arm == "sgd_l2":
            return torch.optim.SGD(params, lr=0.1, weight_decay=lam)
        raise ValueError(arm)

    def train(Q, opt, X, Y, g):
        perm = torch.randperm(len(Y), device=DEV, generator=gen)
        for s0 in range(0, len(perm), B):
            b = perm[s0:s0 + B]
            x, y = X[b], Y[b]
            cand = torch.unique(torch.cat([y, torch.randint(0, V, (neg,), device=DEV, generator=gen)]))
            pos = torch.searchsorted(cand, y)
            hid = torch.relu(Q["E"][x].flatten(1) @ Q["W1"] + Q["b1"])
            logit = hid @ Q["W2"][cand].T + Q["b2"][cand]
            loss = F.cross_entropy(logit, pos)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    t0 = time.time()
    hist = []
    if arm == "central_l2":
        Q = {k_: v.clone().requires_grad_() for k_, v in P.items()}
        opt = make_opt(list(Q.values()))
        X, Y = torch.cat(Xs), torch.cat(Ys)
        for rd in range(rounds):
            train(Q, opt, X, Y, rng)
        P = {k_: v.detach() for k_, v in Q.items()}
    else:
        states = [None] * K
        for rd in range(rounds):
            acc = {k_: torch.zeros_like(v) for k_, v in P.items()}
            hwE, hwW = torch.zeros(V, d, device=DEV), torch.zeros(V, device=DEV)
            for k in range(K):
                Q = {k_: v.clone().requires_grad_() for k_, v in P.items()}
                opt = make_opt(list(Q.values()))
                if states[k] is not None and arm != "adam_l2_reset":
                    opt.load_state_dict(states[k])
                train(Q, opt, Xs[k], Ys[k], rng)
                states[k] = opt.state_dict()
                for k_ in P:
                    acc[k_] += n_k[k] * Q[k_].detach()
                w = torch.from_numpy(counts[k]).float().to(DEV)
                hwE += w[:, None] * Q["E"].detach()
                hwW += w
            P = {k_: v / n_k.sum() for k_, v in acc.items()}
            if arm == "adam_l2_hwa":
                held = hwW > 0
                P["E"][held] = hwE[held] / hwW[held, None]
            if rd % log_every == 0 or rd == rounds - 1:
                hist.append(dict(round=rd, median_norm_ratio=float(np.median(P["E"].norm(dim=1).cpu().numpy() / init_norm))))
    # evaluation: full softmax on the test articles
    Xt = np.concatenate([windows(a, ctx)[0] for a in te_arts]); Yt = np.concatenate([windows(a, ctx)[1] for a in te_arts])
    nll = []
    with torch.no_grad():
        for s0 in range(0, len(Yt), 4096):
            x, y = torch.from_numpy(Xt[s0:s0 + 4096]).to(DEV), torch.from_numpy(Yt[s0:s0 + 4096]).to(DEV)
            hid = torch.relu(P["E"][x].flatten(1) @ P["W1"] + P["b1"])
            nll.append(F.cross_entropy(hid @ P["W2"].T + P["b2"], y, reduction="none"))
    nll = torch.cat(nll).cpu().numpy()
    norm = P["E"].norm(dim=1).cpu().numpy() / init_norm
    lo = kap < 1
    lo[0] = False  # <unk> is held everywhere anyway
    rare_ctx = lo[Xt].any(1)
    fin = np.isfinite(kap)
    from scipy.stats import spearmanr
    return dict(task="lm", arm=arm, K=K, seed=seed, rounds=rounds, lam=lam, lr=lr, d=d, h=h, B=B, V=V,
                ppl=float(np.exp(nll.mean())), ppl_rare_ctx=float(np.exp(nll[rare_ctx].mean())) if rare_ctx.any() else None,
                ppl_other_ctx=float(np.exp(nll[~rare_ctx].mean())), n_test_rare_ctx=int(rare_ctx.sum()),
                n_rows=int(V), n_kappa_lt1=int(lo.sum()), collapsed_kappa_lt1=int((norm[lo] <= 0.15).sum()),
                collapsed_kappa_ge1=int((norm[~lo] <= 0.15).sum()),
                max_norm_kappa_lt1=float(norm[lo].max()) if lo.any() else None,
                median_norm_kappa_lt1=float(np.median(norm[lo])) if lo.any() else None,
                median_norm_kappa_ge1=float(np.median(norm[~lo])),
                spearman_logkappa_norm=float(spearmanr(np.log(kap[fin]), norm[fin])[0]) if fin.sum() > 2 else None,
                time_s=round(time.time() - t0, 1), history=hist,
                kappa=[round(float(x), 4) if np.isfinite(x) else None for x in kap],
                norm_ratio=[round(float(x), 4) for x in norm], train_count=counts.sum(0).tolist(),
                fbar=[float(f"{x:.4g}") for x in fbar])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", default="mf")
    ap.add_argument("--arms", nargs="+", default=["adam_l2", "adamw", "adam", "adam_l2_hwa", "sgd_l2", "adam_l2_reset",
                                                  "central_l2"])
    ap.add_argument("--K", type=int, nargs="+", default=[10, 50])
    ap.add_argument("--seeds", type=int, nargs="+", default=[0])
    ap.add_argument("--rounds", type=int, default=60)
    ap.add_argument("--out", default=None)
    a = ap.parse_args()
    torch.set_num_threads(int(os.environ.get("NT", "2")))
    out = a.out or os.path.join(OUT, "E40_fedembed_mf" if a.task == "mf" else "E41_fedembed_lm")
    os.makedirs(out, exist_ok=True)
    jl = os.path.join(out, "runs.jsonl")
    done = set()
    if os.path.exists(jl):
        done = {(json.loads(l)["arm"], json.loads(l)["K"], json.loads(l)["seed"]) for l in open(jl)}
    for seed in a.seeds:
        for K in a.K:
            for arm in a.arms:
                if arm == "central_l2" and K != a.K[0]:
                    continue  # one centralised run per seed
                if (arm, K, seed) in done:
                    continue
                res = (run_mf if a.task == "mf" else run_lm)(arm, K, seed, rounds=a.rounds)
                with open(jl, "a") as f:
                    f.write(json.dumps(res) + "\n")
                print(json.dumps({k: v for k, v in res.items() if k not in ("kappa", "norm_ratio", "train_count", "history")}),
                      flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import os, subprocess, threading, queue
out = "/kaggle/working/results/E41_fedembed_lm"
os.makedirs(out, exist_ok=True)
jobs = queue.Queue()
for a in ['--task lm --K 10 --seeds 0 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s0_K10 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset central_l2', '--task lm --K 50 --seeds 0 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s0_K50 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset', '--task lm --K 100 --seeds 0 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s0_K100 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset', '--task lm --K 10 --seeds 1 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s1_K10 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset central_l2', '--task lm --K 50 --seeds 1 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s1_K50 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset', '--task lm --K 100 --seeds 1 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s1_K100 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset', '--task lm --K 10 --seeds 2 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s2_K10 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset central_l2', '--task lm --K 50 --seeds 2 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s2_K50 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset', '--task lm --K 100 --seeds 2 --rounds 20 --out /kaggle/working/results/E41_fedembed_lm/s2_K100 --arms adam_l2 adamw adam adam_l2_hwa sgd_l2 adam_l2_reset']:
    jobs.put(a)
def worker(gpu):
    while True:
        try:
            a = jobs.get_nowait()
        except queue.Empty:
            return
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), GEFL_DATA_ROOT="/kaggle/working/data", GEFL_OUT=out, **{'NT': '2'})
        with open(os.path.join(out, "log_" + a.replace(" ", "_").replace(",", "-").replace("-", "")[:80] + ".txt"), "w") as f:
            rc = subprocess.run("python fedembed.py " + a, shell=True, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        print("GPU", gpu, a, "exit", rc, flush=True)
# data download once, before the two workers start (core datasets for the comparator script)
import sys; sys.path.insert(0, "/kaggle/working"); import core
for d in []:
    core.load_dataset(d, "/kaggle/working/data")
ts = [threading.Thread(target=worker, args=(g,)) for g in (0, 1)]
[t.start() for t in ts]; [t.join() for t in ts]

In [ ]:
pack()

In [ ]:
# Last cell - pack results and models into two separate zips
import os, shutil, glob, subprocess
for src, dst in [("/kaggle/working/results", "/kaggle/working/embedding_collapse_lm"), ("/kaggle/working/models", "/kaggle/working/embedding_collapse_lm__models")]:
    if os.path.isdir(src):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        shutil.make_archive(dst, "zip", "/kaggle/working", os.path.basename(dst))
        shutil.rmtree(dst)
        print("wrote", dst + ".zip")